In [1]:
# Cell 0 | 设置Data_split和IL_split内层折专属MPNN表征调参路径与配置


import gc
import hashlib
import json
import os
import random
import sys
import time
import traceback
import warnings

from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from scipy.stats import pearsonr

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim

from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, TensorDataset


try:
    import lightgbm

    from lightgbm import LGBMRegressor

except ImportError as error:
    raise ImportError(
        "当前环境未安装lightgbm。\n"
        "请在Anaconda Prompt中运行：\n"
        "conda activate rdkit-env\n"
        "conda install -c conda-forge lightgbm"
    ) from error


try:
    import optuna

except ImportError as error:
    raise ImportError(
        "当前环境未安装optuna。\n"
        "请在Anaconda Prompt中运行：\n"
        "conda activate rdkit-env\n"
        "pip install optuna"
    ) from error


warnings.filterwarnings(
    "ignore"
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# ============================================================
# 1. 通用函数
# ============================================================

def find_project_root(
    start_path=".",
):
    """向上查找同时包含data和notebooks的项目根目录。"""

    current_path = Path(
        start_path
    ).resolve()


    for _ in range(
        20
    ):
        if (
            (
                current_path
                / "data"
            ).exists()
            and (
                current_path
                / "notebooks"
            ).exists()
        ):
            return current_path


        if (
            current_path.parent
            == current_path
        ):
            break


        current_path = (
            current_path.parent
        )


    raise FileNotFoundError(
        "未找到项目根目录。\n"
        "请确认当前Notebook位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks"
    )


def read_json(
    path,
):
    """读取JSON文件。"""

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(
            file
        )


def read_csv_robust(
    path,
):
    """依次尝试常见编码读取CSV。"""

    encodings = [
        "utf-8-sig",
        "utf-8",
        "gb18030",
    ]

    last_error = None


    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                encoding=encoding,
            )

        except UnicodeDecodeError as error:
            last_error = error


    raise last_error


def calculate_sha256(
    path,
):
    """计算文件SHA256。"""

    sha256 = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            sha256.update(
                block
            )


    return sha256.hexdigest()


def seed_everything(
    seed=42,
):
    """固定Python、NumPy和PyTorch随机种子。"""

    random.seed(
        seed
    )

    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():
        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )


    torch.backends.cudnn.deterministic = (
        True
    )

    torch.backends.cudnn.benchmark = (
        False
    )


def to_jsonable(
    value,
):
    """将NumPy类型递归转换为JSON可写类型。"""

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): to_jsonable(
                item
            )
            for key, item
            in value.items()
        }


    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            to_jsonable(
                item
            )
            for item in value
        ]


    if isinstance(
        value,
        np.integer,
    ):
        return int(
            value
        )


    if isinstance(
        value,
        np.floating,
    ):
        return float(
            value
        )


    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()


    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )


    return value


def save_json_atomic(
    data,
    output_path,
):
    """原子方式保存JSON文件。"""

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / (
            output_path.name
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            to_jsonable(
                data
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )


    os.replace(
        temporary_path,
        output_path,
    )


# ============================================================
# 2. 项目路径
# ============================================================

PROJECT_ROOT = (
    find_project_root(
        Path.cwd()
    )
)


EXPERIMENT_ROOT = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)


MODEL_OUTPUT_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "model_training_3models_cv_no_leakage_2splits"
)


# 本Notebook唯一允许读取的MPNN表征根目录。
INNER_MPNN_FEATURE_ROOT = (
    MODEL_OUTPUT_ROOT
    / "inner_foldwise_mpnn_tuning_features"
)


# 使用新的独立调参目录，不覆盖此前旧调参结果。
TUNING_ROOT = (
    MODEL_OUTPUT_ROOT
    / "hyperparameter_tuning_inner_foldwise_mpnn"
)


STUDY_DIR = (
    TUNING_ROOT
    / "studies"
)


TRIAL_RESULT_DIR = (
    TUNING_ROOT
    / "trial_results"
)


BEST_PARAM_DIR = (
    TUNING_ROOT
    / "best_params_by_outer_fold"
)


SUMMARY_DIR = (
    TUNING_ROOT
    / "summaries"
)


TUNING_LOG_DIR = (
    TUNING_ROOT
    / "logs"
)


for folder in [
    TUNING_ROOT,
    STUDY_DIR,
    TRIAL_RESULT_DIR,
    BEST_PARAM_DIR,
    SUMMARY_DIR,
    TUNING_LOG_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


if not (
    INNER_MPNN_FEATURE_ROOT
    .exists()
):
    raise FileNotFoundError(
        "找不到30套内层折专属MPNN表征根目录：\n"
        f"{INNER_MPNN_FEATURE_ROOT}"
    )


# ============================================================
# 3. 划分、模型与调参配置
# ============================================================

SPLIT_ORDER = [
    "Data_split",
    "IL_split",
]


MODEL_ORDER = [
    "ExtraTrees",
    "LightGBM",
    "Transformer",
]


N_OUTER_FOLDS = 5

N_INNER_FOLDS = 3

EXPECTED_TASK_COUNT = (
    len(
        SPLIT_ORDER
    )
    * N_OUTER_FOLDS
    * N_INNER_FOLDS
)


RANDOM_SEED = 42

TUNING_RANDOM_SEED = (
    20260723
)


# 每个“划分方式×外层折×模型”分别建立Optuna study。
N_TRIALS_PER_OUTER_FOLD = {
    "ExtraTrees": 15,
    "LightGBM": 20,
    "Transformer": 10,
}


TRANSFORMER_SEARCH_EPOCHS = 50

TRANSFORMER_FORMAL_EPOCHS = 150

TRANSFORMER_NUM_TOKENS = 8

TRANSFORMER_NOISE_STD = 0.01


RUN_CODE_VERSION = (
    "11_nested_tuning_inner_foldwise_"
    "mpnn_Data_IL_v1_20260723"
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


OPTUNA_DB_PATH = (
    STUDY_DIR
    / (
        "optuna_viscosity_3models_"
        "Data_IL_inner_foldwise.db"
    )
)


INPUT_BUNDLE_AUDIT_PATH = (
    TUNING_LOG_DIR
    / (
        "inner_mpnn_input_bundle_"
        "audit_Data_IL.csv"
    )
)


INPUT_MANIFEST_PATH = (
    TUNING_LOG_DIR
    / (
        "inner_mpnn_input_manifest_"
        "Data_IL.json"
    )
)


# ============================================================
# 4. 明确禁止的输入文件
# ============================================================

FORBIDDEN_INPUT_BASENAMES = {
    "X_outer_train_mpnn.npy",
    "X_outer_test_mpnn.npy",
    "y_outer_train_lg_eta.npy",
    "y_outer_test_lg_eta.npy",
    "mpnn_features_eta.npy",
}


FORBIDDEN_PATH_TOKENS = {
    "external_10point",
    "10point_external",
    "external_holdout",
}


# ============================================================
# 5. 30套内层表征目录和文件路径
# ============================================================

def get_inner_task_id(
    split_name,
    outer_fold,
    inner_fold,
):
    """返回内层MPNN任务编号。"""

    return (
        f"{split_name}_"
        f"OuterFold{outer_fold}_"
        f"InnerFold{inner_fold}"
    )


def get_inner_task_dir(
    split_name,
    outer_fold,
    inner_fold,
):
    """返回一套内层MPNN任务目录。"""

    return (
        INNER_MPNN_FEATURE_ROOT
        / split_name
        / f"OuterFold_{outer_fold}"
        / f"InnerFold_{inner_fold}"
    )


def get_inner_tuning_bundle_paths(
    split_name,
    outer_fold,
    inner_fold,
):
    """
    返回下游调参唯一允许读取的文件。

    不返回任何外层训练、外层测试或外部10点文件。
    """

    task_dir = (
        get_inner_task_dir(
            split_name=split_name,
            outer_fold=outer_fold,
            inner_fold=inner_fold,
        )
    )


    return {
        "task_dir": task_dir,

        "X_inner_train": (
            task_dir
            / "X_inner_train_mpnn.npy"
        ),

        "X_inner_valid": (
            task_dir
            / "X_inner_valid_mpnn.npy"
        ),

        "y_inner_train": (
            task_dir
            / "y_inner_train_lg_eta.npy"
        ),

        "y_inner_valid": (
            task_dir
            / "y_inner_valid_lg_eta.npy"
        ),

        "feature_metadata": (
            task_dir
            / "inner_feature_metadata.json"
        ),

        "output_audit": (
            task_dir
            / "inner_output_audit.json"
        ),

        "training_summary": (
            task_dir
            / "mpnn_training_summary.json"
        ),
    }


# ============================================================
# 6. 检查全部30套输入
# ============================================================

input_bundle_rows = []

missing_input_files = []


for split_name in SPLIT_ORDER:
    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):
        for inner_fold in range(
            1,
            N_INNER_FOLDS + 1,
        ):
            task_id = (
                get_inner_task_id(
                    split_name=split_name,
                    outer_fold=outer_fold,
                    inner_fold=inner_fold,
                )
            )


            paths = (
                get_inner_tuning_bundle_paths(
                    split_name=split_name,
                    outer_fold=outer_fold,
                    inner_fold=inner_fold,
                )
            )


            for (
                file_role,
                file_path,
            ) in paths.items():

                if file_role == "task_dir":
                    continue


                if not file_path.exists():
                    missing_input_files.append(
                        file_path
                    )


            if missing_input_files:
                continue


            feature_metadata = (
                read_json(
                    paths[
                        "feature_metadata"
                    ]
                )
            )


            output_audit = (
                read_json(
                    paths[
                        "output_audit"
                    ]
                )
            )


            training_summary = (
                read_json(
                    paths[
                        "training_summary"
                    ]
                )
            )


            metadata_task_match = bool(
                feature_metadata.get(
                    "task_id"
                )
                == task_id
            )


            output_audit_task_match = bool(
                output_audit.get(
                    "task_id"
                )
                == task_id
            )


            training_task_match = bool(
                training_summary.get(
                    "task_id"
                )
                == task_id
            )


            feature_status_passed = bool(
                feature_metadata.get(
                    "status"
                )
                == "feature_extraction_completed"
            )


            output_audit_passed = bool(
                output_audit.get(
                    "audit_status"
                )
                == "passed"
            )


            training_status_passed = bool(
                training_summary.get(
                    "status"
                )
                == "training_completed"
            )


            if not all(
                [
                    metadata_task_match,
                    output_audit_task_match,
                    training_task_match,
                    feature_status_passed,
                    output_audit_passed,
                    training_status_passed,
                ]
            ):
                raise RuntimeError(
                    f"{task_id}的内层MPNN输出状态"
                    "或任务身份检查未通过。"
                )


            bundle_path_strings = [
                str(
                    path
                ).lower()
                for role, path
                in paths.items()
                if role != "task_dir"
            ]


            forbidden_basename_hits = [
                Path(
                    path
                ).name
                for path in bundle_path_strings
                if Path(
                    path
                ).name
                in {
                    name.lower()
                    for name
                    in FORBIDDEN_INPUT_BASENAMES
                }
            ]


            forbidden_token_hits = [
                token
                for token in FORBIDDEN_PATH_TOKENS
                if any(
                    token in path
                    for path
                    in bundle_path_strings
                )
            ]


            if (
                forbidden_basename_hits
                or forbidden_token_hits
            ):
                raise RuntimeError(
                    f"{task_id}的允许输入路径中"
                    "出现禁止文件或禁止目录标记。"
                )


            input_bundle_rows.append(
                {
                    "Task_id": task_id,

                    "Split": split_name,

                    "Outer_fold": int(
                        outer_fold
                    ),

                    "Inner_fold": int(
                        inner_fold
                    ),

                    "Task_dir": str(
                        paths[
                            "task_dir"
                        ]
                    ),

                    "X_inner_train_path": str(
                        paths[
                            "X_inner_train"
                        ]
                    ),

                    "X_inner_valid_path": str(
                        paths[
                            "X_inner_valid"
                        ]
                    ),

                    "y_inner_train_path": str(
                        paths[
                            "y_inner_train"
                        ]
                    ),

                    "y_inner_valid_path": str(
                        paths[
                            "y_inner_valid"
                        ]
                    ),

                    "Feature_metadata_path": str(
                        paths[
                            "feature_metadata"
                        ]
                    ),

                    "Output_audit_path": str(
                        paths[
                            "output_audit"
                        ]
                    ),

                    "Training_summary_path": str(
                        paths[
                            "training_summary"
                        ]
                    ),

                    "Feature_status_passed": (
                        feature_status_passed
                    ),

                    "Output_audit_passed": (
                        output_audit_passed
                    ),

                    "Training_status_passed": (
                        training_status_passed
                    ),

                    "Forbidden_input_detected": False,
                }
            )


if missing_input_files:
    unique_missing_files = sorted(
        {
            str(
                path
            )
            for path
            in missing_input_files
        }
    )


    raise FileNotFoundError(
        "缺少以下内层折专属MPNN输入文件：\n"
        + "\n".join(
            unique_missing_files
        )
    )


df_inner_tuning_input_manifest = (
    pd.DataFrame(
        input_bundle_rows
    )
    .sort_values(
        [
            "Split",
            "Outer_fold",
            "Inner_fold",
        ],
        key=lambda series: (
            series.map(
                {
                    "Data_split": 0,
                    "IL_split": 1,
                }
            )
            if series.name == "Split"
            else series
        ),
    )
    .reset_index(
        drop=True
    )
)


if len(
    df_inner_tuning_input_manifest
) != EXPECTED_TASK_COUNT:
    raise RuntimeError(
        "内层MPNN输入任务数错误："
        f"实际={len(df_inner_tuning_input_manifest)}，"
        f"预期={EXPECTED_TASK_COUNT}"
    )


split_task_counts = (
    df_inner_tuning_input_manifest
    .groupby(
        "Split"
    )
    .size()
    .to_dict()
)


expected_split_task_counts = {
    "Data_split": 15,
    "IL_split": 15,
}


if (
    split_task_counts
    != expected_split_task_counts
):
    raise RuntimeError(
        "两种划分方式的内层任务数量错误：\n"
        f"实际={split_task_counts}\n"
        f"预期={expected_split_task_counts}"
    )


if not (
    df_inner_tuning_input_manifest[
        "Feature_status_passed"
    ].all()
    and df_inner_tuning_input_manifest[
        "Output_audit_passed"
    ].all()
    and df_inner_tuning_input_manifest[
        "Training_status_passed"
    ].all()
):
    raise RuntimeError(
        "30套任务中存在未通过审计的任务。"
    )


if (
    df_inner_tuning_input_manifest[
        "Forbidden_input_detected"
    ].any()
):
    raise RuntimeError(
        "30套任务输入中发现禁止读取的文件。"
    )


# ============================================================
# 7. 保存输入清单和Cell 0审计
# ============================================================

df_inner_tuning_input_manifest.to_csv(
    INPUT_BUNDLE_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)


input_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "inner_mpnn_input_check_passed"
    ),

    "project_root": str(
        PROJECT_ROOT
    ),

    "experiment_root": str(
        EXPERIMENT_ROOT
    ),

    "inner_mpnn_feature_root": str(
        INNER_MPNN_FEATURE_ROOT
    ),

    "tuning_root": str(
        TUNING_ROOT
    ),

    "splits": (
        SPLIT_ORDER
    ),

    "models": (
        MODEL_ORDER
    ),

    "outer_folds_per_split": int(
        N_OUTER_FOLDS
    ),

    "inner_folds_per_outer_fold": int(
        N_INNER_FOLDS
    ),

    "expected_task_count": int(
        EXPECTED_TASK_COUNT
    ),

    "actual_task_count": int(
        len(
            df_inner_tuning_input_manifest
        )
    ),

    "task_count_by_split": (
        split_task_counts
    ),

    "trial_budgets_per_outer_fold": (
        N_TRIALS_PER_OUTER_FOLD
    ),

    "transformer_search_epochs": int(
        TRANSFORMER_SEARCH_EPOCHS
    ),

    "transformer_formal_epochs": int(
        TRANSFORMER_FORMAL_EPOCHS
    ),

    "input_policy": {
        "allowed_feature_files": [
            "X_inner_train_mpnn.npy",
            "X_inner_valid_mpnn.npy",
        ],

        "allowed_target_files": [
            "y_inner_train_lg_eta.npy",
            "y_inner_valid_lg_eta.npy",
        ],

        "forbidden_input_basenames": sorted(
            FORBIDDEN_INPUT_BASENAMES
        ),

        "forbidden_path_tokens": sorted(
            FORBIDDEN_PATH_TOKENS
        ),
    },

    "leakage_control": {
        "old_global_mpnn_feature_read": False,

        "outer_train_mpnn_feature_read": False,

        "outer_test_mpnn_feature_read": False,

        "outer_train_target_read": False,

        "outer_test_target_read": False,

        "external_10point_holdout_read": False,

        "inner_fold_specific_mpnn_outputs_read": (
            True
        ),
    },

    "outputs": {
        "input_bundle_audit": str(
            INPUT_BUNDLE_AUDIT_PATH
        ),

        "input_manifest": str(
            INPUT_MANIFEST_PATH
        ),

        "optuna_database": str(
            OPTUNA_DB_PATH
        ),
    },
}


save_json_atomic(
    data=input_manifest,
    output_path=INPUT_MANIFEST_PATH,
)


seed_everything(
    RANDOM_SEED
)


# ============================================================
# 8. 输出检查
# ============================================================

print("=" * 104)
print("Cell 0 内层折专属MPNN调参路径与30套输入检查通过 ✓")
print("=" * 104)

print(
    "python                           =",
    sys.executable,
)

print(
    "optuna                          =",
    optuna.__version__,
)

print(
    "lightgbm                        =",
    lightgbm.__version__,
)

print(
    "torch                           =",
    torch.__version__,
)

print(
    "device                          =",
    DEVICE,
)

print(
    "PROJECT_ROOT                    =",
    PROJECT_ROOT,
)

print(
    "EXPERIMENT_ROOT                 =",
    EXPERIMENT_ROOT,
)

print(
    "INNER_MPNN_FEATURE_ROOT         =",
    INNER_MPNN_FEATURE_ROOT,
)

print(
    "TUNING_ROOT                     =",
    TUNING_ROOT,
)

print(
    "splits                          =",
    SPLIT_ORDER,
)

print(
    "models                          =",
    MODEL_ORDER,
)

print(
    "outer folds/split               =",
    N_OUTER_FOLDS,
)

print(
    "inner folds/outer fold          =",
    N_INNER_FOLDS,
)

print(
    "expected inner MPNN tasks       =",
    EXPECTED_TASK_COUNT,
)

print(
    "checked inner MPNN tasks        =",
    len(
        df_inner_tuning_input_manifest
    ),
)

print(
    "task counts by split            =",
    split_task_counts,
)

print(
    "trial budgets/outer fold        =",
    N_TRIALS_PER_OUTER_FOLD,
)

print(
    "Transformer search epochs      =",
    TRANSFORMER_SEARCH_EPOCHS,
)

print(
    "all feature audits passed       =",
    bool(
        df_inner_tuning_input_manifest[
            "Feature_status_passed"
        ].all()
    ),
)

print(
    "all output audits passed        =",
    bool(
        df_inner_tuning_input_manifest[
            "Output_audit_passed"
        ].all()
    ),
)

print(
    "all training summaries passed   =",
    bool(
        df_inner_tuning_input_manifest[
            "Training_status_passed"
        ].all()
    ),
)

print(
    "forbidden input detected        =",
    bool(
        df_inner_tuning_input_manifest[
            "Forbidden_input_detected"
        ].any()
    ),
)

print(
    "old global feature read         =",
    False,
)

print(
    "outer train feature read        =",
    False,
)

print(
    "outer test feature read         =",
    False,
)

print(
    "outer test target read          =",
    False,
)

print(
    "external 10 points read         =",
    False,
)

print(
    "run code version                =",
    RUN_CODE_VERSION,
)


print(
    "\n输入任务汇总："
)

display(
    df_inner_tuning_input_manifest.groupby(
        [
            "Split",
            "Feature_status_passed",
            "Output_audit_passed",
            "Training_status_passed",
        ],
        as_index=False,
    ).size()
)


print(
    "\n30套输入审计表："
)

print(
    INPUT_BUNDLE_AUDIT_PATH
)


print(
    "\n输入manifest："
)

print(
    INPUT_MANIFEST_PATH
)


print(
    "\nOptuna数据库："
)

print(
    OPTUNA_DB_PATH
)

Cell 0 内层折专属MPNN调参路径与30套输入检查通过 ✓
python                           = D:\conda\envs\rdkit-env\python.exe
optuna                          = 4.8.0
lightgbm                        = 4.6.0
torch                           = 2.2.1+cpu
device                          = cpu
PROJECT_ROOT                    = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT                 = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
INNER_MPNN_FEATURE_ROOT         = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\inner_foldwise_mpnn_tuning_features
TUNING_ROOT                     = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn
splits                          = ['Data_split', 'IL_split']
models                          = ['ExtraTrees', 'LightGBM', 'Transformer']
outer folds/split               = 5
inner folds/o

,Split,Feature_status_passed,Output_audit_passed,Training_status_passed,size
0,Data_split,True,True,True,15
1,IL_split,True,True,True,15



30套输入审计表：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\inner_mpnn_input_bundle_audit_Data_IL.csv

输入manifest：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\inner_mpnn_input_manifest_Data_IL.json

Optuna数据库：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\studies\optuna_viscosity_3models_Data_IL_inner_foldwise.db


In [2]:
# Cell 1 | 读取并审计30套内层折专属MPNN调参数组、索引与数据边界


INNER_ARRAY_AUDIT_PATH = (
    TUNING_LOG_DIR
    / "inner_tuning_array_audit_Data_IL.csv"
)


INNER_ARRAY_MANIFEST_PATH = (
    TUNING_LOG_DIR
    / "inner_tuning_array_manifest_Data_IL.json"
)


# ============================================================
# 1. 通用检查函数
# ============================================================

def normalize_columns(
    dataframe,
):
    """统一清单字段名格式。"""

    dataframe = dataframe.copy()

    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
    )

    return dataframe


def normalize_path_for_compare(
    path,
):
    """将路径转换为可比较形式。"""

    return os.path.normcase(
        os.path.abspath(
            os.path.normpath(
                str(
                    path
                )
            )
        )
    )


def paths_are_equal(
    path_a,
    path_b,
):
    """判断两个路径是否指向同一位置。"""

    return bool(
        normalize_path_for_compare(
            path_a
        )
        == normalize_path_for_compare(
            path_b
        )
    )


def get_alias_value(
    mapping,
    candidate_keys,
    task_id,
    value_name,
):
    """兼容第一套任务和批量任务中的审计字段命名。"""

    for key in candidate_keys:
        if key in mapping:
            return mapping[
                key
            ]


    raise KeyError(
        f"{task_id}：找不到审计字段"
        f"{value_name}。\n"
        f"候选字段={candidate_keys}"
    )


def require_true_alias(
    mapping,
    candidate_keys,
    task_id,
    value_name,
):
    """要求候选字段中的实际值为True。"""

    value = get_alias_value(
        mapping=mapping,
        candidate_keys=candidate_keys,
        task_id=task_id,
        value_name=value_name,
    )


    if not bool(
        value
    ):
        raise RuntimeError(
            f"{task_id}：{value_name}"
            "没有通过。"
        )


    return True


def require_false_alias(
    mapping,
    candidate_keys,
    task_id,
    value_name,
):
    """要求候选字段中的实际值为False。"""

    value = get_alias_value(
        mapping=mapping,
        candidate_keys=candidate_keys,
        task_id=task_id,
        value_name=value_name,
    )


    if bool(
        value
    ):
        raise RuntimeError(
            f"{task_id}：{value_name}=True，"
            "已停止运行。"
        )


    return True


def resolve_il_group_series(
    manifest,
):
    """从清单中取得完整离子液体分组标识。"""

    manifest = normalize_columns(
        manifest
    )


    direct_candidates = [
        "IL_key",
        "il_key",
        "IL_ID",
        "il_id",
    ]


    for column in direct_candidates:
        if column in manifest.columns:
            groups = (
                manifest[
                    column
                ]
                .astype(str)
                .str.strip()
            )


            invalid_groups = (
                groups.eq("")
                | groups.str.lower().isin(
                    [
                        "nan",
                        "none",
                        "null",
                    ]
                )
            )


            if invalid_groups.any():
                raise ValueError(
                    f"分组列{column}中存在空值。"
                )


            return (
                groups.to_numpy(
                    dtype=str
                ),
                column,
            )


    if (
        "cation_smiles"
        in manifest.columns
        and "anion_smiles"
        in manifest.columns
    ):
        cation = (
            manifest[
                "cation_smiles"
            ]
            .astype(str)
            .str.strip()
        )

        anion = (
            manifest[
                "anion_smiles"
            ]
            .astype(str)
            .str.strip()
        )


        invalid_cation = (
            cation.eq("")
            | cation.str.lower().isin(
                [
                    "nan",
                    "none",
                    "null",
                ]
            )
        )

        invalid_anion = (
            anion.eq("")
            | anion.str.lower().isin(
                [
                    "nan",
                    "none",
                    "null",
                ]
            )
        )


        if (
            invalid_cation.any()
            or invalid_anion.any()
        ):
            raise ValueError(
                "清单缺少可用IL_key，"
                "且阳、阴离子SMILES中存在空值。"
            )


        groups = (
            cation
            + "||"
            + anion
        )


        return (
            groups.to_numpy(
                dtype=str
            ),
            "cation_smiles||anion_smiles",
        )


    raise KeyError(
        "清单中既没有IL_key，"
        "也没有完整的cation_smiles和anion_smiles。"
    )


# ============================================================
# 2. 30套任务的完整路径
# ============================================================

def get_complete_inner_tuning_paths(
    split_name,
    outer_fold,
    inner_fold,
):
    """返回一套内层调参任务的数组、索引和审计文件。"""

    paths = (
        get_inner_tuning_bundle_paths(
            split_name=split_name,
            outer_fold=outer_fold,
            inner_fold=inner_fold,
        )
    )


    task_dir = Path(
        paths[
            "task_dir"
        ]
    )


    paths.update(
        {
            "inner_train_manifest": (
                task_dir
                / "inner_train_manifest.csv"
            ),

            "inner_valid_structure_manifest": (
                task_dir
                / (
                    "inner_valid_structure_"
                    "manifest_no_targets.csv"
                )
            ),

            "mpnn_internal_indices": (
                task_dir
                / "mpnn_internal_indices.npz"
            ),

            "mpnn_preprocessing_audit": (
                task_dir
                / "mpnn_preprocessing_audit.json"
            ),

            "mpnn_dataloader_audit": (
                task_dir
                / "mpnn_dataloader_audit.json"
            ),
        }
    )


    return paths


# 后续Cell只通过这个字典定位30套任务。
INNER_TUNING_BUNDLE_PATHS = {}


# ============================================================
# 3. 下游模型使用的轻量读取函数
# ============================================================

def load_inner_tuning_arrays(
    split_name,
    outer_fold,
    inner_fold,
):
    """
    读取一套下游调参数组。

    该函数只返回：
    X_inner_train
    X_inner_valid
    y_inner_train
    y_inner_valid
    """

    task_key = (
        split_name,
        int(
            outer_fold
        ),
        int(
            inner_fold
        ),
    )


    if (
        task_key
        not in INNER_TUNING_BUNDLE_PATHS
    ):
        paths = (
            get_complete_inner_tuning_paths(
                split_name=split_name,
                outer_fold=outer_fold,
                inner_fold=inner_fold,
            )
        )

    else:
        paths = (
            INNER_TUNING_BUNDLE_PATHS[
                task_key
            ]
        )


    X_inner_train = np.load(
        paths[
            "X_inner_train"
        ],
        allow_pickle=False,
    )

    X_inner_valid = np.load(
        paths[
            "X_inner_valid"
        ],
        allow_pickle=False,
    )

    y_inner_train = np.load(
        paths[
            "y_inner_train"
        ],
        allow_pickle=False,
    )

    y_inner_valid = np.load(
        paths[
            "y_inner_valid"
        ],
        allow_pickle=False,
    )


    if (
        X_inner_train.ndim != 2
        or X_inner_valid.ndim != 2
    ):
        raise ValueError(
            f"{task_key}：特征数组不是二维矩阵。"
        )


    if (
        y_inner_train.ndim != 1
        or y_inner_valid.ndim != 1
    ):
        raise ValueError(
            f"{task_key}：目标数组不是一维数组。"
        )


    if (
        X_inner_train.shape[1] != 130
        or X_inner_valid.shape[1] != 130
    ):
        raise ValueError(
            f"{task_key}：MPNN表征维度不是130。"
        )


    if (
        len(
            X_inner_train
        )
        != len(
            y_inner_train
        )
        or len(
            X_inner_valid
        )
        != len(
            y_inner_valid
        )
    ):
        raise ValueError(
            f"{task_key}：特征和目标行数不一致。"
        )


    if not all(
        np.isfinite(
            array
        ).all()
        for array in [
            X_inner_train,
            X_inner_valid,
            y_inner_train,
            y_inner_valid,
        ]
    ):
        raise ValueError(
            f"{task_key}：调参数组中存在NaN或无穷值。"
        )


    return {
        "X_inner_train": (
            X_inner_train
        ),

        "X_inner_valid": (
            X_inner_valid
        ),

        "y_inner_train": (
            y_inner_train
        ),

        "y_inner_valid": (
            y_inner_valid
        ),

        "paths": paths,
    }


# ============================================================
# 4. 审计一套内层调参任务
# ============================================================

def audit_one_inner_tuning_task(
    split_name,
    outer_fold,
    inner_fold,
):
    """读取并完整审计一套内层折专属调参输入。"""

    task_id = (
        get_inner_task_id(
            split_name=split_name,
            outer_fold=outer_fold,
            inner_fold=inner_fold,
        )
    )


    paths = (
        get_complete_inner_tuning_paths(
            split_name=split_name,
            outer_fold=outer_fold,
            inner_fold=inner_fold,
        )
    )


    required_paths = {
        key: path
        for key, path
        in paths.items()
        if key != "task_dir"
    }


    missing_paths = [
        path
        for path in required_paths.values()
        if not Path(
            path
        ).exists()
    ]


    if missing_paths:
        raise FileNotFoundError(
            f"{task_id}缺少文件：\n"
            + "\n".join(
                str(
                    path
                )
                for path
                in missing_paths
            )
        )


    # --------------------------------------------------------
    # 4.1 检查路径中没有禁止输入
    # --------------------------------------------------------

    all_path_strings = [
        str(
            path
        ).lower()
        for path in required_paths.values()
    ]


    forbidden_basename_hits = [
        Path(
            path
        ).name
        for path in all_path_strings
        if Path(
            path
        ).name
        in {
            name.lower()
            for name
            in FORBIDDEN_INPUT_BASENAMES
        }
    ]


    forbidden_token_hits = [
        token
        for token in FORBIDDEN_PATH_TOKENS
        if any(
            token in path
            for path
            in all_path_strings
        )
    ]


    if (
        forbidden_basename_hits
        or forbidden_token_hits
    ):
        raise RuntimeError(
            f"{task_id}发现禁止输入：\n"
            f"文件={forbidden_basename_hits}\n"
            f"路径标记={forbidden_token_hits}"
        )


    # --------------------------------------------------------
    # 4.2 读取四个数组
    # --------------------------------------------------------

    arrays = {
        "X_inner_train_mpnn": np.load(
            paths[
                "X_inner_train"
            ],
            allow_pickle=False,
            mmap_mode="r",
        ),

        "X_inner_valid_mpnn": np.load(
            paths[
                "X_inner_valid"
            ],
            allow_pickle=False,
            mmap_mode="r",
        ),

        "y_inner_train_lg_eta": np.load(
            paths[
                "y_inner_train"
            ],
            allow_pickle=False,
            mmap_mode="r",
        ),

        "y_inner_valid_lg_eta": np.load(
            paths[
                "y_inner_valid"
            ],
            allow_pickle=False,
            mmap_mode="r",
        ),
    }


    X_inner_train = arrays[
        "X_inner_train_mpnn"
    ]

    X_inner_valid = arrays[
        "X_inner_valid_mpnn"
    ]

    y_inner_train = arrays[
        "y_inner_train_lg_eta"
    ]

    y_inner_valid = arrays[
        "y_inner_valid_lg_eta"
    ]


    if (
        X_inner_train.ndim != 2
        or X_inner_valid.ndim != 2
    ):
        raise ValueError(
            f"{task_id}：特征数组不是二维矩阵。"
        )


    if (
        y_inner_train.ndim != 1
        or y_inner_valid.ndim != 1
    ):
        raise ValueError(
            f"{task_id}：目标数组不是一维数组。"
        )


    if (
        X_inner_train.shape[1] != 130
        or X_inner_valid.shape[1] != 130
    ):
        raise ValueError(
            f"{task_id}：特征维度不是130。"
        )


    if (
        len(
            X_inner_train
        )
        != len(
            y_inner_train
        )
        or len(
            X_inner_valid
        )
        != len(
            y_inner_valid
        )
    ):
        raise ValueError(
            f"{task_id}：特征和目标行数不一致。"
        )


    if not all(
        np.isfinite(
            array
        ).all()
        for array in arrays.values()
    ):
        raise ValueError(
            f"{task_id}：数组中存在NaN或无穷值。"
        )


    if not all(
        str(
            array.dtype
        )
        == "float32"
        for array in arrays.values()
    ):
        raise ValueError(
            f"{task_id}：部分数组不是float32。"
        )


    n_inner_train = int(
        len(
            y_inner_train
        )
    )

    n_inner_valid = int(
        len(
            y_inner_valid
        )
    )

    outer_train_rows = int(
        n_inner_train
        + n_inner_valid
    )


    # --------------------------------------------------------
    # 4.3 读取元数据、输出审计和训练摘要
    # --------------------------------------------------------

    feature_metadata = read_json(
        paths[
            "feature_metadata"
        ]
    )

    output_audit = read_json(
        paths[
            "output_audit"
        ]
    )

    training_summary = read_json(
        paths[
            "training_summary"
        ]
    )


    identity_checks = {
        "feature_metadata_task": (
            feature_metadata.get(
                "task_id"
            )
            == task_id
        ),

        "output_audit_task": (
            output_audit.get(
                "task_id"
            )
            == task_id
        ),

        "training_summary_task": (
            training_summary.get(
                "task_id"
            )
            == task_id
        ),

        "metadata_split": (
            feature_metadata.get(
                "split_name"
            )
            == split_name
        ),

        "metadata_outer_fold": (
            int(
                feature_metadata.get(
                    "outer_fold",
                    -1,
                )
            )
            == int(
                outer_fold
            )
        ),

        "metadata_inner_fold": (
            int(
                feature_metadata.get(
                    "inner_fold",
                    -1,
                )
            )
            == int(
                inner_fold
            )
        ),
    }


    if not all(
        identity_checks.values()
    ):
        failed_identity_checks = [
            key
            for key, value
            in identity_checks.items()
            if not value
        ]

        raise RuntimeError(
            f"{task_id}身份检查失败："
            f"{failed_identity_checks}"
        )


    if (
        feature_metadata.get(
            "status"
        )
        != "feature_extraction_completed"
    ):
        raise RuntimeError(
            f"{task_id}：表征提取状态未完成。"
        )


    if (
        output_audit.get(
            "audit_status"
        )
        != "passed"
    ):
        raise RuntimeError(
            f"{task_id}：输出审计未通过。"
        )


    if (
        training_summary.get(
            "status"
        )
        != "training_completed"
    ):
        raise RuntimeError(
            f"{task_id}：MPNN训练状态未完成。"
        )


    # --------------------------------------------------------
    # 4.4 核对元数据中的形状、类型、路径和哈希
    # --------------------------------------------------------

    array_path_mapping = {
        "X_inner_train_mpnn": (
            paths[
                "X_inner_train"
            ]
        ),

        "X_inner_valid_mpnn": (
            paths[
                "X_inner_valid"
            ]
        ),

        "y_inner_train_lg_eta": (
            paths[
                "y_inner_train"
            ]
        ),

        "y_inner_valid_lg_eta": (
            paths[
                "y_inner_valid"
            ]
        ),
    }


    metadata_shapes = (
        feature_metadata.get(
            "output_shapes",
            {},
        )
    )

    metadata_dtypes = (
        feature_metadata.get(
            "output_dtypes",
            {},
        )
    )

    metadata_hashes = (
        feature_metadata.get(
            "output_sha256",
            {},
        )
    )

    metadata_paths = (
        feature_metadata.get(
            "output_paths",
            {},
        )
    )


    actual_hashes = {}

    shape_checks = {}

    dtype_checks = {}

    path_checks = {}

    hash_checks = {}


    for (
        array_role,
        array,
    ) in arrays.items():

        actual_path = (
            array_path_mapping[
                array_role
            ]
        )


        actual_hash = calculate_sha256(
            actual_path
        )


        actual_hashes[
            array_role
        ] = actual_hash


        shape_checks[
            array_role
        ] = bool(
            list(
                array.shape
            )
            == list(
                metadata_shapes.get(
                    array_role,
                    [],
                )
            )
        )


        dtype_checks[
            array_role
        ] = bool(
            str(
                array.dtype
            )
            == str(
                metadata_dtypes.get(
                    array_role,
                    "",
                )
            )
        )


        path_checks[
            array_role
        ] = bool(
            array_role
            in metadata_paths
            and paths_are_equal(
                actual_path,
                metadata_paths[
                    array_role
                ],
            )
        )


        hash_checks[
            array_role
        ] = bool(
            actual_hash
            == metadata_hashes.get(
                array_role
            )
        )


    if not all(
        shape_checks.values()
    ):
        raise RuntimeError(
            f"{task_id}：数组形状"
            "与表征元数据不一致。"
        )


    if not all(
        dtype_checks.values()
    ):
        raise RuntimeError(
            f"{task_id}：数组类型"
            "与表征元数据不一致。"
        )


    if not all(
        path_checks.values()
    ):
        raise RuntimeError(
            f"{task_id}：数组路径"
            "与表征元数据不一致。"
        )


    if not all(
        hash_checks.values()
    ):
        failed_hashes = [
            key
            for key, value
            in hash_checks.items()
            if not value
        ]

        raise RuntimeError(
            f"{task_id}：以下数组SHA256不一致："
            f"{failed_hashes}"
        )


    descriptor_dim = int(
        feature_metadata.get(
            "feature_extraction",
            {},
        ).get(
            "descriptor_dim",
            -1,
        )
    )


    if descriptor_dim != 130:
        raise RuntimeError(
            f"{task_id}：元数据中的"
            f"descriptor_dim={descriptor_dim}，"
            "不是130。"
        )


    # --------------------------------------------------------
    # 4.5 核对输出审计中的关键检查
    # --------------------------------------------------------

    output_checks = (
        output_audit.get(
            "checks",
            {},
        )
    )


    required_true_output_checks = {
        "train_feature_shape_correct": [
            "train_feature_shape_correct",
            "inner_train_feature_shape_correct",
        ],

        "valid_feature_shape_correct": [
            "valid_feature_shape_correct",
            "inner_valid_feature_shape_correct",
        ],

        "train_target_shape_correct": [
            "train_target_shape_correct",
            "inner_train_target_shape_correct",
        ],

        "valid_target_shape_correct": [
            "valid_target_shape_correct",
            "inner_valid_target_shape_correct",
        ],

        "all_features_finite": [
            "all_features_finite",
        ],

        "all_targets_finite": [
            "all_targets_finite",
        ],

        "train_tp_columns_match": [
            "train_tp_columns_match",
        ],

        "valid_tp_columns_match": [
            "valid_tp_columns_match",
        ],

        "train_feature_target_order_match": [
            "train_feature_target_order_match",
        ],

        "valid_feature_target_order_match": [
            "valid_feature_target_order_match",
        ],
    }


    for (
        check_name,
        aliases,
    ) in required_true_output_checks.items():

        require_true_alias(
            mapping=output_checks,
            candidate_keys=aliases,
            task_id=task_id,
            value_name=check_name,
        )


    output_leakage = (
        output_audit.get(
            "leakage_control",
            {},
        )
    )


    false_output_leakage_checks = {
        "validation used for MPNN training": [
            (
                "inner_validation_target_"
                "used_for_mpnn_training"
            ),
            (
                "inner_validation_label_"
                "used_for_mpnn_training"
            ),
        ],

        "validation used for MPNN early stopping": [
            (
                "inner_validation_target_"
                "used_for_mpnn_early_stopping"
            ),
            (
                "inner_validation_label_"
                "used_for_mpnn_early_stopping"
            ),
        ],

        "validation used for MPNN scheduler": [
            (
                "inner_validation_label_"
                "used_for_mpnn_scheduler"
            ),
            (
                "inner_validation_target_"
                "used_for_mpnn_scheduler"
            ),
        ],

        "validation used for feature extraction": [
            (
                "inner_validation_target_"
                "used_for_mpnn_feature_extraction"
            ),
            (
                "inner_validation_label_"
                "used_for_mpnn_feature_extraction"
            ),
        ],

        "outer test structure used": [
            "outer_test_structure_read",
            "outer_test_structure_used",
        ],

        "outer test target used": [
            "outer_test_target_read",
            "outer_test_target_used",
        ],

        "outer test feature extracted": [
            "outer_test_feature_extracted",
        ],

        "existing outer MPNN features read": [
            "existing_outer_mpnn_features_read",
        ],

        "old global MPNN feature read": [
            "old_global_mpnn_feature_read",
        ],

        "external 10-point holdout read": [
            "external_10point_holdout_read",
        ],
    }


    # 第一套任务未单独记录scheduler字段，
    # 其训练摘要中已经记录该项。
    for (
        check_name,
        aliases,
    ) in false_output_leakage_checks.items():

        available_aliases = [
            key
            for key in aliases
            if key in output_leakage
        ]


        if available_aliases:
            require_false_alias(
                mapping=output_leakage,
                candidate_keys=available_aliases,
                task_id=task_id,
                value_name=check_name,
            )


    validation_saved_for_ml = bool(
        get_alias_value(
            mapping=output_leakage,
            candidate_keys=[
                (
                    "inner_validation_target_"
                    "saved_for_downstream_ml_validation"
                ),
                (
                    "inner_validation_label_used_after_"
                    "feature_extraction_for_downstream_"
                    "validation"
                ),
            ],
            task_id=task_id,
            value_name=(
                "validation saved for downstream ML"
            ),
        )
    )


    if not validation_saved_for_ml:
        raise RuntimeError(
            f"{task_id}：内层验证目标"
            "没有被保留用于下游模型验证。"
        )


    # --------------------------------------------------------
    # 4.6 核对MPNN训练摘要的数据边界
    # --------------------------------------------------------

    training_counts = (
        training_summary.get(
            "data_counts",
            {},
        )
    )


    mpnn_supervised_train_rows = int(
        training_counts.get(
            "mpnn_supervised_train_rows",
            -1,
        )
    )

    mpnn_early_stop_rows = int(
        training_counts.get(
            "mpnn_early_stop_rows",
            -1,
        )
    )

    inner_valid_rows_used_by_mpnn = int(
        training_counts.get(
            "inner_ml_valid_rows_used",
            -1,
        )
    )

    outer_test_rows_used_by_mpnn = int(
        training_counts.get(
            "outer_test_rows_used",
            -1,
        )
    )


    if (
        mpnn_supervised_train_rows
        + mpnn_early_stop_rows
        != n_inner_train
    ):
        raise RuntimeError(
            f"{task_id}：MPNN监督训练集"
            "和早停集没有完整组成内层ML训练集。"
        )


    if inner_valid_rows_used_by_mpnn != 0:
        raise RuntimeError(
            f"{task_id}：训练摘要显示"
            "内层ML验证样本参与了MPNN训练。"
        )


    if outer_test_rows_used_by_mpnn != 0:
        raise RuntimeError(
            f"{task_id}：训练摘要显示"
            "外层测试样本参与了MPNN训练。"
        )


    training_leakage = (
        training_summary.get(
            "leakage_control",
            {},
        )
    )


    false_training_leakage_checks = {
        "validation used for backpropagation": [
            (
                "inner_ml_validation_"
                "used_for_backpropagation"
            ),
            (
                "inner_validation_"
                "used_for_backpropagation"
            ),
        ],

        "validation used for early stopping": [
            (
                "inner_ml_validation_"
                "used_for_early_stopping"
            ),
            (
                "inner_validation_"
                "used_for_early_stopping"
            ),
        ],

        "validation used for scheduler": [
            (
                "inner_ml_validation_"
                "used_for_scheduler"
            ),
            (
                "inner_validation_"
                "used_for_scheduler"
            ),
        ],

        "outer test used for backpropagation": [
            (
                "outer_test_used_for_"
                "backpropagation"
            ),
        ],

        "outer test used for early stopping": [
            (
                "outer_test_used_for_"
                "early_stopping"
            ),
        ],

        "outer test used for scheduler": [
            (
                "outer_test_used_for_"
                "scheduler"
            ),
        ],

        "existing outer MPNN features read": [
            "existing_outer_mpnn_features_read",
        ],

        "old global MPNN feature read": [
            "old_global_mpnn_feature_read",
        ],

        "external 10-point holdout read": [
            "external_10point_holdout_read",
        ],
    }


    for (
        check_name,
        aliases,
    ) in false_training_leakage_checks.items():

        require_false_alias(
            mapping=training_leakage,
            candidate_keys=aliases,
            task_id=task_id,
            value_name=check_name,
        )


    # --------------------------------------------------------
    # 4.7 核对清单、索引和目标顺序
    # --------------------------------------------------------

    inner_train_manifest = (
        normalize_columns(
            read_csv_robust(
                paths[
                    "inner_train_manifest"
                ]
            )
        )
    )

    inner_valid_structure_manifest = (
        normalize_columns(
            read_csv_robust(
                paths[
                    "inner_valid_structure_manifest"
                ]
            )
        )
    )


    if len(
        inner_train_manifest
    ) != n_inner_train:
        raise RuntimeError(
            f"{task_id}：内层训练清单"
            "与训练数组行数不一致。"
        )


    if len(
        inner_valid_structure_manifest
    ) != n_inner_valid:
        raise RuntimeError(
            f"{task_id}：内层验证结构清单"
            "与验证数组行数不一致。"
        )


    if "lg_eta" not in (
        inner_train_manifest.columns
    ):
        raise KeyError(
            f"{task_id}：内层训练清单"
            "缺少lg_eta。"
        )


    for forbidden_target_column in [
        "eta",
        "lg_eta",
    ]:
        if forbidden_target_column in (
            inner_valid_structure_manifest.columns
        ):
            raise RuntimeError(
                f"{task_id}：无目标验证结构清单"
                f"意外包含{forbidden_target_column}。"
            )


    if (
        "original_feature_row_idx"
        not in inner_train_manifest.columns
        or "original_feature_row_idx"
        not in inner_valid_structure_manifest.columns
    ):
        raise KeyError(
            f"{task_id}：训练或验证清单"
            "缺少original_feature_row_idx。"
        )


    manifest_y_train = pd.to_numeric(
        inner_train_manifest[
            "lg_eta"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.float32
    )


    maximum_train_target_error = float(
        np.max(
            np.abs(
                manifest_y_train
                - np.asarray(
                    y_inner_train,
                    dtype=np.float32,
                )
            )
        )
    )


    if maximum_train_target_error > 1e-6:
        raise RuntimeError(
            f"{task_id}：训练目标顺序不一致，"
            f"最大误差={maximum_train_target_error}"
        )


    expected_index_keys = {
        (
            "inner_train_relative_to_"
            "outer_train_idx"
        ),
        (
            "inner_valid_relative_to_"
            "outer_train_idx"
        ),
        (
            "mpnn_supervised_train_relative_"
            "to_inner_train_idx"
        ),
        (
            "mpnn_early_stop_relative_"
            "to_inner_train_idx"
        ),
        "inner_train_global_idx",
        "inner_valid_global_idx",
        "mpnn_supervised_train_global_idx",
        "mpnn_early_stop_global_idx",
    }


    with np.load(
        paths[
            "mpnn_internal_indices"
        ],
        allow_pickle=False,
    ) as index_file:

        actual_index_keys = set(
            index_file.files
        )


        if (
            actual_index_keys
            != expected_index_keys
        ):
            raise KeyError(
                f"{task_id}：内部索引键不完整。\n"
                f"实际={sorted(actual_index_keys)}\n"
                f"预期={sorted(expected_index_keys)}"
            )


        inner_train_global_idx = (
            index_file[
                "inner_train_global_idx"
            ].astype(
                np.int64
            )
        )

        inner_valid_global_idx = (
            index_file[
                "inner_valid_global_idx"
            ].astype(
                np.int64
            )
        )

        mpnn_train_global_idx = (
            index_file[
                "mpnn_supervised_train_global_idx"
            ].astype(
                np.int64
            )
        )

        mpnn_stop_global_idx = (
            index_file[
                "mpnn_early_stop_global_idx"
            ].astype(
                np.int64
            )
        )


    if len(
        inner_train_global_idx
    ) != n_inner_train:
        raise RuntimeError(
            f"{task_id}：训练全局索引数量错误。"
        )


    if len(
        inner_valid_global_idx
    ) != n_inner_valid:
        raise RuntimeError(
            f"{task_id}：验证全局索引数量错误。"
        )


    if len(
        mpnn_train_global_idx
    ) != mpnn_supervised_train_rows:
        raise RuntimeError(
            f"{task_id}：MPNN监督训练索引数量错误。"
        )


    if len(
        mpnn_stop_global_idx
    ) != mpnn_early_stop_rows:
        raise RuntimeError(
            f"{task_id}：MPNN早停索引数量错误。"
        )


    for index_name, index_array in {
        "inner_train": inner_train_global_idx,
        "inner_valid": inner_valid_global_idx,
        "mpnn_train": mpnn_train_global_idx,
        "mpnn_stop": mpnn_stop_global_idx,
    }.items():

        if len(
            np.unique(
                index_array
            )
        ) != len(
            index_array
        ):
            raise RuntimeError(
                f"{task_id}：{index_name}"
                "索引中存在重复行。"
            )


    row_overlap_count = int(
        np.intersect1d(
            inner_train_global_idx,
            inner_valid_global_idx,
        ).size
    )

    mpnn_train_stop_overlap = int(
        np.intersect1d(
            mpnn_train_global_idx,
            mpnn_stop_global_idx,
        ).size
    )

    mpnn_train_valid_overlap = int(
        np.intersect1d(
            mpnn_train_global_idx,
            inner_valid_global_idx,
        ).size
    )

    mpnn_stop_valid_overlap = int(
        np.intersect1d(
            mpnn_stop_global_idx,
            inner_valid_global_idx,
        ).size
    )


    if any(
        overlap != 0
        for overlap in [
            row_overlap_count,
            mpnn_train_stop_overlap,
            mpnn_train_valid_overlap,
            mpnn_stop_valid_overlap,
        ]
    ):
        raise RuntimeError(
            f"{task_id}：内部索引存在重叠。"
        )


    mpnn_train_stop_union = np.sort(
        np.concatenate(
            [
                mpnn_train_global_idx,
                mpnn_stop_global_idx,
            ]
        )
    )


    if not np.array_equal(
        mpnn_train_stop_union,
        np.sort(
            inner_train_global_idx
        ),
    ):
        raise RuntimeError(
            f"{task_id}：MPNN监督训练和早停索引"
            "没有完整覆盖内层ML训练集。"
        )


    train_manifest_global_idx = pd.to_numeric(
        inner_train_manifest[
            "original_feature_row_idx"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.int64
    )

    valid_manifest_global_idx = pd.to_numeric(
        inner_valid_structure_manifest[
            "original_feature_row_idx"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.int64
    )


    train_manifest_order_match = bool(
        np.array_equal(
            train_manifest_global_idx,
            inner_train_global_idx,
        )
    )

    valid_manifest_order_match = bool(
        np.array_equal(
            valid_manifest_global_idx,
            inner_valid_global_idx,
        )
    )


    if not train_manifest_order_match:
        raise RuntimeError(
            f"{task_id}：训练清单顺序"
            "与训练全局索引不一致。"
        )


    if not valid_manifest_order_match:
        raise RuntimeError(
            f"{task_id}：验证结构清单顺序"
            "与验证全局索引不一致。"
        )


    train_groups, train_group_column = (
        resolve_il_group_series(
            inner_train_manifest
        )
    )

    valid_groups, valid_group_column = (
        resolve_il_group_series(
            inner_valid_structure_manifest
        )
    )


    if (
        train_group_column
        != valid_group_column
    ):
        raise RuntimeError(
            f"{task_id}：训练和验证清单"
            "使用的IL分组字段不一致。"
        )


    il_group_overlap_count = int(
        np.intersect1d(
            np.unique(
                train_groups
            ),
            np.unique(
                valid_groups
            ),
        ).size
    )


    if (
        split_name == "IL_split"
        and il_group_overlap_count != 0
    ):
        raise RuntimeError(
            f"{task_id}：IL_split训练与验证"
            f"存在{il_group_overlap_count}个IL分组重叠。"
        )


    # --------------------------------------------------------
    # 4.8 返回审计记录，不保留大数组
    # --------------------------------------------------------

    audit_row = {
        "Task_id": task_id,

        "Split": split_name,

        "Outer_fold": int(
            outer_fold
        ),

        "Inner_fold": int(
            inner_fold
        ),

        "Inner_train_rows": (
            n_inner_train
        ),

        "Inner_valid_rows": (
            n_inner_valid
        ),

        "Outer_train_rows": (
            outer_train_rows
        ),

        "Feature_dim": 130,

        "X_train_dtype": str(
            X_inner_train.dtype
        ),

        "X_valid_dtype": str(
            X_inner_valid.dtype
        ),

        "y_train_dtype": str(
            y_inner_train.dtype
        ),

        "y_valid_dtype": str(
            y_inner_valid.dtype
        ),

        "MPNN_supervised_train_rows": (
            mpnn_supervised_train_rows
        ),

        "MPNN_early_stop_rows": (
            mpnn_early_stop_rows
        ),

        "Row_overlap_count": (
            row_overlap_count
        ),

        "MPNN_train_stop_overlap": (
            mpnn_train_stop_overlap
        ),

        "MPNN_train_valid_overlap": (
            mpnn_train_valid_overlap
        ),

        "MPNN_stop_valid_overlap": (
            mpnn_stop_valid_overlap
        ),

        "IL_group_column": (
            train_group_column
        ),

        "Train_unique_IL": int(
            pd.Series(
                train_groups
            ).nunique()
        ),

        "Valid_unique_IL": int(
            pd.Series(
                valid_groups
            ).nunique()
        ),

        "IL_group_overlap_count": (
            il_group_overlap_count
        ),

        "Train_target_max_error": (
            maximum_train_target_error
        ),

        "Train_manifest_order_match": (
            train_manifest_order_match
        ),

        "Valid_manifest_order_match": (
            valid_manifest_order_match
        ),

        "All_shape_checks_passed": bool(
            all(
                shape_checks.values()
            )
        ),

        "All_dtype_checks_passed": bool(
            all(
                dtype_checks.values()
            )
        ),

        "All_path_checks_passed": bool(
            all(
                path_checks.values()
            )
        ),

        "All_hash_checks_passed": bool(
            all(
                hash_checks.values()
            )
        ),

        "Output_audit_passed": True,

        "Training_summary_passed": True,

        "Validation_saved_for_downstream_ML": (
            validation_saved_for_ml
        ),

        "Forbidden_input_detected": False,

        "Old_global_feature_read": False,

        "Outer_train_feature_read": False,

        "Outer_test_feature_read": False,

        "Outer_test_target_read": False,

        "External_10points_read": False,

        "X_inner_train_sha256": (
            actual_hashes[
                "X_inner_train_mpnn"
            ]
        ),

        "X_inner_valid_sha256": (
            actual_hashes[
                "X_inner_valid_mpnn"
            ]
        ),

        "y_inner_train_sha256": (
            actual_hashes[
                "y_inner_train_lg_eta"
            ]
        ),

        "y_inner_valid_sha256": (
            actual_hashes[
                "y_inner_valid_lg_eta"
            ]
        ),
    }


    task_key = (
        split_name,
        int(
            outer_fold
        ),
        int(
            inner_fold
        ),
    )


    INNER_TUNING_BUNDLE_PATHS[
        task_key
    ] = paths


    del (
        arrays,
        X_inner_train,
        X_inner_valid,
        y_inner_train,
        y_inner_valid,
        inner_train_manifest,
        inner_valid_structure_manifest,
        inner_train_global_idx,
        inner_valid_global_idx,
        mpnn_train_global_idx,
        mpnn_stop_global_idx,
        train_groups,
        valid_groups,
    )

    gc.collect()


    return audit_row


# ============================================================
# 5. 逐套审计30套内层调参输入
# ============================================================

inner_array_audit_rows = []


for split_name in SPLIT_ORDER:
    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):
        for inner_fold in range(
            1,
            N_INNER_FOLDS + 1,
        ):
            audit_row = (
                audit_one_inner_tuning_task(
                    split_name=split_name,
                    outer_fold=outer_fold,
                    inner_fold=inner_fold,
                )
            )


            inner_array_audit_rows.append(
                audit_row
            )


df_inner_array_audit = (
    pd.DataFrame(
        inner_array_audit_rows
    )
    .sort_values(
        [
            "Split",
            "Outer_fold",
            "Inner_fold",
        ],
        key=lambda series: (
            series.map(
                {
                    "Data_split": 0,
                    "IL_split": 1,
                }
            )
            if series.name == "Split"
            else series
        ),
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 6. 30套任务总体检查
# ============================================================

if len(
    df_inner_array_audit
) != EXPECTED_TASK_COUNT:
    raise RuntimeError(
        "数组审计任务数错误："
        f"实际={len(df_inner_array_audit)}，"
        f"预期={EXPECTED_TASK_COUNT}"
    )


task_count_by_split = (
    df_inner_array_audit
    .groupby(
        "Split"
    )
    .size()
    .to_dict()
)


if task_count_by_split != {
    "Data_split": 15,
    "IL_split": 15,
}:
    raise RuntimeError(
        "Data_split或IL_split任务数量错误。"
    )


tasks_per_outer_fold = (
    df_inner_array_audit
    .groupby(
        [
            "Split",
            "Outer_fold",
        ]
    )
    .size()
)


if not tasks_per_outer_fold.eq(
    3
).all():
    raise RuntimeError(
        "部分外层折不是3套内层任务。"
    )


outer_row_consistency = (
    df_inner_array_audit
    .groupby(
        [
            "Split",
            "Outer_fold",
        ]
    )[
        "Outer_train_rows"
    ]
    .nunique()
)


if not outer_row_consistency.eq(
    1
).all():
    raise RuntimeError(
        "同一外层折的3套内层任务"
        "对应的外层训练总行数不一致。"
    )


if not (
    df_inner_array_audit[
        "Feature_dim"
    ].eq(
        130
    ).all()
):
    raise RuntimeError(
        "部分内层任务的特征维度不是130。"
    )


if int(
    df_inner_array_audit[
        "Row_overlap_count"
    ].sum()
) != 0:
    raise RuntimeError(
        "内层训练集和验证集存在行重叠。"
    )


if int(
    df_inner_array_audit[
        "MPNN_train_stop_overlap"
    ].sum()
) != 0:
    raise RuntimeError(
        "MPNN监督训练集和早停集存在重叠。"
    )


if int(
    df_inner_array_audit[
        "MPNN_train_valid_overlap"
    ].sum()
) != 0:
    raise RuntimeError(
        "MPNN监督训练集和内层验证集存在重叠。"
    )


if int(
    df_inner_array_audit[
        "MPNN_stop_valid_overlap"
    ].sum()
) != 0:
    raise RuntimeError(
        "MPNN早停集和内层验证集存在重叠。"
    )


il_split_group_overlap_total = int(
    df_inner_array_audit.loc[
        df_inner_array_audit[
            "Split"
        ].eq(
            "IL_split"
        ),
        "IL_group_overlap_count",
    ].sum()
)


if il_split_group_overlap_total != 0:
    raise RuntimeError(
        "IL_split中存在IL分组重叠。"
    )


required_all_true_columns = [
    "Train_manifest_order_match",
    "Valid_manifest_order_match",
    "All_shape_checks_passed",
    "All_dtype_checks_passed",
    "All_path_checks_passed",
    "All_hash_checks_passed",
    "Output_audit_passed",
    "Training_summary_passed",
    "Validation_saved_for_downstream_ML",
]


for column in required_all_true_columns:
    if not df_inner_array_audit[
        column
    ].all():
        raise RuntimeError(
            f"数组审计字段{column}"
            "没有全部通过。"
        )


if df_inner_array_audit[
    "Forbidden_input_detected"
].any():
    raise RuntimeError(
        "读取过程中发现禁止输入。"
    )


if len(
    INNER_TUNING_BUNDLE_PATHS
) != EXPECTED_TASK_COUNT:
    raise RuntimeError(
        "内层任务路径缓存数量不是30。"
    )


# ============================================================
# 7. 保存数组审计表与manifest
# ============================================================

df_inner_array_audit.to_csv(
    INNER_ARRAY_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)


summary_table = (
    df_inner_array_audit
    .groupby(
        "Split",
        as_index=False,
    )
    .agg(
        Outer_fold_count=(
            "Outer_fold",
            "nunique",
        ),

        Inner_task_count=(
            "Task_id",
            "size",
        ),

        Min_inner_train_rows=(
            "Inner_train_rows",
            "min",
        ),

        Max_inner_train_rows=(
            "Inner_train_rows",
            "max",
        ),

        Min_inner_valid_rows=(
            "Inner_valid_rows",
            "min",
        ),

        Max_inner_valid_rows=(
            "Inner_valid_rows",
            "max",
        ),

        Min_outer_train_rows=(
            "Outer_train_rows",
            "min",
        ),

        Max_outer_train_rows=(
            "Outer_train_rows",
            "max",
        ),

        Row_overlap_total=(
            "Row_overlap_count",
            "sum",
        ),

        MPNN_train_stop_overlap_total=(
            "MPNN_train_stop_overlap",
            "sum",
        ),

        IL_group_overlap_total=(
            "IL_group_overlap_count",
            "sum",
        ),
    )
)


array_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "inner_tuning_array_audit_passed"
    ),

    "splits": (
        SPLIT_ORDER
    ),

    "models": (
        MODEL_ORDER
    ),

    "task_count": int(
        len(
            df_inner_array_audit
        )
    ),

    "task_count_by_split": (
        task_count_by_split
    ),

    "outer_folds_per_split": int(
        N_OUTER_FOLDS
    ),

    "inner_folds_per_outer_fold": int(
        N_INNER_FOLDS
    ),

    "feature_dim": 130,

    "array_dtype": "float32",

    "row_ranges": {
        "inner_train_min": int(
            df_inner_array_audit[
                "Inner_train_rows"
            ].min()
        ),

        "inner_train_max": int(
            df_inner_array_audit[
                "Inner_train_rows"
            ].max()
        ),

        "inner_valid_min": int(
            df_inner_array_audit[
                "Inner_valid_rows"
            ].min()
        ),

        "inner_valid_max": int(
            df_inner_array_audit[
                "Inner_valid_rows"
            ].max()
        ),

        "outer_train_min": int(
            df_inner_array_audit[
                "Outer_train_rows"
            ].min()
        ),

        "outer_train_max": int(
            df_inner_array_audit[
                "Outer_train_rows"
            ].max()
        ),
    },

    "overlap_checks": {
        "inner_train_valid_total": int(
            df_inner_array_audit[
                "Row_overlap_count"
            ].sum()
        ),

        "mpnn_train_stop_total": int(
            df_inner_array_audit[
                "MPNN_train_stop_overlap"
            ].sum()
        ),

        "mpnn_train_inner_valid_total": int(
            df_inner_array_audit[
                "MPNN_train_valid_overlap"
            ].sum()
        ),

        "mpnn_stop_inner_valid_total": int(
            df_inner_array_audit[
                "MPNN_stop_valid_overlap"
            ].sum()
        ),

        "IL_split_group_overlap_total": (
            il_split_group_overlap_total
        ),
    },

    "integrity_checks": {
        "all_shape_checks_passed": True,

        "all_dtype_checks_passed": True,

        "all_path_checks_passed": True,

        "all_hash_checks_passed": True,

        "all_train_target_orders_match": True,

        "all_valid_structure_orders_match": True,

        "all_output_audits_passed": True,

        "all_training_summaries_passed": True,
    },

    "leakage_control": {
        "inner_validation_used_for_"
        "mpnn_training": False,

        "inner_validation_used_for_"
        "mpnn_early_stopping": False,

        "inner_validation_used_for_"
        "mpnn_scheduler": False,

        "inner_validation_used_for_"
        "mpnn_feature_extraction": False,

        "inner_validation_saved_for_"
        "downstream_ml_validation": True,

        "outer_train_mpnn_feature_read": False,

        "outer_test_mpnn_feature_read": False,

        "outer_test_target_read": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "memory_policy": {
        "all_30_tasks_kept_in_memory": False,

        "task_arrays_loaded_on_demand": True,

        "path_bundle_count": int(
            len(
                INNER_TUNING_BUNDLE_PATHS
            )
        ),
    },

    "records": (
        df_inner_array_audit
        .to_dict(
            orient="records"
        )
    ),

    "outputs": {
        "array_audit": str(
            INNER_ARRAY_AUDIT_PATH
        ),

        "array_manifest": str(
            INNER_ARRAY_MANIFEST_PATH
        ),
    },
}


save_json_atomic(
    data=array_manifest,
    output_path=INNER_ARRAY_MANIFEST_PATH,
)


# ============================================================
# 8. 输出检查
# ============================================================

print("=" * 104)
print("Cell 1 30套内层调参数组、索引和数据边界审计通过 ✓")
print("=" * 104)

print(
    "inner task bundles checked       =",
    len(
        df_inner_array_audit
    ),
)

print(
    "task counts by split             =",
    task_count_by_split,
)

print(
    "feature dimensions               =",
    sorted(
        df_inner_array_audit[
            "Feature_dim"
        ].unique().tolist()
    ),
)

print(
    "array dtypes                     =",
    sorted(
        set(
            df_inner_array_audit[
                [
                    "X_train_dtype",
                    "X_valid_dtype",
                    "y_train_dtype",
                    "y_valid_dtype",
                ]
            ]
            .astype(str)
            .to_numpy()
            .reshape(-1)
            .tolist()
        )
    ),
)

print(
    "inner train row range            =",
    (
        int(
            df_inner_array_audit[
                "Inner_train_rows"
            ].min()
        ),
        int(
            df_inner_array_audit[
                "Inner_train_rows"
            ].max()
        ),
    ),
)

print(
    "inner valid row range            =",
    (
        int(
            df_inner_array_audit[
                "Inner_valid_rows"
            ].min()
        ),
        int(
            df_inner_array_audit[
                "Inner_valid_rows"
            ].max()
        ),
    ),
)

print(
    "inner train/valid overlap total  =",
    int(
        df_inner_array_audit[
            "Row_overlap_count"
        ].sum()
    ),
)

print(
    "MPNN train/stop overlap total    =",
    int(
        df_inner_array_audit[
            "MPNN_train_stop_overlap"
        ].sum()
    ),
)

print(
    "MPNN train/valid overlap total   =",
    int(
        df_inner_array_audit[
            "MPNN_train_valid_overlap"
        ].sum()
    ),
)

print(
    "MPNN stop/valid overlap total    =",
    int(
        df_inner_array_audit[
            "MPNN_stop_valid_overlap"
        ].sum()
    ),
)

print(
    "IL_split group overlap total     =",
    il_split_group_overlap_total,
)

print(
    "train target order checks passed =",
    int(
        df_inner_array_audit[
            "Train_manifest_order_match"
        ].sum()
    ),
    "/",
    EXPECTED_TASK_COUNT,
)

print(
    "valid order checks passed        =",
    int(
        df_inner_array_audit[
            "Valid_manifest_order_match"
        ].sum()
    ),
    "/",
    EXPECTED_TASK_COUNT,
)

print(
    "array hash checks passed         =",
    int(
        df_inner_array_audit[
            "All_hash_checks_passed"
        ].sum()
    ),
    "/",
    EXPECTED_TASK_COUNT,
)

print(
    "output audits passed             =",
    int(
        df_inner_array_audit[
            "Output_audit_passed"
        ].sum()
    ),
    "/",
    EXPECTED_TASK_COUNT,
)

print(
    "all 30 tasks kept in memory      =",
    False,
)

print(
    "path bundles cached              =",
    len(
        INNER_TUNING_BUNDLE_PATHS
    ),
)

print(
    "forbidden input detected         =",
    bool(
        df_inner_array_audit[
            "Forbidden_input_detected"
        ].any()
    ),
)

print(
    "outer train MPNN feature read    =",
    False,
)

print(
    "outer test MPNN feature read     =",
    False,
)

print(
    "outer test target read           =",
    False,
)

print(
    "old global MPNN feature read     =",
    False,
)

print(
    "external 10 points read          =",
    False,
)


print(
    "\n内层调参数组汇总："
)

display(
    summary_table
)


print(
    "\n数组与边界审计表："
)

print(
    INNER_ARRAY_AUDIT_PATH
)


print(
    "\n数组审计manifest："
)

print(
    INNER_ARRAY_MANIFEST_PATH
)

Cell 1 30套内层调参数组、索引和数据边界审计通过 ✓
inner task bundles checked       = 30
task counts by split             = {'Data_split': 15, 'IL_split': 15}
feature dimensions               = [130]
array dtypes                     = ['float32']
inner train row range            = (8648, 8649)
inner valid row range            = (4324, 4325)
inner train/valid overlap total  = 0
MPNN train/stop overlap total    = 0
MPNN train/valid overlap total   = 0
MPNN stop/valid overlap total    = 0
IL_split group overlap total     = 0
train target order checks passed = 30 / 30
valid order checks passed        = 30 / 30
array hash checks passed         = 30 / 30
output audits passed             = 30 / 30
all 30 tasks kept in memory      = False
path bundles cached              = 30
forbidden input detected         = False
outer train MPNN feature read    = False
outer test MPNN feature read     = False
outer test target read           = False
old global MPNN feature read     = False
external 10 points read          = F

,Split,Outer_fold_count,Inner_task_count,Min_inner_train_rows,Max_inner_train_rows,Min_inner_valid_rows,Max_inner_valid_rows,Min_outer_train_rows,Max_outer_train_rows,Row_overlap_total,MPNN_train_stop_overlap_total,IL_group_overlap_total
0,Data_split,5,15,8648,8649,4324,4325,12972,12973,0,0,14707
1,IL_split,5,15,8648,8649,4324,4325,12972,12973,0,0,0



数组与边界审计表：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\inner_tuning_array_audit_Data_IL.csv

数组审计manifest：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\inner_tuning_array_manifest_Data_IL.json


In [3]:
# Cell 2 | 定义统一评价指标、折内标准化规则及树模型搜索空间


METRIC_ORDER = [
    "R2",
    "RMSE",
    "MAE",
    "Pearson_r",
]


TREE_MODEL_ORDER = [
    "ExtraTrees",
    "LightGBM",
]


OPTUNA_OBJECTIVE_METRIC = "R2"

OPTUNA_DIRECTION = "maximize"

INNER_FOLD_AGGREGATION = (
    "equal_weight_mean_across_3_inner_folds"
)


SCALING_INTERFACE_AUDIT_PATH = (
    TUNING_LOG_DIR
    / "downstream_scaling_interface_audit_Data_IL.json"
)


TREE_TUNING_DEFINITION_PATH = (
    TUNING_LOG_DIR
    / "tree_model_tuning_definition_Data_IL.json"
)


# ============================================================
# 1. 评价指标
# ============================================================

def safe_pearson_r(
    y_true,
    y_pred,
):
    """安全计算Pearson相关系数。"""

    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    ).reshape(-1)

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64,
    ).reshape(-1)


    if len(
        y_true
    ) != len(
        y_pred
    ):
        raise ValueError(
            "计算Pearson r时，"
            "真实值与预测值长度不一致。"
        )


    if len(
        y_true
    ) < 2:
        return np.nan


    if not (
        np.isfinite(
            y_true
        ).all()
        and np.isfinite(
            y_pred
        ).all()
    ):
        raise ValueError(
            "计算Pearson r时发现NaN或无穷值。"
        )


    if (
        np.std(
            y_true
        ) == 0
        or np.std(
            y_pred
        ) == 0
    ):
        return np.nan


    return float(
        pearsonr(
            y_true,
            y_pred,
        )[0]
    )


def calculate_metrics(
    y_true,
    y_pred,
):
    """计算R²、RMSE、MAE和Pearson r。"""

    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    ).reshape(-1)

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64,
    ).reshape(-1)


    if len(
        y_true
    ) != len(
        y_pred
    ):
        raise ValueError(
            "计算评价指标时，"
            "真实值与预测值长度不一致。"
        )


    if len(
        y_true
    ) < 2:
        raise ValueError(
            "计算评价指标至少需要2个样本。"
        )


    if not (
        np.isfinite(
            y_true
        ).all()
        and np.isfinite(
            y_pred
        ).all()
    ):
        raise ValueError(
            "计算评价指标时发现NaN或无穷值。"
        )


    metrics = {
        "R2": float(
            r2_score(
                y_true,
                y_pred,
            )
        ),

        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred,
                )
            )
        ),

        "MAE": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "Pearson_r": (
            safe_pearson_r(
                y_true,
                y_pred,
            )
        ),
    }


    if set(
        metrics.keys()
    ) != set(
        METRIC_ORDER
    ):
        raise RuntimeError(
            "评价指标字段与METRIC_ORDER不一致。"
        )


    return metrics


def aggregate_inner_fold_metrics(
    fold_metric_records,
):
    """
    对同一组参数在3个内层折上的指标进行等权平均。

    Optuna只使用Mean_R2作为优化目标；
    其余指标用于结果记录和辅助判断。
    """

    if len(
        fold_metric_records
    ) != N_INNER_FOLDS:
        raise ValueError(
            "内层折指标记录数不是3。"
        )


    metric_dataframe = pd.DataFrame(
        fold_metric_records
    )


    missing_columns = [
        metric_name
        for metric_name in METRIC_ORDER
        if metric_name not in (
            metric_dataframe.columns
        )
    ]


    if missing_columns:
        raise KeyError(
            "内层折指标记录缺少字段："
            f"{missing_columns}"
        )


    aggregated = {}


    for metric_name in METRIC_ORDER:
        metric_values = pd.to_numeric(
            metric_dataframe[
                metric_name
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.float64
        )


        if not np.isfinite(
            metric_values
        ).all():
            raise ValueError(
                f"{metric_name}的内层折结果"
                "存在NaN或无穷值。"
            )


        aggregated[
            f"Mean_{metric_name}"
        ] = float(
            np.mean(
                metric_values
            )
        )


        aggregated[
            f"Std_{metric_name}"
        ] = float(
            np.std(
                metric_values,
                ddof=0,
            )
        )


    aggregated[
        "Objective_value"
    ] = float(
        aggregated[
            "Mean_R2"
        ]
    )


    return aggregated


# ============================================================
# 2. 随机种子
# ============================================================

def make_trial_seed(
    split_name,
    outer_fold,
    model_name,
    trial_number,
    inner_fold,
):
    """为每个模型、trial和内层折生成稳定随机种子。"""

    split_offsets = {
        "Data_split": 0,
        "IL_split": 1_000_000,
    }


    model_offsets = {
        "ExtraTrees": 10_000,
        "LightGBM": 20_000,
        "Transformer": 30_000,
    }


    if split_name not in split_offsets:
        raise KeyError(
            f"未知划分方式：{split_name}"
        )


    if model_name not in model_offsets:
        raise KeyError(
            f"未知模型：{model_name}"
        )


    seed = (
        int(
            TUNING_RANDOM_SEED
        )
        + int(
            split_offsets[
                split_name
            ]
        )
        + int(
            outer_fold
        )
        * 1_000
        + int(
            model_offsets[
                model_name
            ]
        )
        + int(
            trial_number
        )
        * 10
        + int(
            inner_fold
        )
    )


    return int(
        seed
    )


# ============================================================
# 3. 下游模型折内标准化
# ============================================================

def get_scaler_fitted_rows(
    scaler,
):
    """读取StandardScaler实际拟合样本数。"""

    if not hasattr(
        scaler,
        "n_samples_seen_",
    ):
        raise AttributeError(
            "StandardScaler尚未拟合。"
        )


    fitted_count = np.asarray(
        scaler.n_samples_seen_
    )


    if fitted_count.ndim == 0:
        return int(
            fitted_count.item()
        )


    fitted_count = fitted_count.reshape(
        -1
    )


    if not np.all(
        fitted_count
        == fitted_count[
            0
        ]
    ):
        raise RuntimeError(
            "StandardScaler不同特征记录的"
            "拟合样本数不一致。"
        )


    return int(
        fitted_count[
            0
        ]
    )


def prepare_downstream_inner_fold(
    split_name,
    outer_fold,
    inner_fold,
):
    """
    读取一套内层折专属MPNN表征。

    StandardScaler只在当前内层训练特征上拟合，
    再用于当前内层训练和验证特征。

    lgη目标不进行标准化。
    """

    bundle = load_inner_tuning_arrays(
        split_name=split_name,
        outer_fold=outer_fold,
        inner_fold=inner_fold,
    )


    X_inner_train_raw = np.asarray(
        bundle[
            "X_inner_train"
        ],
        dtype=np.float32,
    )

    X_inner_valid_raw = np.asarray(
        bundle[
            "X_inner_valid"
        ],
        dtype=np.float32,
    )

    y_inner_train = np.asarray(
        bundle[
            "y_inner_train"
        ],
        dtype=np.float32,
    ).reshape(-1)

    y_inner_valid = np.asarray(
        bundle[
            "y_inner_valid"
        ],
        dtype=np.float32,
    ).reshape(-1)


    if (
        X_inner_train_raw.ndim != 2
        or X_inner_valid_raw.ndim != 2
    ):
        raise ValueError(
            "内层训练或验证特征不是二维矩阵。"
        )


    if (
        X_inner_train_raw.shape[
            1
        ] != 130
        or X_inner_valid_raw.shape[
            1
        ] != 130
    ):
        raise ValueError(
            "内层MPNN表征维度不是130。"
        )


    if (
        len(
            X_inner_train_raw
        )
        != len(
            y_inner_train
        )
        or len(
            X_inner_valid_raw
        )
        != len(
            y_inner_valid
        )
    ):
        raise ValueError(
            "内层特征和目标行数不一致。"
        )


    if not all(
        np.isfinite(
            array
        ).all()
        for array in [
            X_inner_train_raw,
            X_inner_valid_raw,
            y_inner_train,
            y_inner_valid,
        ]
    ):
        raise ValueError(
            "内层调参数组中存在NaN或无穷值。"
        )


    downstream_scaler = StandardScaler()


    X_inner_train_scaled = (
        downstream_scaler
        .fit_transform(
            X_inner_train_raw
        )
        .astype(
            np.float32
        )
    )


    X_inner_valid_scaled = (
        downstream_scaler
        .transform(
            X_inner_valid_raw
        )
        .astype(
            np.float32
        )
    )


    scaler_fitted_rows = (
        get_scaler_fitted_rows(
            downstream_scaler
        )
    )


    if scaler_fitted_rows != len(
        X_inner_train_raw
    ):
        raise RuntimeError(
            "StandardScaler拟合样本数"
            "与内层训练样本数不一致。"
        )


    if not (
        np.isfinite(
            X_inner_train_scaled
        ).all()
        and np.isfinite(
            X_inner_valid_scaled
        ).all()
    ):
        raise ValueError(
            "标准化后出现NaN或无穷值。"
        )


    if not np.array_equal(
        y_inner_train,
        np.asarray(
            bundle[
                "y_inner_train"
            ],
            dtype=np.float32,
        ).reshape(-1),
    ):
        raise RuntimeError(
            "内层训练目标在预处理过程中发生改变。"
        )


    if not np.array_equal(
        y_inner_valid,
        np.asarray(
            bundle[
                "y_inner_valid"
            ],
            dtype=np.float32,
        ).reshape(-1),
    ):
        raise RuntimeError(
            "内层验证目标在预处理过程中发生改变。"
        )


    return {
        "X_train": (
            X_inner_train_scaled
        ),

        "X_valid": (
            X_inner_valid_scaled
        ),

        "y_train": (
            y_inner_train
        ),

        "y_valid": (
            y_inner_valid
        ),

        "scaler": (
            downstream_scaler
        ),

        "scaler_fitted_rows": (
            scaler_fitted_rows
        ),

        "train_rows": int(
            len(
                X_inner_train_scaled
            )
        ),

        "valid_rows": int(
            len(
                X_inner_valid_scaled
            )
        ),

        "feature_dim": int(
            X_inner_train_scaled.shape[
                1
            ]
        ),

        "validation_used_for_scaler_fit": False,

        "target_scaled": False,

        "paths": (
            bundle[
                "paths"
            ]
        ),
    }


# ============================================================
# 4. ExtraTrees搜索空间
# ============================================================

def decode_max_features(
    value,
):
    """把Optuna保存的字符串转换为sklearn参数。"""

    mapping = {
        "sqrt": "sqrt",
        "log2": "log2",
        "0.50": 0.50,
        "0.75": 0.75,
        "1.00": 1.00,
    }


    if value not in mapping:
        raise KeyError(
            f"未知max_features：{value}"
        )


    return mapping[
        value
    ]


def suggest_extratrees_parameters(
    trial,
):
    """定义ExtraTrees的Optuna搜索空间。"""

    bootstrap = trial.suggest_categorical(
        "bootstrap",
        [
            False,
            True,
        ],
    )


    search_parameters = {
        "n_estimators": (
            trial.suggest_categorical(
                "n_estimators",
                [
                    200,
                    300,
                    450,
                    600,
                    800,
                ],
            )
        ),

        "max_depth": (
            trial.suggest_categorical(
                "max_depth",
                [
                    None,
                    15,
                    20,
                    25,
                    30,
                    40,
                ],
            )
        ),

        "min_samples_split": (
            trial.suggest_int(
                "min_samples_split",
                2,
                10,
            )
        ),

        "min_samples_leaf": (
            trial.suggest_int(
                "min_samples_leaf",
                1,
                5,
            )
        ),

        "max_features": (
            trial.suggest_categorical(
                "max_features",
                [
                    "sqrt",
                    "log2",
                    "0.50",
                    "0.75",
                    "1.00",
                ],
            )
        ),

        "bootstrap": (
            bootstrap
        ),
    }


    if bootstrap:
        search_parameters[
            "max_samples"
        ] = trial.suggest_float(
            "max_samples",
            0.65,
            1.00,
        )


    return search_parameters


def extratrees_search_to_formal(
    search_parameters,
    seed,
):
    """将ExtraTrees搜索参数转换为正式模型参数。"""

    formal_parameters = {
        "n_estimators": int(
            search_parameters[
                "n_estimators"
            ]
        ),

        "max_depth": (
            None
            if search_parameters[
                "max_depth"
            ] is None
            else int(
                search_parameters[
                    "max_depth"
                ]
            )
        ),

        "min_samples_split": int(
            search_parameters[
                "min_samples_split"
            ]
        ),

        "min_samples_leaf": int(
            search_parameters[
                "min_samples_leaf"
            ]
        ),

        "max_features": (
            decode_max_features(
                search_parameters[
                    "max_features"
                ]
            )
        ),

        "bootstrap": bool(
            search_parameters[
                "bootstrap"
            ]
        ),

        "criterion": "squared_error",

        "random_state": int(
            seed
        ),

        "n_jobs": -1,
    }


    if formal_parameters[
        "bootstrap"
    ]:
        formal_parameters[
            "max_samples"
        ] = float(
            search_parameters[
                "max_samples"
            ]
        )


    return formal_parameters


def build_extratrees_from_trial(
    trial,
    seed,
):
    """根据Optuna trial创建ExtraTrees。"""

    search_parameters = (
        suggest_extratrees_parameters(
            trial
        )
    )


    formal_parameters = (
        extratrees_search_to_formal(
            search_parameters=(
                search_parameters
            ),
            seed=seed,
        )
    )


    model = ExtraTreesRegressor(
        **formal_parameters
    )


    return (
        model,
        search_parameters,
        formal_parameters,
    )


# ============================================================
# 5. LightGBM搜索空间
# ============================================================

def suggest_lightgbm_parameters(
    trial,
):
    """定义LightGBM的Optuna搜索空间。"""

    return {
        "n_estimators": (
            trial.suggest_int(
                "n_estimators",
                200,
                1000,
                step=50,
            )
        ),

        "learning_rate": (
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.15,
                log=True,
            )
        ),

        "max_depth": (
            trial.suggest_categorical(
                "max_depth",
                [
                    -1,
                    6,
                    8,
                    10,
                    12,
                    16,
                ],
            )
        ),

        "num_leaves": (
            trial.suggest_int(
                "num_leaves",
                15,
                127,
                step=4,
            )
        ),

        "min_child_samples": (
            trial.suggest_int(
                "min_child_samples",
                5,
                60,
                step=5,
            )
        ),

        "subsample": (
            trial.suggest_float(
                "subsample",
                0.65,
                1.00,
            )
        ),

        "colsample_bytree": (
            trial.suggest_float(
                "colsample_bytree",
                0.60,
                1.00,
            )
        ),

        "reg_alpha": (
            trial.suggest_float(
                "reg_alpha",
                1e-8,
                2.0,
                log=True,
            )
        ),

        "reg_lambda": (
            trial.suggest_float(
                "reg_lambda",
                1e-8,
                5.0,
                log=True,
            )
        ),

        "min_split_gain": (
            trial.suggest_float(
                "min_split_gain",
                0.0,
                0.20,
            )
        ),
    }


def lightgbm_search_to_formal(
    search_parameters,
    seed,
):
    """将LightGBM搜索参数转换为正式模型参数。"""

    return {
        "n_estimators": int(
            search_parameters[
                "n_estimators"
            ]
        ),

        "learning_rate": float(
            search_parameters[
                "learning_rate"
            ]
        ),

        "max_depth": int(
            search_parameters[
                "max_depth"
            ]
        ),

        "num_leaves": int(
            search_parameters[
                "num_leaves"
            ]
        ),

        "min_child_samples": int(
            search_parameters[
                "min_child_samples"
            ]
        ),

        "subsample": float(
            search_parameters[
                "subsample"
            ]
        ),

        "subsample_freq": 1,

        "colsample_bytree": float(
            search_parameters[
                "colsample_bytree"
            ]
        ),

        "reg_alpha": float(
            search_parameters[
                "reg_alpha"
            ]
        ),

        "reg_lambda": float(
            search_parameters[
                "reg_lambda"
            ]
        ),

        "min_split_gain": float(
            search_parameters[
                "min_split_gain"
            ]
        ),

        "objective": "regression",

        "random_state": int(
            seed
        ),

        "n_jobs": -1,

        "verbosity": -1,

        "deterministic": True,

        "force_col_wise": True,
    }


def build_lightgbm_from_trial(
    trial,
    seed,
):
    """根据Optuna trial创建LightGBM。"""

    search_parameters = (
        suggest_lightgbm_parameters(
            trial
        )
    )


    formal_parameters = (
        lightgbm_search_to_formal(
            search_parameters=(
                search_parameters
            ),
            seed=seed,
        )
    )


    model = LGBMRegressor(
        **formal_parameters
    )


    return (
        model,
        search_parameters,
        formal_parameters,
    )


def build_tree_model_from_trial(
    model_name,
    trial,
    seed,
):
    """统一创建ExtraTrees或LightGBM模型。"""

    if model_name == "ExtraTrees":
        return build_extratrees_from_trial(
            trial=trial,
            seed=seed,
        )


    if model_name == "LightGBM":
        return build_lightgbm_from_trial(
            trial=trial,
            seed=seed,
        )


    raise KeyError(
        f"未知树模型：{model_name}"
    )


# ============================================================
# 6. 搜索空间定义
# ============================================================

TREE_SEARCH_SPACE_DESCRIPTION = {
    "ExtraTrees": {
        "trial_budget_per_outer_fold": int(
            N_TRIALS_PER_OUTER_FOLD[
                "ExtraTrees"
            ]
        ),

        "parameters": {
            "n_estimators": [
                200,
                300,
                450,
                600,
                800,
            ],

            "max_depth": [
                None,
                15,
                20,
                25,
                30,
                40,
            ],

            "min_samples_split": [
                2,
                10,
            ],

            "min_samples_leaf": [
                1,
                5,
            ],

            "max_features": [
                "sqrt",
                "log2",
                0.50,
                0.75,
                1.00,
            ],

            "bootstrap": [
                False,
                True,
            ],

            "max_samples_when_bootstrap": [
                0.65,
                1.00,
            ],
        },
    },

    "LightGBM": {
        "trial_budget_per_outer_fold": int(
            N_TRIALS_PER_OUTER_FOLD[
                "LightGBM"
            ]
        ),

        "parameters": {
            "n_estimators": {
                "minimum": 200,
                "maximum": 1000,
                "step": 50,
            },

            "learning_rate": {
                "minimum": 0.01,
                "maximum": 0.15,
                "log": True,
            },

            "max_depth": [
                -1,
                6,
                8,
                10,
                12,
                16,
            ],

            "num_leaves": {
                "minimum": 15,
                "maximum": 127,
                "step": 4,
            },

            "min_child_samples": {
                "minimum": 5,
                "maximum": 60,
                "step": 5,
            },

            "subsample": [
                0.65,
                1.00,
            ],

            "colsample_bytree": [
                0.60,
                1.00,
            ],

            "reg_alpha": {
                "minimum": 1e-8,
                "maximum": 2.0,
                "log": True,
            },

            "reg_lambda": {
                "minimum": 1e-8,
                "maximum": 5.0,
                "log": True,
            },

            "min_split_gain": [
                0.0,
                0.20,
            ],
        },
    },
}


# ============================================================
# 7. 标准化接口测试
# ============================================================

TEST_SCALING_SPLIT = "Data_split"

TEST_SCALING_OUTER_FOLD = 1

TEST_SCALING_INNER_FOLD = 1


test_prepared_fold = (
    prepare_downstream_inner_fold(
        split_name=(
            TEST_SCALING_SPLIT
        ),
        outer_fold=(
            TEST_SCALING_OUTER_FOLD
        ),
        inner_fold=(
            TEST_SCALING_INNER_FOLD
        ),
    )
)


test_training_mean_max_abs = float(
    np.max(
        np.abs(
            np.mean(
                test_prepared_fold[
                    "X_train"
                ],
                axis=0,
            )
        )
    )
)


if test_training_mean_max_abs > 1e-4:
    raise RuntimeError(
        "标准化后的内层训练特征"
        "均值偏离0过大。"
    )


if test_prepared_fold[
    "scaler_fitted_rows"
] != test_prepared_fold[
    "train_rows"
]:
    raise RuntimeError(
        "标准化器拟合样本数错误。"
    )


if test_prepared_fold[
    "validation_used_for_scaler_fit"
]:
    raise RuntimeError(
        "内层验证特征参与了标准化器拟合。"
    )


if test_prepared_fold[
    "target_scaled"
]:
    raise RuntimeError(
        "lgη目标不应在该步骤中标准化。"
    )


# ============================================================
# 8. 树模型构造接口测试
# ============================================================

test_extratrees_trial = (
    optuna.trial.FixedTrial(
        {
            "bootstrap": True,
            "n_estimators": 300,
            "max_depth": 20,
            "min_samples_split": 4,
            "min_samples_leaf": 2,
            "max_features": "0.75",
            "max_samples": 0.80,
        }
    )
)


(
    test_extratrees_model,
    test_extratrees_search_parameters,
    test_extratrees_formal_parameters,
) = build_extratrees_from_trial(
    trial=test_extratrees_trial,
    seed=make_trial_seed(
        split_name="Data_split",
        outer_fold=1,
        model_name="ExtraTrees",
        trial_number=0,
        inner_fold=1,
    ),
)


test_lightgbm_trial = (
    optuna.trial.FixedTrial(
        {
            "n_estimators": 400,
            "learning_rate": 0.05,
            "max_depth": 10,
            "num_leaves": 43,
            "min_child_samples": 20,
            "subsample": 0.80,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.01,
            "reg_lambda": 0.10,
            "min_split_gain": 0.02,
        }
    )
)


(
    test_lightgbm_model,
    test_lightgbm_search_parameters,
    test_lightgbm_formal_parameters,
) = build_lightgbm_from_trial(
    trial=test_lightgbm_trial,
    seed=make_trial_seed(
        split_name="Data_split",
        outer_fold=1,
        model_name="LightGBM",
        trial_number=0,
        inner_fold=1,
    ),
)


if not isinstance(
    test_extratrees_model,
    ExtraTreesRegressor,
):
    raise TypeError(
        "ExtraTrees模型构造接口失败。"
    )


if not isinstance(
    test_lightgbm_model,
    LGBMRegressor,
):
    raise TypeError(
        "LightGBM模型构造接口失败。"
    )


# ============================================================
# 9. 保存标准化接口审计
# ============================================================

scaling_interface_audit = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "downstream_scaling_interface_passed"
    ),

    "test_task": {
        "split": (
            TEST_SCALING_SPLIT
        ),

        "outer_fold": int(
            TEST_SCALING_OUTER_FOLD
        ),

        "inner_fold": int(
            TEST_SCALING_INNER_FOLD
        ),
    },

    "input_shapes": {
        "X_train": list(
            test_prepared_fold[
                "X_train"
            ].shape
        ),

        "X_valid": list(
            test_prepared_fold[
                "X_valid"
            ].shape
        ),

        "y_train": list(
            test_prepared_fold[
                "y_train"
            ].shape
        ),

        "y_valid": list(
            test_prepared_fold[
                "y_valid"
            ].shape
        ),
    },

    "input_dtypes": {
        "X_train": str(
            test_prepared_fold[
                "X_train"
            ].dtype
        ),

        "X_valid": str(
            test_prepared_fold[
                "X_valid"
            ].dtype
        ),

        "y_train": str(
            test_prepared_fold[
                "y_train"
            ].dtype
        ),

        "y_valid": str(
            test_prepared_fold[
                "y_valid"
            ].dtype
        ),
    },

    "standardization_policy": {
        "scaler": "StandardScaler",

        "applied_to_models": [
            "ExtraTrees",
            "LightGBM",
            "Transformer",
        ],

        "fit_data": (
            "current_inner_training_features_only"
        ),

        "transform_data": [
            "current_inner_training_features",
            "current_inner_validation_features",
        ],

        "scaler_fitted_rows": int(
            test_prepared_fold[
                "scaler_fitted_rows"
            ]
        ),

        "validation_used_for_scaler_fit": False,

        "target_scaled": False,

        "scaled_training_mean_max_abs": (
            test_training_mean_max_abs
        ),
    },

    "leakage_control": {
        "inner_validation_feature_used_for_"
        "scaler_fit": False,

        "inner_validation_target_used_for_"
        "scaler_fit": False,

        "outer_train_mpnn_feature_read": False,

        "outer_test_mpnn_feature_read": False,

        "outer_test_target_read": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },
}


save_json_atomic(
    data=scaling_interface_audit,
    output_path=(
        SCALING_INTERFACE_AUDIT_PATH
    ),
)


# ============================================================
# 10. 保存树模型调参定义
# ============================================================

tree_tuning_definition = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "tree_tuning_definition_passed"
    ),

    "models": (
        TREE_MODEL_ORDER
    ),

    "objective": {
        "metric": (
            OPTUNA_OBJECTIVE_METRIC
        ),

        "direction": (
            OPTUNA_DIRECTION
        ),

        "inner_fold_aggregation": (
            INNER_FOLD_AGGREGATION
        ),

        "inner_fold_count": int(
            N_INNER_FOLDS
        ),

        "secondary_metrics": [
            "RMSE",
            "MAE",
            "Pearson_r",
        ],
    },

    "trial_budgets_per_outer_fold": {
        "ExtraTrees": int(
            N_TRIALS_PER_OUTER_FOLD[
                "ExtraTrees"
            ]
        ),

        "LightGBM": int(
            N_TRIALS_PER_OUTER_FOLD[
                "LightGBM"
            ]
        ),
    },

    "search_spaces": (
        TREE_SEARCH_SPACE_DESCRIPTION
    ),

    "fixed_parameters": {
        "ExtraTrees": {
            "criterion": (
                "squared_error"
            ),

            "n_jobs": -1,
        },

        "LightGBM": {
            "objective": "regression",

            "subsample_freq": 1,

            "n_jobs": -1,

            "deterministic": True,

            "force_col_wise": True,
        },
    },

    "interface_tests": {
        "ExtraTrees_constructed": True,

        "LightGBM_constructed": True,

        "models_fitted_in_Cell_2": False,
    },

    "leakage_control": {
        "inner_fold_specific_mpnn_features": (
            True
        ),

        "scaler_fitted_only_on_inner_training": (
            True
        ),

        "inner_validation_used_only_for_"
        "downstream_model_evaluation": True,

        "outer_test_used_for_tuning": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "outputs": {
        "scaling_interface_audit": str(
            SCALING_INTERFACE_AUDIT_PATH
        ),

        "tree_tuning_definition": str(
            TREE_TUNING_DEFINITION_PATH
        ),
    },
}


save_json_atomic(
    data=tree_tuning_definition,
    output_path=(
        TREE_TUNING_DEFINITION_PATH
    ),
)


# ============================================================
# 11. 清理测试对象
# ============================================================

del (
    test_extratrees_model,
    test_lightgbm_model,
    test_extratrees_trial,
    test_lightgbm_trial,
)

gc.collect()


# ============================================================
# 12. 输出检查
# ============================================================

print("=" * 104)
print("Cell 2 评价指标、折内标准化及树模型搜索空间检查通过 ✓")
print("=" * 104)

print(
    "metrics                          =",
    METRIC_ORDER,
)

print(
    "Optuna objective                 =",
    OPTUNA_OBJECTIVE_METRIC,
)

print(
    "Optuna direction                 =",
    OPTUNA_DIRECTION,
)

print(
    "inner fold aggregation           =",
    INNER_FOLD_AGGREGATION,
)

print(
    "inner folds per objective        =",
    N_INNER_FOLDS,
)

print(
    "tree models                      =",
    TREE_MODEL_ORDER,
)

print(
    "trial budgets/outer fold         =",
    {
        model_name: (
            N_TRIALS_PER_OUTER_FOLD[
                model_name
            ]
        )
        for model_name
        in TREE_MODEL_ORDER
    },
)

print(
    "test scaling task                =",
    (
        TEST_SCALING_SPLIT,
        TEST_SCALING_OUTER_FOLD,
        TEST_SCALING_INNER_FOLD,
    ),
)

print(
    "test X train shape               =",
    test_prepared_fold[
        "X_train"
    ].shape,
)

print(
    "test X valid shape               =",
    test_prepared_fold[
        "X_valid"
    ].shape,
)

print(
    "test y train shape               =",
    test_prepared_fold[
        "y_train"
    ].shape,
)

print(
    "test y valid shape               =",
    test_prepared_fold[
        "y_valid"
    ].shape,
)

print(
    "downstream scaler fitted rows    =",
    test_prepared_fold[
        "scaler_fitted_rows"
    ],
)

print(
    "validation used for scaler fit   =",
    test_prepared_fold[
        "validation_used_for_scaler_fit"
    ],
)

print(
    "target scaled                    =",
    test_prepared_fold[
        "target_scaled"
    ],
)

print(
    "scaled train mean max abs        =",
    test_training_mean_max_abs,
)

print(
    "ExtraTrees interface constructed =",
    True,
)

print(
    "LightGBM interface constructed   =",
    True,
)

print(
    "models fitted in Cell 2          =",
    False,
)

print(
    "outer train MPNN feature read    =",
    False,
)

print(
    "outer test MPNN feature read     =",
    False,
)

print(
    "outer test target read           =",
    False,
)

print(
    "old global MPNN feature read     =",
    False,
)

print(
    "external 10 points read          =",
    False,
)


print(
    "\n标准化接口审计："
)

print(
    SCALING_INTERFACE_AUDIT_PATH
)


print(
    "\n树模型调参定义："
)

print(
    TREE_TUNING_DEFINITION_PATH
)

Cell 2 评价指标、折内标准化及树模型搜索空间检查通过 ✓
metrics                          = ['R2', 'RMSE', 'MAE', 'Pearson_r']
Optuna objective                 = R2
Optuna direction                 = maximize
inner fold aggregation           = equal_weight_mean_across_3_inner_folds
inner folds per objective        = 3
tree models                      = ['ExtraTrees', 'LightGBM']
trial budgets/outer fold         = {'ExtraTrees': 15, 'LightGBM': 20}
test scaling task                = ('Data_split', 1, 1)
test X train shape               = (8648, 130)
test X valid shape               = (4324, 130)
test y train shape               = (8648,)
test y valid shape               = (4324,)
downstream scaler fitted rows    = 8648
validation used for scaler fit   = False
target scaled                    = False
scaled train mean max abs        = 1.0280036804033443e-05
ExtraTrees interface constructed = True
LightGBM interface constructed   = True
models fitted in Cell 2          = False
outer train MPNN feature read    = F

In [4]:
# Cell 3 | 定义Transformer结构、搜索空间和严格不读取验证标签的训练接口


TRANSFORMER_INTERFACE_AUDIT_PATH = (
    TUNING_LOG_DIR
    / "transformer_training_interface_audit_Data_IL.json"
)


TRANSFORMER_TUNING_DEFINITION_PATH = (
    TUNING_LOG_DIR
    / "transformer_tuning_definition_Data_IL.json"
)


# ============================================================
# 1. Transformer注意力模块
# ============================================================

class OptunaAttentionBlock(
    nn.Module
):
    """Transformer注意力与前馈模块。"""

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout,
    ):
        super().__init__()


        embed_dim = int(
            embed_dim
        )

        num_heads = int(
            num_heads
        )

        dropout = float(
            dropout
        )


        if embed_dim <= 0:
            raise ValueError(
                "embed_dim必须大于0。"
            )


        if num_heads <= 0:
            raise ValueError(
                "num_heads必须大于0。"
            )


        if embed_dim % num_heads != 0:
            raise ValueError(
                "embed_dim必须能被num_heads整除。"
            )


        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )


        self.norm1 = nn.LayerNorm(
            embed_dim
        )


        self.norm2 = nn.LayerNorm(
            embed_dim
        )


        self.feed_forward = nn.Sequential(
            nn.Linear(
                embed_dim,
                embed_dim * 4,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                embed_dim * 4,
                embed_dim,
            ),
        )


    def forward(
        self,
        x,
    ):
        attention_output, _ = (
            self.attention(
                x,
                x,
                x,
                need_weights=False,
            )
        )


        x = self.norm1(
            x
            + attention_output
        )


        feed_forward_output = (
            self.feed_forward(
                x
            )
        )


        x = self.norm2(
            x
            + feed_forward_output
        )


        return x


# ============================================================
# 2. Transformer回归模型
# ============================================================

class OptunaTransformerRegressor(
    nn.Module
):
    """用于离子液体lgη下游回归的Transformer模型。"""

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_tokens,
        num_heads,
        num_layers,
        dropout,
        noise_std=0.01,
    ):
        super().__init__()


        input_dim = int(
            input_dim
        )

        hidden_dim = int(
            hidden_dim
        )

        num_tokens = int(
            num_tokens
        )

        num_heads = int(
            num_heads
        )

        num_layers = int(
            num_layers
        )

        dropout = float(
            dropout
        )

        noise_std = float(
            noise_std
        )


        if input_dim <= 0:
            raise ValueError(
                "input_dim必须大于0。"
            )


        if hidden_dim <= 0:
            raise ValueError(
                "hidden_dim必须大于0。"
            )


        if num_tokens <= 0:
            raise ValueError(
                "num_tokens必须大于0。"
            )


        if hidden_dim % num_tokens != 0:
            raise ValueError(
                "hidden_dim必须能被num_tokens整除。"
            )


        token_dim = int(
            hidden_dim
            // num_tokens
        )


        if token_dim % num_heads != 0:
            raise ValueError(
                "token_dim必须能被num_heads整除。"
            )


        if num_layers < 1:
            raise ValueError(
                "num_layers至少为1。"
            )


        if not (
            0.0
            <= dropout
            < 1.0
        ):
            raise ValueError(
                "dropout必须位于[0, 1)范围内。"
            )


        if noise_std < 0:
            raise ValueError(
                "noise_std不能小于0。"
            )


        self.input_dim = (
            input_dim
        )

        self.hidden_dim = (
            hidden_dim
        )

        self.num_tokens = (
            num_tokens
        )

        self.token_dim = (
            token_dim
        )

        self.num_heads = (
            num_heads
        )

        self.num_layers = (
            num_layers
        )

        self.noise_std = (
            noise_std
        )


        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim,
        )


        self.blocks = nn.ModuleList(
            [
                OptunaAttentionBlock(
                    embed_dim=token_dim,
                    num_heads=num_heads,
                    dropout=dropout,
                )
                for _ in range(
                    num_layers
                )
            ]
        )


        self.regressor = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                hidden_dim,
                64,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                64,
                1,
            ),
        )


    def forward(
        self,
        x,
    ):
        if x.ndim != 2:
            raise ValueError(
                "Transformer输入必须为二维矩阵。"
            )


        if x.shape[
            1
        ] != self.input_dim:
            raise ValueError(
                "Transformer输入特征维度错误。"
            )


        if (
            self.training
            and self.noise_std > 0
        ):
            x = (
                x
                + torch.randn_like(
                    x
                )
                * self.noise_std
            )


        x = self.input_projection(
            x
        )


        x = x.reshape(
            x.shape[
                0
            ],
            self.num_tokens,
            self.token_dim,
        )


        for block in self.blocks:
            x = block(
                x
            )


        prediction = self.regressor(
            x
        )


        return prediction


# ============================================================
# 3. Transformer搜索空间
# ============================================================

def suggest_transformer_parameters(
    trial,
):
    """定义Transformer的Optuna搜索空间。"""

    return {
        "hidden_dim": (
            trial.suggest_categorical(
                "hidden_dim",
                [
                    64,
                    128,
                ],
            )
        ),

        "num_tokens": int(
            TRANSFORMER_NUM_TOKENS
        ),

        "num_heads": (
            trial.suggest_categorical(
                "num_heads",
                [
                    1,
                    2,
                    4,
                ],
            )
        ),

        "num_layers": (
            trial.suggest_int(
                "num_layers",
                1,
                4,
            )
        ),

        "lr": (
            trial.suggest_float(
                "lr",
                5e-4,
                5e-3,
                log=True,
            )
        ),

        "dropout": (
            trial.suggest_float(
                "dropout",
                0.20,
                0.60,
            )
        ),

        "weight_decay": (
            trial.suggest_float(
                "weight_decay",
                1e-4,
                5e-2,
                log=True,
            )
        ),

        "batch_size": (
            trial.suggest_categorical(
                "batch_size",
                [
                    32,
                    64,
                    128,
                ],
            )
        ),

        "noise_std": float(
            TRANSFORMER_NOISE_STD
        ),
    }


def transformer_search_to_formal(
    search_parameters,
    seed,
):
    """将Transformer搜索参数转换为正式训练参数。"""

    return {
        "input_dim": 130,

        "hidden_dim": int(
            search_parameters[
                "hidden_dim"
            ]
        ),

        "num_tokens": int(
            search_parameters.get(
                "num_tokens",
                TRANSFORMER_NUM_TOKENS,
            )
        ),

        "num_heads": int(
            search_parameters[
                "num_heads"
            ]
        ),

        "num_layers": int(
            search_parameters[
                "num_layers"
            ]
        ),

        "lr": float(
            search_parameters[
                "lr"
            ]
        ),

        "dropout": float(
            search_parameters[
                "dropout"
            ]
        ),

        "weight_decay": float(
            search_parameters[
                "weight_decay"
            ]
        ),

        "batch_size": int(
            search_parameters[
                "batch_size"
            ]
        ),

        "noise_std": float(
            search_parameters.get(
                "noise_std",
                TRANSFORMER_NOISE_STD,
            )
        ),

        "search_epochs": int(
            TRANSFORMER_SEARCH_EPOCHS
        ),

        "formal_epochs": int(
            TRANSFORMER_FORMAL_EPOCHS
        ),

        "seed": int(
            seed
        ),
    }


def build_transformer_model(
    parameters,
):
    """根据参数创建Transformer模型。"""

    return OptunaTransformerRegressor(
        input_dim=int(
            parameters.get(
                "input_dim",
                130,
            )
        ),

        hidden_dim=int(
            parameters[
                "hidden_dim"
            ]
        ),

        num_tokens=int(
            parameters.get(
                "num_tokens",
                TRANSFORMER_NUM_TOKENS,
            )
        ),

        num_heads=int(
            parameters[
                "num_heads"
            ]
        ),

        num_layers=int(
            parameters[
                "num_layers"
            ]
        ),

        dropout=float(
            parameters[
                "dropout"
            ]
        ),

        noise_std=float(
            parameters.get(
                "noise_std",
                TRANSFORMER_NOISE_STD,
            )
        ),
    )


# ============================================================
# 4. Transformer批量预测
# ============================================================

def predict_transformer_in_batches(
    model,
    X,
    device,
    batch_size=1024,
):
    """
    只依据特征进行批量预测。

    本函数不接收任何目标值。
    """

    X = np.asarray(
        X,
        dtype=np.float32,
    )


    if X.ndim != 2:
        raise ValueError(
            "待预测特征必须为二维矩阵。"
        )


    if len(
        X
    ) == 0:
        raise ValueError(
            "待预测特征为空。"
        )


    if not np.isfinite(
        X
    ).all():
        raise ValueError(
            "待预测特征中存在NaN或无穷值。"
        )


    prediction_dataset = TensorDataset(
        torch.from_numpy(
            X
        )
    )


    prediction_loader = DataLoader(
        prediction_dataset,

        batch_size=int(
            batch_size
        ),

        shuffle=False,

        num_workers=0,

        pin_memory=bool(
            device.type
            == "cuda"
        ),
    )


    prediction_parts = []


    model.eval()


    with torch.no_grad():
        for (
            batch_X,
        ) in prediction_loader:

            batch_X = batch_X.to(
                device,

                non_blocking=bool(
                    device.type
                    == "cuda"
                ),
            )


            batch_prediction = (
                model(
                    batch_X
                )
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )


            prediction_parts.append(
                batch_prediction
            )


    predictions = np.concatenate(
        prediction_parts
    ).astype(
        np.float32
    )


    if len(
        predictions
    ) != len(
        X
    ):
        raise RuntimeError(
            "Transformer预测结果行数错误。"
        )


    if not np.isfinite(
        predictions
    ).all():
        raise ValueError(
            "Transformer预测中存在NaN或无穷值。"
        )


    return predictions


# ============================================================
# 5. 单个内层折Transformer训练函数
# ============================================================

def train_predict_transformer(
    X_train,
    y_train,
    X_validation,
    parameters,
    seed,
    epochs=None,
):
    """
    仅使用内层训练特征和训练目标训练Transformer。

    该函数的参数中没有y_validation。

    X_validation只在全部训练轮次完成以后用于前向预测，
    不用于反向传播、早停、学习率调整或模型选择。
    """

    seed_everything(
        int(
            seed
        )
    )


    if epochs is None:
        epochs = int(
            TRANSFORMER_SEARCH_EPOCHS
        )

    else:
        epochs = int(
            epochs
        )


    if epochs < 1:
        raise ValueError(
            "Transformer训练轮数必须大于0。"
        )


    X_train = np.asarray(
        X_train,
        dtype=np.float32,
    )


    y_train = np.asarray(
        y_train,
        dtype=np.float32,
    ).reshape(
        -1,
        1,
    )


    X_validation = np.asarray(
        X_validation,
        dtype=np.float32,
    )


    if (
        X_train.ndim != 2
        or X_validation.ndim != 2
    ):
        raise ValueError(
            "Transformer训练和验证特征必须为二维矩阵。"
        )


    if (
        X_train.shape[
            1
        ] != 130
        or X_validation.shape[
            1
        ] != 130
    ):
        raise ValueError(
            "Transformer输入特征维度不是130。"
        )


    if len(
        X_train
    ) != len(
        y_train
    ):
        raise ValueError(
            "Transformer训练特征和目标行数不一致。"
        )


    if (
        len(
            X_train
        ) == 0
        or len(
            X_validation
        ) == 0
    ):
        raise ValueError(
            "Transformer训练集或验证特征为空。"
        )


    if not all(
        np.isfinite(
            array
        ).all()
        for array in [
            X_train,
            y_train,
            X_validation,
        ]
    ):
        raise ValueError(
            "Transformer输入中存在NaN或无穷值。"
        )


    model_parameters = {
        "input_dim": int(
            X_train.shape[
                1
            ]
        ),

        "hidden_dim": int(
            parameters[
                "hidden_dim"
            ]
        ),

        "num_tokens": int(
            parameters.get(
                "num_tokens",
                TRANSFORMER_NUM_TOKENS,
            )
        ),

        "num_heads": int(
            parameters[
                "num_heads"
            ]
        ),

        "num_layers": int(
            parameters[
                "num_layers"
            ]
        ),

        "dropout": float(
            parameters[
                "dropout"
            ]
        ),

        "noise_std": float(
            parameters.get(
                "noise_std",
                TRANSFORMER_NOISE_STD,
            )
        ),
    }


    model = build_transformer_model(
        model_parameters
    ).to(
        DEVICE
    )


    optimizer = optim.AdamW(
        model.parameters(),

        lr=float(
            parameters[
                "lr"
            ]
        ),

        weight_decay=float(
            parameters[
                "weight_decay"
            ]
        ),
    )


    criterion = nn.MSELoss()


    training_dataset = TensorDataset(
        torch.from_numpy(
            X_train
        ),

        torch.from_numpy(
            y_train
        ),
    )


    loader_generator = torch.Generator()


    loader_generator.manual_seed(
        int(
            seed
        )
    )


    training_loader = DataLoader(
        training_dataset,

        batch_size=int(
            parameters[
                "batch_size"
            ]
        ),

        shuffle=True,

        num_workers=0,

        generator=loader_generator,

        pin_memory=bool(
            DEVICE.type
            == "cuda"
        ),

        drop_last=False,
    )


    if len(
        training_loader
    ) == 0:
        raise RuntimeError(
            "Transformer训练DataLoader为空。"
        )


    scheduler = OneCycleLR(
        optimizer,

        max_lr=float(
            parameters[
                "lr"
            ]
        ),

        steps_per_epoch=len(
            training_loader
        ),

        epochs=epochs,

        pct_start=0.10,

        anneal_strategy="cos",
    )


    training_history = []


    last_gradient_norm = np.nan


    for epoch in range(
        1,
        epochs + 1,
    ):
        model.train()


        epoch_loss_sum = 0.0

        epoch_row_count = 0


        for (
            batch_X,
            batch_y,
        ) in training_loader:

            batch_X = batch_X.to(
                DEVICE,

                non_blocking=bool(
                    DEVICE.type
                    == "cuda"
                ),
            )


            batch_y = batch_y.to(
                DEVICE,

                non_blocking=bool(
                    DEVICE.type
                    == "cuda"
                ),
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            batch_prediction = model(
                batch_X
            )


            loss = criterion(
                batch_prediction,
                batch_y,
            )


            if not torch.isfinite(
                loss
            ):
                raise RuntimeError(
                    "Transformer训练损失出现NaN或无穷值。"
                )


            loss.backward()


            gradient_norm = (
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )
            )


            if not torch.isfinite(
                gradient_norm
            ):
                raise RuntimeError(
                    "Transformer梯度范数出现NaN或无穷值。"
                )


            last_gradient_norm = float(
                gradient_norm
                .detach()
                .cpu()
                .item()
            )


            optimizer.step()


            scheduler.step()


            batch_rows = int(
                len(
                    batch_y
                )
            )


            epoch_loss_sum += (
                float(
                    loss.item()
                )
                * batch_rows
            )


            epoch_row_count += (
                batch_rows
            )


        if epoch_row_count != len(
            X_train
        ):
            raise RuntimeError(
                "Transformer某轮训练使用的"
                "样本数与训练集行数不一致。"
            )


        epoch_mse = float(
            epoch_loss_sum
            / epoch_row_count
        )


        training_history.append(
            {
                "Epoch": int(
                    epoch
                ),

                "Train_MSE": (
                    epoch_mse
                ),

                "Learning_rate": float(
                    optimizer.param_groups[
                        0
                    ][
                        "lr"
                    ]
                ),

                "Training_rows_used": int(
                    epoch_row_count
                ),

                "Validation_rows_used_for_"
                "backpropagation": 0,
            }
        )


    validation_prediction = (
        predict_transformer_in_batches(
            model=model,

            X=X_validation,

            device=DEVICE,

            batch_size=1024,
        )
    )


    final_train_mse = float(
        training_history[
            -1
        ][
            "Train_MSE"
        ]
    )


    training_result = {
        "validation_prediction": (
            validation_prediction
        ),

        "final_train_mse": (
            final_train_mse
        ),

        "last_gradient_norm": (
            last_gradient_norm
        ),

        "training_history": (
            training_history
        ),

        "epochs_completed": int(
            epochs
        ),

        "training_rows": int(
            len(
                X_train
            )
        ),

        "validation_feature_rows": int(
            len(
                X_validation
            )
        ),

        "validation_target_argument_exists": False,

        "validation_target_received": False,

        "validation_used_for_backpropagation": False,

        "validation_used_for_early_stopping": False,

        "validation_used_for_scheduler": False,

        "validation_used_for_model_selection": False,

        "outer_test_used": False,
    }


    del (
        model,
        optimizer,
        scheduler,
        criterion,
        training_dataset,
        training_loader,
        loader_generator,
    )


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return training_result


# ============================================================
# 6. Transformer搜索空间记录
# ============================================================

TRANSFORMER_SEARCH_SPACE_DESCRIPTION = {
    "trial_budget_per_outer_fold": int(
        N_TRIALS_PER_OUTER_FOLD[
            "Transformer"
        ]
    ),

    "parameters": {
        "hidden_dim": [
            64,
            128,
        ],

        "num_tokens": {
            "fixed": int(
                TRANSFORMER_NUM_TOKENS
            )
        },

        "num_heads": [
            1,
            2,
            4,
        ],

        "num_layers": {
            "minimum": 1,
            "maximum": 4,
            "step": 1,
        },

        "learning_rate": {
            "minimum": 5e-4,
            "maximum": 5e-3,
            "log": True,
        },

        "dropout": {
            "minimum": 0.20,
            "maximum": 0.60,
        },

        "weight_decay": {
            "minimum": 1e-4,
            "maximum": 5e-2,
            "log": True,
        },

        "batch_size": [
            32,
            64,
            128,
        ],

        "noise_std": {
            "fixed": float(
                TRANSFORMER_NOISE_STD
            )
        },

        "search_epochs": {
            "fixed": int(
                TRANSFORMER_SEARCH_EPOCHS
            )
        },

        "formal_epochs": {
            "fixed": int(
                TRANSFORMER_FORMAL_EPOCHS
            )
        },
    },
}


# ============================================================
# 7. 严格加载接口测试输入
# ============================================================

def load_transformer_interface_test_inputs_strict(
    split_name,
    outer_fold,
    inner_fold,
):
    """
    严格只读取：
    1. X_inner_train_mpnn.npy
    2. y_inner_train_lg_eta.npy
    3. X_inner_valid_mpnn.npy

    不读取、加载或计算哈希：
    y_inner_valid_lg_eta.npy
    """

    paths = get_complete_inner_tuning_paths(
        split_name=split_name,
        outer_fold=outer_fold,
        inner_fold=inner_fold,
    )


    allowed_open_roles = [
        "X_inner_train",
        "y_inner_train",
        "X_inner_valid",
    ]


    forbidden_open_roles = [
        "y_inner_valid",
    ]


    opened_file_roles = []


    X_inner_train_raw = np.load(
        paths[
            "X_inner_train"
        ],
        allow_pickle=False,
    )


    opened_file_roles.append(
        "X_inner_train"
    )


    y_inner_train = np.load(
        paths[
            "y_inner_train"
        ],
        allow_pickle=False,
    )


    opened_file_roles.append(
        "y_inner_train"
    )


    X_inner_valid_raw = np.load(
        paths[
            "X_inner_valid"
        ],
        allow_pickle=False,
    )


    opened_file_roles.append(
        "X_inner_valid"
    )


    if any(
        role in opened_file_roles
        for role in forbidden_open_roles
    ):
        raise RuntimeError(
            "严格接口测试意外打开了验证目标文件。"
        )


    if set(
        opened_file_roles
    ) != set(
        allowed_open_roles
    ):
        raise RuntimeError(
            "严格接口测试实际打开的文件角色"
            "与允许列表不一致。"
        )


    X_inner_train_raw = np.asarray(
        X_inner_train_raw,
        dtype=np.float32,
    )


    X_inner_valid_raw = np.asarray(
        X_inner_valid_raw,
        dtype=np.float32,
    )


    y_inner_train = np.asarray(
        y_inner_train,
        dtype=np.float32,
    ).reshape(-1)


    if (
        X_inner_train_raw.ndim != 2
        or X_inner_valid_raw.ndim != 2
    ):
        raise ValueError(
            "严格接口测试的特征数组不是二维矩阵。"
        )


    if (
        X_inner_train_raw.shape[
            1
        ] != 130
        or X_inner_valid_raw.shape[
            1
        ] != 130
    ):
        raise ValueError(
            "严格接口测试的特征维度不是130。"
        )


    if y_inner_train.ndim != 1:
        raise ValueError(
            "严格接口测试的训练目标不是一维数组。"
        )


    if len(
        X_inner_train_raw
    ) != len(
        y_inner_train
    ):
        raise ValueError(
            "严格接口测试的训练特征和目标"
            "行数不一致。"
        )


    if not all(
        np.isfinite(
            array
        ).all()
        for array in [
            X_inner_train_raw,
            X_inner_valid_raw,
            y_inner_train,
        ]
    ):
        raise ValueError(
            "严格接口测试输入中存在NaN或无穷值。"
        )


    downstream_scaler = StandardScaler()


    X_inner_train_scaled = (
        downstream_scaler
        .fit_transform(
            X_inner_train_raw
        )
        .astype(
            np.float32
        )
    )


    X_inner_valid_scaled = (
        downstream_scaler
        .transform(
            X_inner_valid_raw
        )
        .astype(
            np.float32
        )
    )


    scaler_fitted_rows = (
        get_scaler_fitted_rows(
            downstream_scaler
        )
    )


    if scaler_fitted_rows != len(
        X_inner_train_raw
    ):
        raise RuntimeError(
            "严格接口测试的StandardScaler"
            "拟合样本数错误。"
        )


    if not (
        np.isfinite(
            X_inner_train_scaled
        ).all()
        and np.isfinite(
            X_inner_valid_scaled
        ).all()
    ):
        raise ValueError(
            "严格接口测试标准化结果"
            "存在NaN或无穷值。"
        )


    validation_target_path = Path(
        paths[
            "y_inner_valid"
        ]
    )


    if not validation_target_path.exists():
        raise FileNotFoundError(
            "验证目标文件不存在。"
        )


    # 这里只检查Path是否存在。
    # 不调用np.load、不打开文件、不计算哈希、不读取内容。
    validation_target_file_opened = False

    validation_target_file_hashed = False


    return {
        "X_train": (
            X_inner_train_scaled
        ),

        "X_valid": (
            X_inner_valid_scaled
        ),

        "y_train": (
            y_inner_train
        ),

        "scaler": (
            downstream_scaler
        ),

        "scaler_fitted_rows": int(
            scaler_fitted_rows
        ),

        "train_rows": int(
            len(
                X_inner_train_scaled
            )
        ),

        "valid_feature_rows": int(
            len(
                X_inner_valid_scaled
            )
        ),

        "feature_dim": 130,

        "opened_file_roles": (
            opened_file_roles
        ),

        "validation_target_path": (
            validation_target_path
        ),

        "validation_target_path_registered_only": (
            True
        ),

        "validation_target_file_opened": (
            validation_target_file_opened
        ),

        "validation_target_file_hashed": (
            validation_target_file_hashed
        ),

        "prepare_downstream_inner_fold_called": (
            False
        ),

        "load_inner_tuning_arrays_called": (
            False
        ),

        "validation_used_for_scaler_fit": (
            False
        ),

        "target_scaled": False,

        "paths": paths,
    }


# ============================================================
# 8. Transformer严格接口测试
# ============================================================

TRANSFORMER_TEST_SPLIT = (
    "Data_split"
)

TRANSFORMER_TEST_OUTER_FOLD = 1

TRANSFORMER_TEST_INNER_FOLD = 1

TRANSFORMER_TEST_ROWS = 128

TRANSFORMER_INTERFACE_TEST_EPOCHS = 1


transformer_test_inputs = (
    load_transformer_interface_test_inputs_strict(
        split_name=(
            TRANSFORMER_TEST_SPLIT
        ),

        outer_fold=(
            TRANSFORMER_TEST_OUTER_FOLD
        ),

        inner_fold=(
            TRANSFORMER_TEST_INNER_FOLD
        ),
    )
)


if transformer_test_inputs[
    "validation_target_file_opened"
]:
    raise RuntimeError(
        "Cell 3测试阶段读取了验证目标文件。"
    )


if transformer_test_inputs[
    "validation_target_file_hashed"
]:
    raise RuntimeError(
        "Cell 3测试阶段计算了验证目标文件哈希。"
    )


if transformer_test_inputs[
    "prepare_downstream_inner_fold_called"
]:
    raise RuntimeError(
        "Cell 3不允许调用"
        "prepare_downstream_inner_fold。"
    )


if transformer_test_inputs[
    "load_inner_tuning_arrays_called"
]:
    raise RuntimeError(
        "Cell 3不允许调用"
        "load_inner_tuning_arrays。"
    )


transformer_fixed_trial = (
    optuna.trial.FixedTrial(
        {
            "hidden_dim": 64,

            "num_heads": 2,

            "num_layers": 2,

            "lr": 1e-3,

            "dropout": 0.30,

            "weight_decay": 1e-3,

            "batch_size": 64,
        }
    )
)


transformer_test_search_parameters = (
    suggest_transformer_parameters(
        transformer_fixed_trial
    )
)


transformer_test_seed = (
    make_trial_seed(
        split_name=(
            TRANSFORMER_TEST_SPLIT
        ),

        outer_fold=(
            TRANSFORMER_TEST_OUTER_FOLD
        ),

        model_name="Transformer",

        trial_number=0,

        inner_fold=(
            TRANSFORMER_TEST_INNER_FOLD
        ),
    )
)


transformer_test_formal_parameters = (
    transformer_search_to_formal(
        search_parameters=(
            transformer_test_search_parameters
        ),

        seed=(
            transformer_test_seed
        ),
    )
)


transformer_test_model_for_count = (
    build_transformer_model(
        transformer_test_formal_parameters
    )
)


transformer_parameter_count = int(
    sum(
        parameter.numel()
        for parameter
        in transformer_test_model_for_count.parameters()
    )
)


del (
    transformer_test_model_for_count
)


transformer_test_training_result = (
    train_predict_transformer(
        X_train=(
            transformer_test_inputs[
                "X_train"
            ][
                :TRANSFORMER_TEST_ROWS
            ]
        ),

        y_train=(
            transformer_test_inputs[
                "y_train"
            ][
                :TRANSFORMER_TEST_ROWS
            ]
        ),

        X_validation=(
            transformer_test_inputs[
                "X_valid"
            ][
                :TRANSFORMER_TEST_ROWS
            ]
        ),

        parameters=(
            transformer_test_formal_parameters
        ),

        seed=(
            transformer_test_seed
        ),

        epochs=(
            TRANSFORMER_INTERFACE_TEST_EPOCHS
        ),
    )
)


transformer_test_valid_prediction = (
    transformer_test_training_result[
        "validation_prediction"
    ]
)


if transformer_test_valid_prediction.shape != (
    TRANSFORMER_TEST_ROWS,
):
    raise RuntimeError(
        "Transformer验证预测形状错误。"
    )


if not np.isfinite(
    transformer_test_valid_prediction
).all():
    raise RuntimeError(
        "Transformer验证预测存在NaN或无穷值。"
    )


if transformer_test_training_result[
    "validation_target_argument_exists"
]:
    raise RuntimeError(
        "Transformer训练函数不应存在"
        "验证目标参数。"
    )


if transformer_test_training_result[
    "validation_target_received"
]:
    raise RuntimeError(
        "Transformer训练函数接收了验证目标。"
    )


if transformer_test_training_result[
    "validation_used_for_backpropagation"
]:
    raise RuntimeError(
        "验证数据参与了Transformer反向传播。"
    )


if transformer_test_training_result[
    "validation_used_for_early_stopping"
]:
    raise RuntimeError(
        "验证数据参与了Transformer早停。"
    )


if transformer_test_training_result[
    "validation_used_for_scheduler"
]:
    raise RuntimeError(
        "验证数据参与了Transformer学习率调整。"
    )


# ============================================================
# 9. 保存Transformer严格接口审计
# ============================================================

transformer_interface_audit = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "transformer_strict_training_"
        "interface_passed"
    ),

    "test_task": {
        "split": (
            TRANSFORMER_TEST_SPLIT
        ),

        "outer_fold": int(
            TRANSFORMER_TEST_OUTER_FOLD
        ),

        "inner_fold": int(
            TRANSFORMER_TEST_INNER_FOLD
        ),

        "training_test_rows": int(
            TRANSFORMER_TEST_ROWS
        ),

        "validation_feature_test_rows": int(
            TRANSFORMER_TEST_ROWS
        ),

        "interface_test_epochs": int(
            TRANSFORMER_INTERFACE_TEST_EPOCHS
        ),
    },

    "device": str(
        DEVICE
    ),

    "model_parameters": (
        transformer_test_formal_parameters
    ),

    "model_parameter_count": (
        transformer_parameter_count
    ),

    "strict_input_loading": {
        "opened_file_roles": (
            transformer_test_inputs[
                "opened_file_roles"
            ]
        ),

        "allowed_opened_file_roles": [
            "X_inner_train",
            "y_inner_train",
            "X_inner_valid",
        ],

        "validation_target_path": str(
            transformer_test_inputs[
                "validation_target_path"
            ]
        ),

        "validation_target_path_registered_only": (
            True
        ),

        "validation_target_file_opened": (
            False
        ),

        "validation_target_file_hashed": (
            False
        ),

        "prepare_downstream_inner_fold_called": (
            False
        ),

        "load_inner_tuning_arrays_called": (
            False
        ),
    },

    "standardization": {
        "scaler": "StandardScaler",

        "scaler_fit_data": (
            "full_current_inner_training_"
            "features_only"
        ),

        "scaler_fitted_rows": int(
            transformer_test_inputs[
                "scaler_fitted_rows"
            ]
        ),

        "validation_feature_used_for_"
        "scaler_fit": False,

        "validation_target_used_for_"
        "scaler_fit": False,

        "target_scaled": False,
    },

    "interface_checks": {
        "model_constructed": True,

        "one_epoch_training_passed": True,

        "training_loss_finite": bool(
            np.isfinite(
                transformer_test_training_result[
                    "final_train_mse"
                ]
            )
        ),

        "gradient_norm_finite": bool(
            np.isfinite(
                transformer_test_training_result[
                    "last_gradient_norm"
                ]
            )
        ),

        "validation_feature_prediction_passed": (
            True
        ),

        "validation_prediction_shape": list(
            transformer_test_valid_prediction.shape
        ),

        "full_50_epoch_training_in_Cell_3": (
            False
        ),

        "optuna_started_in_Cell_3": False,
    },

    "leakage_control": {
        "training_target_used": True,

        "validation_feature_used_only_for_"
        "post_training_forward_prediction": True,

        "validation_target_file_opened": False,

        "validation_target_value_read": False,

        "validation_target_file_hashed": False,

        "validation_target_received_by_"
        "training_function": False,

        "validation_target_used_for_"
        "backpropagation": False,

        "validation_target_used_for_"
        "early_stopping": False,

        "validation_target_used_for_"
        "scheduler": False,

        "validation_target_used_for_"
        "model_selection": False,

        "outer_train_mpnn_feature_read": False,

        "outer_test_mpnn_feature_read": False,

        "outer_test_target_read": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },
}


save_json_atomic(
    data=transformer_interface_audit,
    output_path=(
        TRANSFORMER_INTERFACE_AUDIT_PATH
    ),
)


# ============================================================
# 10. 保存Transformer调参定义
# ============================================================

transformer_tuning_definition = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "transformer_tuning_definition_passed"
    ),

    "model": "Transformer",

    "objective": {
        "metric": (
            OPTUNA_OBJECTIVE_METRIC
        ),

        "direction": (
            OPTUNA_DIRECTION
        ),

        "inner_fold_aggregation": (
            INNER_FOLD_AGGREGATION
        ),

        "inner_fold_count": int(
            N_INNER_FOLDS
        ),

        "secondary_metrics": [
            "RMSE",
            "MAE",
            "Pearson_r",
        ],
    },

    "trial_budget_per_outer_fold": int(
        N_TRIALS_PER_OUTER_FOLD[
            "Transformer"
        ]
    ),

    "search_space": (
        TRANSFORMER_SEARCH_SPACE_DESCRIPTION
    ),

    "training_policy": {
        "loss": "MSELoss",

        "optimizer": "AdamW",

        "scheduler": "OneCycleLR",

        "gradient_clip_max_norm": 1.0,

        "search_epochs": int(
            TRANSFORMER_SEARCH_EPOCHS
        ),

        "formal_epochs": int(
            TRANSFORMER_FORMAL_EPOCHS
        ),

        "early_stopping_used_during_tuning": (
            False
        ),

        "target_scaled": False,

        "input_scaler": "StandardScaler",

        "input_scaler_fit_data": (
            "current_inner_training_features_only"
        ),
    },

    "validation_policy": {
        "validation_features_used_after_training": (
            True
        ),

        "validation_target_received_by_"
        "training_function": False,

        "validation_target_read_by_strict_"
        "Cell_3_interface_test": False,

        "validation_target_used_only_later_"
        "for_downstream_metric_calculation": True,
    },

    "leakage_control": {
        "inner_fold_specific_mpnn_features": (
            True
        ),

        "inner_validation_used_for_"
        "transformer_backpropagation": False,

        "inner_validation_used_for_"
        "transformer_early_stopping": False,

        "inner_validation_used_for_"
        "transformer_scheduler": False,

        "outer_test_used_for_tuning": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "outputs": {
        "transformer_interface_audit": str(
            TRANSFORMER_INTERFACE_AUDIT_PATH
        ),

        "transformer_tuning_definition": str(
            TRANSFORMER_TUNING_DEFINITION_PATH
        ),
    },
}


save_json_atomic(
    data=transformer_tuning_definition,
    output_path=(
        TRANSFORMER_TUNING_DEFINITION_PATH
    ),
)


# ============================================================
# 11. 清理接口测试对象
# ============================================================

transformer_test_loss_value = float(
    transformer_test_training_result[
        "final_train_mse"
    ]
)


transformer_test_gradient_norm_value = float(
    transformer_test_training_result[
        "last_gradient_norm"
    ]
)


validation_target_file_opened = bool(
    transformer_test_inputs[
        "validation_target_file_opened"
    ]
)


validation_target_file_hashed = bool(
    transformer_test_inputs[
        "validation_target_file_hashed"
    ]
)


scaler_fitted_rows = int(
    transformer_test_inputs[
        "scaler_fitted_rows"
    ]
)


opened_file_roles = list(
    transformer_test_inputs[
        "opened_file_roles"
    ]
)


del (
    transformer_fixed_trial,
    transformer_test_training_result,
    transformer_test_inputs,
)


gc.collect()


if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 12. 输出检查
# ============================================================

print("=" * 104)
print("Cell 3 Transformer严格无验证标签训练接口检查通过 ✓")
print("=" * 104)

print(
    "device                              =",
    DEVICE,
)

print(
    "input feature dimension             =",
    130,
)

print(
    "trial budget/outer fold             =",
    N_TRIALS_PER_OUTER_FOLD[
        "Transformer"
    ],
)

print(
    "search epochs                       =",
    TRANSFORMER_SEARCH_EPOCHS,
)

print(
    "formal epochs                       =",
    TRANSFORMER_FORMAL_EPOCHS,
)

print(
    "fixed num tokens                    =",
    TRANSFORMER_NUM_TOKENS,
)

print(
    "fixed noise std                     =",
    TRANSFORMER_NOISE_STD,
)

print(
    "test task                           =",
    (
        TRANSFORMER_TEST_SPLIT,
        TRANSFORMER_TEST_OUTER_FOLD,
        TRANSFORMER_TEST_INNER_FOLD,
    ),
)

print(
    "opened file roles                   =",
    opened_file_roles,
)

print(
    "validation target file opened       =",
    validation_target_file_opened,
)

print(
    "validation target file hashed       =",
    validation_target_file_hashed,
)

print(
    "prepare_downstream_inner_fold used  =",
    False,
)

print(
    "load_inner_tuning_arrays used       =",
    False,
)

print(
    "scaler fitted rows                  =",
    scaler_fitted_rows,
)

print(
    "validation used for scaler fit      =",
    False,
)

print(
    "test training rows                  =",
    TRANSFORMER_TEST_ROWS,
)

print(
    "test validation feature rows        =",
    TRANSFORMER_TEST_ROWS,
)

print(
    "interface test epochs               =",
    TRANSFORMER_INTERFACE_TEST_EPOCHS,
)

print(
    "test model parameter count          =",
    transformer_parameter_count,
)

print(
    "one-epoch final train MSE           =",
    transformer_test_loss_value,
)

print(
    "last gradient norm                  =",
    transformer_test_gradient_norm_value,
)

print(
    "validation prediction shape         =",
    transformer_test_valid_prediction.shape,
)

print(
    "one-epoch training passed           =",
    True,
)

print(
    "validation feature prediction passed=",
    True,
)

print(
    "full 50-epoch training in Cell 3    =",
    False,
)

print(
    "Optuna started in Cell 3            =",
    False,
)

print(
    "validation target value read        =",
    False,
)

print(
    "validation target used for training =",
    False,
)

print(
    "outer train MPNN feature read       =",
    False,
)

print(
    "outer test MPNN feature read        =",
    False,
)

print(
    "outer test target read              =",
    False,
)

print(
    "old global MPNN feature read        =",
    False,
)

print(
    "external 10 points read             =",
    False,
)


print(
    "\nTransformer严格接口审计："
)

print(
    TRANSFORMER_INTERFACE_AUDIT_PATH
)


print(
    "\nTransformer调参定义："
)

print(
    TRANSFORMER_TUNING_DEFINITION_PATH
)

Cell 3 Transformer严格无验证标签训练接口检查通过 ✓
device                              = cpu
input feature dimension             = 130
trial budget/outer fold             = 10
search epochs                       = 50
formal epochs                       = 150
fixed num tokens                    = 8
fixed noise std                     = 0.01
test task                           = ('Data_split', 1, 1)
opened file roles                   = ['X_inner_train', 'y_inner_train', 'X_inner_valid']
validation target file opened       = False
validation target file hashed       = False
prepare_downstream_inner_fold used  = False
load_inner_tuning_arrays used       = False
scaler fitted rows                  = 8648
validation used for scaler fit      = False
test training rows                  = 128
test validation feature rows        = 128
interface test epochs               = 1
test model parameter count          = 14353
one-epoch final train MSE           = 4.746927499771118
last gradient norm                  =

In [5]:
# Cell 4 | 定义严格延迟读取验证标签的Optuna目标函数、调参签名和30个Study计划


import inspect


OPTUNA_OBJECTIVE_DEFINITION_PATH = (
    TUNING_LOG_DIR
    / "optuna_objective_definition_Data_IL.json"
)


TUNING_SIGNATURE_PATH = (
    TUNING_LOG_DIR
    / "optuna_tuning_signature_Data_IL.json"
)


STUDY_PLAN_PATH = (
    TUNING_LOG_DIR
    / "optuna_study_plan_Data_IL.csv"
)


# ============================================================
# 1. 数组哈希函数
# ============================================================

def calculate_array_sha256(
    array,
):
    """计算已经固定的NumPy数组内容哈希。"""

    array = np.ascontiguousarray(
        np.asarray(
            array
        )
    )


    sha256 = hashlib.sha256()


    sha256.update(
        str(
            array.dtype
        ).encode(
            "utf-8"
        )
    )


    sha256.update(
        str(
            tuple(
                array.shape
            )
        ).encode(
            "utf-8"
        )
    )


    sha256.update(
        array.tobytes(
            order="C"
        )
    )


    return sha256.hexdigest()


# ============================================================
# 2. 严格读取训练数据和验证特征
# ============================================================

def load_inner_fold_before_prediction(
    split_name,
    outer_fold,
    inner_fold,
):
    """
    在模型预测固定以前，只读取：

    X_inner_train_mpnn.npy
    y_inner_train_lg_eta.npy
    X_inner_valid_mpnn.npy

    不读取y_inner_valid_lg_eta.npy。
    """

    task_key = (
        split_name,
        int(
            outer_fold
        ),
        int(
            inner_fold
        ),
    )


    if (
        task_key
        not in INNER_TUNING_BUNDLE_PATHS
    ):
        raise KeyError(
            f"找不到内层任务路径：{task_key}"
        )


    paths = (
        INNER_TUNING_BUNDLE_PATHS[
            task_key
        ]
    )


    opened_file_roles = []


    X_inner_train_raw = np.load(
        paths[
            "X_inner_train"
        ],
        allow_pickle=False,
    )


    opened_file_roles.append(
        "X_inner_train"
    )


    y_inner_train = np.load(
        paths[
            "y_inner_train"
        ],
        allow_pickle=False,
    )


    opened_file_roles.append(
        "y_inner_train"
    )


    X_inner_valid_raw = np.load(
        paths[
            "X_inner_valid"
        ],
        allow_pickle=False,
    )


    opened_file_roles.append(
        "X_inner_valid"
    )


    allowed_opened_roles = {
        "X_inner_train",
        "y_inner_train",
        "X_inner_valid",
    }


    if set(
        opened_file_roles
    ) != allowed_opened_roles:
        raise RuntimeError(
            f"{task_key}：预测前实际读取文件"
            "与允许列表不一致。"
        )


    if "y_inner_valid" in (
        opened_file_roles
    ):
        raise RuntimeError(
            f"{task_key}：模型预测固定以前"
            "读取了验证目标。"
        )


    X_inner_train_raw = np.asarray(
        X_inner_train_raw,
        dtype=np.float32,
    )


    X_inner_valid_raw = np.asarray(
        X_inner_valid_raw,
        dtype=np.float32,
    )


    y_inner_train = np.asarray(
        y_inner_train,
        dtype=np.float32,
    ).reshape(-1)


    if (
        X_inner_train_raw.ndim != 2
        or X_inner_valid_raw.ndim != 2
    ):
        raise ValueError(
            f"{task_key}：训练或验证特征"
            "不是二维矩阵。"
        )


    if (
        X_inner_train_raw.shape[
            1
        ] != 130
        or X_inner_valid_raw.shape[
            1
        ] != 130
    ):
        raise ValueError(
            f"{task_key}：MPNN表征维度不是130。"
        )


    if y_inner_train.ndim != 1:
        raise ValueError(
            f"{task_key}：训练目标不是一维数组。"
        )


    if len(
        X_inner_train_raw
    ) != len(
        y_inner_train
    ):
        raise ValueError(
            f"{task_key}：训练特征和目标"
            "行数不一致。"
        )


    if not all(
        np.isfinite(
            array
        ).all()
        for array in [
            X_inner_train_raw,
            X_inner_valid_raw,
            y_inner_train,
        ]
    ):
        raise ValueError(
            f"{task_key}：预测前输入中"
            "存在NaN或无穷值。"
        )


    downstream_scaler = StandardScaler()


    X_inner_train_scaled = (
        downstream_scaler
        .fit_transform(
            X_inner_train_raw
        )
        .astype(
            np.float32
        )
    )


    X_inner_valid_scaled = (
        downstream_scaler
        .transform(
            X_inner_valid_raw
        )
        .astype(
            np.float32
        )
    )


    scaler_fitted_rows = (
        get_scaler_fitted_rows(
            downstream_scaler
        )
    )


    if scaler_fitted_rows != len(
        X_inner_train_raw
    ):
        raise RuntimeError(
            f"{task_key}：标准化器拟合"
            "样本数错误。"
        )


    if not (
        np.isfinite(
            X_inner_train_scaled
        ).all()
        and np.isfinite(
            X_inner_valid_scaled
        ).all()
    ):
        raise ValueError(
            f"{task_key}：标准化结果中"
            "存在NaN或无穷值。"
        )


    return {
        "task_key": task_key,

        "X_train": (
            X_inner_train_scaled
        ),

        "X_valid": (
            X_inner_valid_scaled
        ),

        "y_train": (
            y_inner_train
        ),

        "train_rows": int(
            len(
                X_inner_train_scaled
            )
        ),

        "valid_rows": int(
            len(
                X_inner_valid_scaled
            )
        ),

        "feature_dim": 130,

        "scaler_fitted_rows": int(
            scaler_fitted_rows
        ),

        "opened_file_roles_before_prediction": (
            opened_file_roles
        ),

        "validation_target_path": Path(
            paths[
                "y_inner_valid"
            ]
        ),

        "validation_target_file_opened": False,

        "validation_target_file_hashed": False,

        "validation_used_for_scaler_fit": False,

        "target_scaled": False,
    }


# ============================================================
# 3. 预测固定后才读取验证目标
# ============================================================

def load_validation_target_after_prediction(
    prediction,
    validation_target_path,
    expected_rows,
):
    """
    确认预测已经形成并固定后，才读取验证目标。

    验证目标只用于计算下游验证指标。
    """

    prediction = np.asarray(
        prediction,
        dtype=np.float32,
    ).reshape(-1)


    if len(
        prediction
    ) != int(
        expected_rows
    ):
        raise ValueError(
            "验证预测行数与预期验证行数不一致。"
        )


    if not np.isfinite(
        prediction
    ).all():
        raise ValueError(
            "验证预测中存在NaN或无穷值。"
        )


    # 生成独立只读副本，确保后续读取标签时预测已经固定。
    fixed_prediction = np.array(
        prediction,
        dtype=np.float32,
        copy=True,
    )


    fixed_prediction.setflags(
        write=False
    )


    prediction_sha256 = (
        calculate_array_sha256(
            fixed_prediction
        )
    )


    prediction_fixed_at = (
        datetime.now().isoformat(
            timespec="microseconds"
        )
    )


    validation_target_path = Path(
        validation_target_path
    )


    if not (
        validation_target_path
        .exists()
    ):
        raise FileNotFoundError(
            "预测已固定，但找不到验证目标文件：\n"
            f"{validation_target_path}"
        )


    y_inner_valid = np.load(
        validation_target_path,
        allow_pickle=False,
    )


    validation_target_loaded_at = (
        datetime.now().isoformat(
            timespec="microseconds"
        )
    )


    y_inner_valid = np.asarray(
        y_inner_valid,
        dtype=np.float32,
    ).reshape(-1)


    if len(
        y_inner_valid
    ) != len(
        fixed_prediction
    ):
        raise ValueError(
            "验证目标行数与固定预测行数不一致。"
        )


    if not np.isfinite(
        y_inner_valid
    ).all():
        raise ValueError(
            "验证目标中存在NaN或无穷值。"
        )


    return {
        "prediction": (
            fixed_prediction
        ),

        "prediction_sha256": (
            prediction_sha256
        ),

        "prediction_fixed_at": (
            prediction_fixed_at
        ),

        "y_valid": (
            y_inner_valid
        ),

        "validation_target_loaded_at": (
            validation_target_loaded_at
        ),

        "validation_target_file_opened": True,

        "validation_target_open_stage": (
            "after_prediction_fixed"
        ),
    }


# ============================================================
# 4. 一次Trial只建议一组超参数
# ============================================================

def suggest_model_parameters_once(
    model_name,
    trial,
):
    """每个Trial只生成一组模型超参数。"""

    if model_name == "ExtraTrees":
        return suggest_extratrees_parameters(
            trial
        )


    if model_name == "LightGBM":
        return suggest_lightgbm_parameters(
            trial
        )


    if model_name == "Transformer":
        return suggest_transformer_parameters(
            trial
        )


    raise KeyError(
        f"未知模型：{model_name}"
    )


# ============================================================
# 5. 使用同一组参数训练一个内层折
# ============================================================

def fit_predict_one_inner_fold(
    model_name,
    search_parameters,
    prepared_fold,
    split_name,
    outer_fold,
    inner_fold,
    trial_number,
):
    """
    仅使用训练特征和训练目标拟合模型。

    本函数不接收验证目标。
    """

    fold_seed = make_trial_seed(
        split_name=split_name,
        outer_fold=outer_fold,
        model_name=model_name,
        trial_number=trial_number,
        inner_fold=inner_fold,
    )


    X_train = prepared_fold[
        "X_train"
    ]

    X_valid = prepared_fold[
        "X_valid"
    ]

    y_train = prepared_fold[
        "y_train"
    ]


    if model_name == "ExtraTrees":
        formal_parameters = (
            extratrees_search_to_formal(
                search_parameters=(
                    search_parameters
                ),

                seed=(
                    fold_seed
                ),
            )
        )


        model = ExtraTreesRegressor(
            **formal_parameters
        )


        model.fit(
            X_train,
            y_train,
        )


        validation_prediction = (
            model.predict(
                X_valid
            )
        )


        training_detail = {
            "epochs_completed": None,

            "final_train_mse": None,

            "validation_target_argument_exists": (
                False
            ),
        }


        del model


    elif model_name == "LightGBM":
        formal_parameters = (
            lightgbm_search_to_formal(
                search_parameters=(
                    search_parameters
                ),

                seed=(
                    fold_seed
                ),
            )
        )


        model = LGBMRegressor(
            **formal_parameters
        )


        model.fit(
            X_train,
            y_train,
        )


        validation_prediction = (
            model.predict(
                X_valid
            )
        )


        training_detail = {
            "epochs_completed": None,

            "final_train_mse": None,

            "validation_target_argument_exists": (
                False
            ),
        }


        del model


    elif model_name == "Transformer":
        formal_parameters = (
            transformer_search_to_formal(
                search_parameters=(
                    search_parameters
                ),

                seed=(
                    fold_seed
                ),
            )
        )


        transformer_result = (
            train_predict_transformer(
                X_train=X_train,

                y_train=y_train,

                X_validation=X_valid,

                parameters=(
                    formal_parameters
                ),

                seed=(
                    fold_seed
                ),

                epochs=(
                    TRANSFORMER_SEARCH_EPOCHS
                ),
            )
        )


        validation_prediction = (
            transformer_result[
                "validation_prediction"
            ]
        )


        training_detail = {
            "epochs_completed": int(
                transformer_result[
                    "epochs_completed"
                ]
            ),

            "final_train_mse": float(
                transformer_result[
                    "final_train_mse"
                ]
            ),

            "validation_target_argument_exists": bool(
                transformer_result[
                    "validation_target_argument_exists"
                ]
            ),

            "validation_target_received": bool(
                transformer_result[
                    "validation_target_received"
                ]
            ),

            "validation_used_for_backpropagation": bool(
                transformer_result[
                    "validation_used_for_backpropagation"
                ]
            ),

            "validation_used_for_early_stopping": bool(
                transformer_result[
                    "validation_used_for_early_stopping"
                ]
            ),

            "validation_used_for_scheduler": bool(
                transformer_result[
                    "validation_used_for_scheduler"
                ]
            ),
        }


        if any(
            [
                training_detail[
                    "validation_target_argument_exists"
                ],

                training_detail[
                    "validation_target_received"
                ],

                training_detail[
                    "validation_used_for_backpropagation"
                ],

                training_detail[
                    "validation_used_for_early_stopping"
                ],

                training_detail[
                    "validation_used_for_scheduler"
                ],
            ]
        ):
            raise RuntimeError(
                "Transformer训练接口显示"
                "验证标签参与了训练流程。"
            )


        del transformer_result


    else:
        raise KeyError(
            f"未知模型：{model_name}"
        )


    validation_prediction = np.asarray(
        validation_prediction,
        dtype=np.float32,
    ).reshape(-1)


    if len(
        validation_prediction
    ) != prepared_fold[
        "valid_rows"
    ]:
        raise RuntimeError(
            "验证预测行数错误。"
        )


    if not np.isfinite(
        validation_prediction
    ).all():
        raise RuntimeError(
            "验证预测中存在NaN或无穷值。"
        )


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return {
        "validation_prediction": (
            validation_prediction
        ),

        "formal_parameters": (
            formal_parameters
        ),

        "fold_seed": int(
            fold_seed
        ),

        "training_detail": (
            training_detail
        ),

        "validation_target_received": False,
    }


# ============================================================
# 6. 计算一个Trial在3个内层折上的结果
# ============================================================

def evaluate_trial_on_outer_fold(
    trial,
    model_name,
    split_name,
    outer_fold,
):
    """
    同一组参数在当前外层训练折中的3个内层折上验证。

    目标值：
    3个内层折验证R²的等权平均。
    """

    if model_name not in MODEL_ORDER:
        raise KeyError(
            f"未知模型：{model_name}"
        )


    if split_name not in SPLIT_ORDER:
        raise KeyError(
            f"未知划分方式：{split_name}"
        )


    if int(
        outer_fold
    ) not in range(
        1,
        N_OUTER_FOLDS + 1,
    ):
        raise ValueError(
            f"外层折编号错误：{outer_fold}"
        )


    search_parameters = (
        suggest_model_parameters_once(
            model_name=model_name,
            trial=trial,
        )
    )


    fold_metric_records = []

    fold_audit_records = []


    trial_started_at = (
        datetime.now().isoformat(
            timespec="seconds"
        )
    )


    for inner_fold in range(
        1,
        N_INNER_FOLDS + 1,
    ):
        # ----------------------------------------------------
        # A. 预测前：不读取验证目标
        # ----------------------------------------------------

        prepared_fold = (
            load_inner_fold_before_prediction(
                split_name=split_name,
                outer_fold=outer_fold,
                inner_fold=inner_fold,
            )
        )


        if prepared_fold[
            "validation_target_file_opened"
        ]:
            raise RuntimeError(
                "模型训练前已打开验证目标。"
            )


        # ----------------------------------------------------
        # B. 仅用训练数据拟合，使用验证特征预测
        # ----------------------------------------------------

        prediction_result = (
            fit_predict_one_inner_fold(
                model_name=model_name,

                search_parameters=(
                    search_parameters
                ),

                prepared_fold=(
                    prepared_fold
                ),

                split_name=split_name,

                outer_fold=outer_fold,

                inner_fold=inner_fold,

                trial_number=int(
                    trial.number
                ),
            )
        )


        if prediction_result[
            "validation_target_received"
        ]:
            raise RuntimeError(
                "模型拟合函数接收了验证目标。"
            )


        # ----------------------------------------------------
        # C. 预测固定后，才读取验证目标
        # ----------------------------------------------------

        delayed_validation = (
            load_validation_target_after_prediction(
                prediction=(
                    prediction_result[
                        "validation_prediction"
                    ]
                ),

                validation_target_path=(
                    prepared_fold[
                        "validation_target_path"
                    ]
                ),

                expected_rows=(
                    prepared_fold[
                        "valid_rows"
                    ]
                ),
            )
        )


        if (
            delayed_validation[
                "validation_target_open_stage"
            ]
            != "after_prediction_fixed"
        ):
            raise RuntimeError(
                "验证目标读取时机错误。"
            )


        # ----------------------------------------------------
        # D. 只在此处使用验证目标计算指标
        # ----------------------------------------------------

        metrics = calculate_metrics(
            y_true=(
                delayed_validation[
                    "y_valid"
                ]
            ),

            y_pred=(
                delayed_validation[
                    "prediction"
                ]
            ),
        )


        fold_metric_record = {
            "Inner_fold": int(
                inner_fold
            ),

            "R2": float(
                metrics[
                    "R2"
                ]
            ),

            "RMSE": float(
                metrics[
                    "RMSE"
                ]
            ),

            "MAE": float(
                metrics[
                    "MAE"
                ]
            ),

            "Pearson_r": float(
                metrics[
                    "Pearson_r"
                ]
            ),
        }


        fold_metric_records.append(
            fold_metric_record
        )


        fold_audit_records.append(
            {
                "Split": split_name,

                "Outer_fold": int(
                    outer_fold
                ),

                "Inner_fold": int(
                    inner_fold
                ),

                "Model": model_name,

                "Trial_number": int(
                    trial.number
                ),

                "Training_rows": int(
                    prepared_fold[
                        "train_rows"
                    ]
                ),

                "Validation_rows": int(
                    prepared_fold[
                        "valid_rows"
                    ]
                ),

                "Scaler_fitted_rows": int(
                    prepared_fold[
                        "scaler_fitted_rows"
                    ]
                ),

                "Opened_before_prediction": (
                    prepared_fold[
                        "opened_file_roles_before_prediction"
                    ]
                ),

                "Validation_target_opened_"
                "before_prediction": False,

                "Prediction_sha256": (
                    delayed_validation[
                        "prediction_sha256"
                    ]
                ),

                "Prediction_fixed_at": (
                    delayed_validation[
                        "prediction_fixed_at"
                    ]
                ),

                "Validation_target_loaded_at": (
                    delayed_validation[
                        "validation_target_loaded_at"
                    ]
                ),

                "Validation_target_open_stage": (
                    delayed_validation[
                        "validation_target_open_stage"
                    ]
                ),

                "Validation_used_for_scaler_fit": (
                    False
                ),

                "Validation_target_used_for_"
                "model_training": False,

                "Outer_test_used": False,

                "Fold_seed": int(
                    prediction_result[
                        "fold_seed"
                    ]
                ),

                "Metrics": (
                    metrics
                ),
            }
        )


        partial_mean_r2 = float(
            np.mean(
                [
                    row[
                        "R2"
                    ]
                    for row in fold_metric_records
                ]
            )
        )


        trial.report(
            partial_mean_r2,
            step=int(
                inner_fold
                - 1
            ),
        )


        del (
            prepared_fold,
            prediction_result,
            delayed_validation,
        )


        gc.collect()


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    aggregated_metrics = (
        aggregate_inner_fold_metrics(
            fold_metric_records
        )
    )


    for metric_name in METRIC_ORDER:
        trial.set_user_attr(
            f"Mean_{metric_name}",
            float(
                aggregated_metrics[
                    f"Mean_{metric_name}"
                ]
            ),
        )


        trial.set_user_attr(
            f"Std_{metric_name}",
            float(
                aggregated_metrics[
                    f"Std_{metric_name}"
                ]
            ),
        )


    for fold_record in fold_metric_records:
        inner_fold = int(
            fold_record[
                "Inner_fold"
            ]
        )


        for metric_name in METRIC_ORDER:
            trial.set_user_attr(
                (
                    f"InnerFold{inner_fold}_"
                    f"{metric_name}"
                ),

                float(
                    fold_record[
                        metric_name
                    ]
                ),
            )


    trial.set_user_attr(
        "Split",
        split_name,
    )


    trial.set_user_attr(
        "Outer_fold",
        int(
            outer_fold
        ),
    )


    trial.set_user_attr(
        "Model",
        model_name,
    )


    trial.set_user_attr(
        "Inner_fold_count",
        int(
            N_INNER_FOLDS
        ),
    )


    trial.set_user_attr(
        "Validation_target_read_stage",
        "after_prediction_fixed",
    )


    trial.set_user_attr(
        "Outer_test_used",
        False,
    )


    trial.set_user_attr(
        "Old_global_feature_read",
        False,
    )


    trial_output_dir = (
        TRIAL_RESULT_DIR
        / split_name
        / f"OuterFold_{outer_fold}"
        / model_name
    )


    trial_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    trial_output_path = (
        trial_output_dir
        / (
            f"trial_{int(trial.number):04d}_"
            "inner_fold_results.json"
        )
    )


    trial_payload = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "trial_started_at": (
            trial_started_at
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "split": split_name,

        "outer_fold": int(
            outer_fold
        ),

        "model": model_name,

        "trial_number": int(
            trial.number
        ),

        "search_parameters": (
            search_parameters
        ),

        "fold_metrics": (
            fold_metric_records
        ),

        "aggregated_metrics": (
            aggregated_metrics
        ),

        "fold_audits": (
            fold_audit_records
        ),

        "leakage_control": {
            "validation_target_opened_before_"
            "prediction": False,

            "validation_target_opened_after_"
            "prediction_fixed": True,

            "validation_target_used_for_"
            "training": False,

            "validation_used_for_scaler_fit": (
                False
            ),

            "outer_test_used": False,

            "old_global_feature_read": False,

            "external_10point_holdout_read": (
                False
            ),
        },
    }


    save_json_atomic(
        data=trial_payload,
        output_path=trial_output_path,
    )


    return float(
        aggregated_metrics[
            "Objective_value"
        ]
    )


# ============================================================
# 7. 生成Optuna调参签名
# ============================================================

input_integrity_columns = [
    "Task_id",
    "Split",
    "Outer_fold",
    "Inner_fold",
    "X_inner_train_sha256",
    "X_inner_valid_sha256",
    "y_inner_train_sha256",
    "y_inner_valid_sha256",
]


input_integrity_text = (
    df_inner_array_audit[
        input_integrity_columns
    ]
    .sort_values(
        [
            "Split",
            "Outer_fold",
            "Inner_fold",
        ]
    )
    .to_csv(
        index=False
    )
)


INPUT_INTEGRITY_DIGEST = (
    hashlib.sha256(
        input_integrity_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


tuning_signature_content = {
    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "input_integrity_digest": (
        INPUT_INTEGRITY_DIGEST
    ),

    "splits": (
        SPLIT_ORDER
    ),

    "models": (
        MODEL_ORDER
    ),

    "outer_folds": int(
        N_OUTER_FOLDS
    ),

    "inner_folds": int(
        N_INNER_FOLDS
    ),

    "objective": {
        "metric": (
            OPTUNA_OBJECTIVE_METRIC
        ),

        "direction": (
            OPTUNA_DIRECTION
        ),

        "aggregation": (
            INNER_FOLD_AGGREGATION
        ),
    },

    "trial_budgets": (
        N_TRIALS_PER_OUTER_FOLD
    ),

    "tree_search_spaces": (
        TREE_SEARCH_SPACE_DESCRIPTION
    ),

    "transformer_search_space": (
        TRANSFORMER_SEARCH_SPACE_DESCRIPTION
    ),

    "transformer_search_epochs": int(
        TRANSFORMER_SEARCH_EPOCHS
    ),

    "validation_target_read_policy": (
        "after_prediction_fixed_only"
    ),

    "outer_test_used": False,

    "external_10point_holdout_used": False,
}


tuning_signature_text = json.dumps(
    to_jsonable(
        tuning_signature_content
    ),
    ensure_ascii=False,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
)


TUNING_SIGNATURE = (
    hashlib.sha256(
        tuning_signature_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


STUDY_SUFFIX = (
    TUNING_SIGNATURE[
        :12
    ]
)


# ============================================================
# 8. Study名称与创建函数
# ============================================================

def get_study_name(
    split_name,
    outer_fold,
    model_name,
):
    """返回唯一Study名称。"""

    return (
        f"viscosity_"
        f"{model_name.lower()}_"
        f"{split_name.lower()}_"
        f"outerfold{int(outer_fold)}_"
        f"{STUDY_SUFFIX}"
    )


def get_study_sampler_seed(
    split_name,
    outer_fold,
    model_name,
):
    """为每个Study生成稳定TPE采样种子。"""

    return make_trial_seed(
        split_name=split_name,
        outer_fold=outer_fold,
        model_name=model_name,
        trial_number=0,
        inner_fold=0,
    )


def create_or_load_outer_fold_study(
    split_name,
    outer_fold,
    model_name,
):
    """创建或恢复一个划分×外层折×模型的Study。"""

    storage_url = (
        "sqlite:///"
        + OPTUNA_DB_PATH.resolve().as_posix()
    )


    storage = optuna.storages.RDBStorage(
        url=storage_url,

        engine_kwargs={
            "connect_args": {
                "timeout": 120,
            }
        },
    )


    sampler = optuna.samplers.TPESampler(
        seed=get_study_sampler_seed(
            split_name=split_name,
            outer_fold=outer_fold,
            model_name=model_name,
        ),

        n_startup_trials=min(
            5,
            int(
                N_TRIALS_PER_OUTER_FOLD[
                    model_name
                ]
            ),
        ),
    )


    # 每组参数必须完整评价3个内层折，因此不提前剪枝。
    pruner = optuna.pruners.NopPruner()


    study = optuna.create_study(
        study_name=get_study_name(
            split_name=split_name,
            outer_fold=outer_fold,
            model_name=model_name,
        ),

        storage=storage,

        direction=OPTUNA_DIRECTION,

        sampler=sampler,

        pruner=pruner,

        load_if_exists=True,
    )


    return study


def make_outer_fold_objective(
    split_name,
    outer_fold,
    model_name,
):
    """生成一个固定划分、外层折和模型的目标函数。"""

    def objective(
        trial,
    ):
        return evaluate_trial_on_outer_fold(
            trial=trial,

            model_name=model_name,

            split_name=split_name,

            outer_fold=outer_fold,
        )


    return objective


def count_completed_trials(
    study,
):
    """统计成功完成的Trial数量。"""

    return int(
        sum(
            trial.state
            == optuna.trial.TrialState.COMPLETE
            for trial in study.trials
        )
    )


def count_failed_trials(
    study,
):
    """统计失败的Trial数量。"""

    return int(
        sum(
            trial.state
            == optuna.trial.TrialState.FAIL
            for trial in study.trials
        )
    )


def save_study_trials(
    study,
    output_path,
):
    """保存Study中的全部Trial明细。"""

    output_path = Path(
        output_path
    )


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    trial_frame = study.trials_dataframe(
        attrs=(
            "number",
            "value",
            "datetime_start",
            "datetime_complete",
            "duration",
            "params",
            "user_attrs",
            "state",
        )
    )


    trial_frame.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )


    return trial_frame


# ============================================================
# 9. 生成30个Study运行计划
# ============================================================

study_plan_rows = []


for split_name in SPLIT_ORDER:
    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):
        for model_name in MODEL_ORDER:
            study_plan_rows.append(
                {
                    "Split": split_name,

                    "Outer_fold": int(
                        outer_fold
                    ),

                    "Model": model_name,

                    "Study_name": (
                        get_study_name(
                            split_name=split_name,
                            outer_fold=outer_fold,
                            model_name=model_name,
                        )
                    ),

                    "Trial_budget": int(
                        N_TRIALS_PER_OUTER_FOLD[
                            model_name
                        ]
                    ),

                    "Inner_folds_per_trial": int(
                        N_INNER_FOLDS
                    ),

                    "Objective_metric": (
                        OPTUNA_OBJECTIVE_METRIC
                    ),

                    "Objective_direction": (
                        OPTUNA_DIRECTION
                    ),

                    "Validation_target_read_policy": (
                        "after_prediction_fixed_only"
                    ),

                    "Optuna_started_in_Cell_4": (
                        False
                    ),
                }
            )


df_study_plan = (
    pd.DataFrame(
        study_plan_rows
    )
    .sort_values(
        [
            "Split",
            "Outer_fold",
            "Model",
        ],

        key=lambda series: (
            series.map(
                {
                    "Data_split": 0,
                    "IL_split": 1,
                    "ExtraTrees": 0,
                    "LightGBM": 1,
                    "Transformer": 2,
                }
            )
            if series.name in [
                "Split",
                "Model",
            ]
            else series
        ),
    )
    .reset_index(
        drop=True
    )
)


if len(
    df_study_plan
) != 30:
    raise RuntimeError(
        "Study计划数量不是30。"
    )


expected_total_trials = int(
    df_study_plan[
        "Trial_budget"
    ].sum()
)


expected_total_inner_fits = int(
    (
        df_study_plan[
            "Trial_budget"
        ]
        * df_study_plan[
            "Inner_folds_per_trial"
        ]
    ).sum()
)


if expected_total_trials != 450:
    raise RuntimeError(
        "总Trial预算不是450。"
    )


if expected_total_inner_fits != 1350:
    raise RuntimeError(
        "总内层模型拟合次数不是1350。"
    )


df_study_plan.to_csv(
    STUDY_PLAN_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 10. 静态接口检查
# ============================================================

transformer_training_signature = (
    inspect.signature(
        train_predict_transformer
    )
)


transformer_training_parameters = set(
    transformer_training_signature
    .parameters.keys()
)


for forbidden_parameter_name in [
    "y_validation",
    "y_valid",
    "validation_target",
]:
    if forbidden_parameter_name in (
        transformer_training_parameters
    ):
        raise RuntimeError(
            "Transformer训练函数意外包含"
            f"{forbidden_parameter_name}参数。"
        )


if set(
    transformer_training_parameters
) != {
    "X_train",
    "y_train",
    "X_validation",
    "parameters",
    "seed",
    "epochs",
}:
    raise RuntimeError(
        "Transformer训练函数参数列表"
        "与预期不一致。"
    )


# ============================================================
# 11. 保存调参签名和目标函数定义
# ============================================================

tuning_signature_payload = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "tuning_signature_created"
    ),

    "tuning_signature": (
        TUNING_SIGNATURE
    ),

    "study_suffix": (
        STUDY_SUFFIX
    ),

    "input_integrity_digest": (
        INPUT_INTEGRITY_DIGEST
    ),

    "signature_content": (
        tuning_signature_content
    ),

    "outputs": {
        "study_plan": str(
            STUDY_PLAN_PATH
        ),

        "signature_file": str(
            TUNING_SIGNATURE_PATH
        ),
    },
}


save_json_atomic(
    data=tuning_signature_payload,
    output_path=TUNING_SIGNATURE_PATH,
)


objective_definition_payload = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "optuna_objective_definition_passed"
    ),

    "study_design": {
        "study_count": int(
            len(
                df_study_plan
            )
        ),

        "splits": (
            SPLIT_ORDER
        ),

        "outer_folds_per_split": int(
            N_OUTER_FOLDS
        ),

        "models": (
            MODEL_ORDER
        ),

        "inner_folds_per_trial": int(
            N_INNER_FOLDS
        ),

        "total_trial_budget": int(
            expected_total_trials
        ),

        "total_inner_model_fits": int(
            expected_total_inner_fits
        ),
    },

    "objective": {
        "metric": (
            OPTUNA_OBJECTIVE_METRIC
        ),

        "direction": (
            OPTUNA_DIRECTION
        ),

        "aggregation": (
            INNER_FOLD_AGGREGATION
        ),

        "equal_inner_fold_weight": True,

        "pruner": "NopPruner",
    },

    "validation_target_policy": {
        "opened_before_prediction": False,

        "opened_after_prediction_fixed": True,

        "used_for_metric_calculation": True,

        "used_for_scaler_fit": False,

        "used_for_model_fit": False,

        "used_for_backpropagation": False,

        "used_for_early_stopping": False,

        "used_for_scheduler": False,
    },

    "training_function_interface": {
        "transformer_parameters": sorted(
            transformer_training_parameters
        ),

        "validation_target_argument_exists": (
            False
        ),
    },

    "leakage_control": {
        "inner_fold_specific_mpnn_features": (
            True
        ),

        "outer_test_used_for_tuning": False,

        "outer_test_target_read": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "cell_4_execution": {
        "model_fitted": False,

        "validation_target_file_opened": False,

        "optuna_study_created": False,

        "optuna_trial_started": False,
    },

    "outputs": {
        "objective_definition": str(
            OPTUNA_OBJECTIVE_DEFINITION_PATH
        ),

        "tuning_signature": str(
            TUNING_SIGNATURE_PATH
        ),

        "study_plan": str(
            STUDY_PLAN_PATH
        ),

        "optuna_database": str(
            OPTUNA_DB_PATH
        ),
    },
}


save_json_atomic(
    data=objective_definition_payload,
    output_path=(
        OPTUNA_OBJECTIVE_DEFINITION_PATH
    ),
)


# ============================================================
# 12. 输出检查
# ============================================================

print("=" * 104)
print("Cell 4 Optuna目标函数、调参签名和30个Study计划检查通过 ✓")
print("=" * 104)

print(
    "study count                         =",
    len(
        df_study_plan
    ),
)

print(
    "splits                              =",
    SPLIT_ORDER,
)

print(
    "outer folds/split                   =",
    N_OUTER_FOLDS,
)

print(
    "models                              =",
    MODEL_ORDER,
)

print(
    "inner folds/trial                   =",
    N_INNER_FOLDS,
)

print(
    "total trial budget                  =",
    expected_total_trials,
)

print(
    "expected total inner model fits     =",
    expected_total_inner_fits,
)

print(
    "objective metric                    =",
    OPTUNA_OBJECTIVE_METRIC,
)

print(
    "objective direction                 =",
    OPTUNA_DIRECTION,
)

print(
    "inner-fold aggregation              =",
    INNER_FOLD_AGGREGATION,
)

print(
    "pruner                              =",
    "NopPruner",
)

print(
    "validation target read policy       =",
    "after_prediction_fixed_only",
)

print(
    "validation target opened in Cell 4  =",
    False,
)

print(
    "models fitted in Cell 4             =",
    False,
)

print(
    "Optuna studies created in Cell 4    =",
    False,
)

print(
    "Optuna trials started in Cell 4     =",
    False,
)

print(
    "Transformer validation target arg   =",
    False,
)

print(
    "outer test used for tuning          =",
    False,
)

print(
    "outer test target read              =",
    False,
)

print(
    "old global MPNN feature read        =",
    False,
)

print(
    "external 10 points read             =",
    False,
)

print(
    "input integrity digest              =",
    INPUT_INTEGRITY_DIGEST,
)

print(
    "tuning signature                    =",
    TUNING_SIGNATURE,
)

print(
    "study suffix                        =",
    STUDY_SUFFIX,
)


print(
    "\nStudy数量汇总："
)

display(
    df_study_plan.groupby(
        [
            "Split",
            "Model",
        ],
        as_index=False,
    ).agg(
        Study_count=(
            "Study_name",
            "size",
        ),

        Trial_budget_total=(
            "Trial_budget",
            "sum",
        ),

        Inner_model_fits=(
            "Inner_folds_per_trial",
            lambda values: int(
                np.sum(
                    values.to_numpy()
                    * df_study_plan.loc[
                        values.index,
                        "Trial_budget",
                    ].to_numpy()
                )
            ),
        ),
    )
)


print(
    "\nOptuna目标函数定义："
)

print(
    OPTUNA_OBJECTIVE_DEFINITION_PATH
)


print(
    "\n调参签名："
)

print(
    TUNING_SIGNATURE_PATH
)


print(
    "\n30个Study计划："
)

print(
    STUDY_PLAN_PATH
)

Cell 4 Optuna目标函数、调参签名和30个Study计划检查通过 ✓
study count                         = 30
splits                              = ['Data_split', 'IL_split']
outer folds/split                   = 5
models                              = ['ExtraTrees', 'LightGBM', 'Transformer']
inner folds/trial                   = 3
total trial budget                  = 450
expected total inner model fits     = 1350
objective metric                    = R2
objective direction                 = maximize
inner-fold aggregation              = equal_weight_mean_across_3_inner_folds
pruner                              = NopPruner
validation target read policy       = after_prediction_fixed_only
validation target opened in Cell 4  = False
models fitted in Cell 4             = False
Optuna studies created in Cell 4    = False
Optuna trials started in Cell 4     = False
Transformer validation target arg   = False
outer test used for tuning          = False
outer test target read              = False
old global MPNN featur

,Split,Model,Study_count,Trial_budget_total,Inner_model_fits
0,Data_split,ExtraTrees,5,75,225
1,Data_split,LightGBM,5,100,300
2,Data_split,Transformer,5,50,150
3,IL_split,ExtraTrees,5,75,225
4,IL_split,LightGBM,5,100,300
5,IL_split,Transformer,5,50,150



Optuna目标函数定义：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\optuna_objective_definition_Data_IL.json

调参签名：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\optuna_tuning_signature_Data_IL.json

30个Study计划：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\optuna_study_plan_Data_IL.csv


In [6]:
# Cell 5 | 对两种划分和三个模型执行不消耗正式Trial预算的端到端烟雾测试


import shutil


SMOKE_TEST_ROOT = (
    TUNING_LOG_DIR
    / "objective_smoke_tests"
)


SMOKE_TRIAL_RESULT_DIR = (
    SMOKE_TEST_ROOT
    / "trial_results"
)


SMOKE_SUMMARY_PATH = (
    SMOKE_TEST_ROOT
    / "optuna_objective_smoke_test_summary_Data_IL.csv"
)


SMOKE_MANIFEST_PATH = (
    SMOKE_TEST_ROOT
    / "optuna_objective_smoke_test_manifest_Data_IL.json"
)


# ============================================================
# 1. 清理上一次烟雾测试，不影响正式调参目录
# ============================================================

if SMOKE_TEST_ROOT.exists():
    shutil.rmtree(
        SMOKE_TEST_ROOT
    )


SMOKE_TRIAL_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. 烟雾测试固定参数
# ============================================================

SMOKE_FIXED_PARAMETERS = {
    "ExtraTrees": {
        "bootstrap": False,

        "n_estimators": 200,

        "max_depth": 20,

        "min_samples_split": 4,

        "min_samples_leaf": 2,

        "max_features": "0.75",
    },

    "LightGBM": {
        "n_estimators": 200,

        "learning_rate": 0.05,

        "max_depth": 10,

        "num_leaves": 43,

        "min_child_samples": 20,

        "subsample": 0.80,

        "colsample_bytree": 0.85,

        "reg_alpha": 0.01,

        "reg_lambda": 0.10,

        "min_split_gain": 0.02,
    },

    "Transformer": {
        "hidden_dim": 64,

        "num_heads": 2,

        "num_layers": 2,

        "lr": 0.001,

        "dropout": 0.30,

        "weight_decay": 0.001,

        "batch_size": 128,
    },
}


SMOKE_TRIAL_NUMBER_MAP = {
    ("Data_split", "ExtraTrees"): 910001,

    ("Data_split", "LightGBM"): 910002,

    ("Data_split", "Transformer"): 910003,

    ("IL_split", "ExtraTrees"): 920001,

    ("IL_split", "LightGBM"): 920002,

    ("IL_split", "Transformer"): 920003,
}


SMOKE_OUTER_FOLD = 1

SMOKE_TRANSFORMER_EPOCHS = 1


# ============================================================
# 3. 审计一份烟雾测试Trial输出
# ============================================================

def audit_one_smoke_trial_payload(
    payload,
    split_name,
    model_name,
    outer_fold,
    trial_number,
):
    """检查一次烟雾测试的完整数据流与标签读取顺序。"""

    if payload.get(
        "split"
    ) != split_name:
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "输出中的划分名称错误。"
        )


    if payload.get(
        "model"
    ) != model_name:
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "输出中的模型名称错误。"
        )


    if int(
        payload.get(
            "outer_fold",
            -1,
        )
    ) != int(
        outer_fold
    ):
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "输出中的外层折编号错误。"
        )


    if int(
        payload.get(
            "trial_number",
            -1,
        )
    ) != int(
        trial_number
    ):
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "输出中的Trial编号错误。"
        )


    fold_metrics = payload.get(
        "fold_metrics",
        [],
    )


    fold_audits = payload.get(
        "fold_audits",
        [],
    )


    aggregated_metrics = payload.get(
        "aggregated_metrics",
        {},
    )


    if len(
        fold_metrics
    ) != N_INNER_FOLDS:
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "内层折指标记录数不是3。"
        )


    if len(
        fold_audits
    ) != N_INNER_FOLDS:
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "内层折审计记录数不是3。"
        )


    expected_opened_roles = {
        "X_inner_train",
        "y_inner_train",
        "X_inner_valid",
    }


    for fold_audit in fold_audits:

        opened_roles = set(
            fold_audit.get(
                "Opened_before_prediction",
                [],
            )
        )


        if opened_roles != expected_opened_roles:
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "预测固定前读取的文件角色错误。"
            )


        if fold_audit.get(
            "Validation_target_opened_"
            "before_prediction"
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "预测固定前打开了验证标签。"
            )


        if (
            fold_audit.get(
                "Validation_target_open_stage"
            )
            != "after_prediction_fixed"
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "验证标签读取阶段错误。"
            )


        if fold_audit.get(
            "Validation_used_for_scaler_fit"
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "验证数据参与了标准化器拟合。"
            )


        if fold_audit.get(
            "Validation_target_used_for_"
            "model_training"
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "验证标签参与了模型训练。"
            )


        if fold_audit.get(
            "Outer_test_used"
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "烟雾测试使用了外层测试集。"
            )


        training_rows = int(
            fold_audit.get(
                "Training_rows",
                -1,
            )
        )


        scaler_fitted_rows = int(
            fold_audit.get(
                "Scaler_fitted_rows",
                -2,
            )
        )


        if (
            training_rows
            != scaler_fitted_rows
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "标准化器拟合行数"
                "与内层训练行数不一致。"
            )


        prediction_fixed_at = (
            fold_audit.get(
                "Prediction_fixed_at"
            )
        )


        validation_loaded_at = (
            fold_audit.get(
                "Validation_target_loaded_at"
            )
        )


        if not (
            prediction_fixed_at
            and validation_loaded_at
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "缺少预测固定或验证标签读取时间。"
            )


        if (
            validation_loaded_at
            < prediction_fixed_at
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "验证标签读取时间早于预测固定时间。"
            )


        prediction_sha256 = str(
            fold_audit.get(
                "Prediction_sha256",
                "",
            )
        )


        if len(
            prediction_sha256
        ) != 64:
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "预测数组SHA256长度错误。"
            )


    metric_rows = []


    for metric_record in fold_metrics:

        metric_row = {
            metric_name: float(
                metric_record[
                    metric_name
                ]
            )
            for metric_name
            in METRIC_ORDER
        }


        if not np.isfinite(
            np.asarray(
                list(
                    metric_row.values()
                ),
                dtype=np.float64,
            )
        ).all():
            raise RuntimeError(
                f"{split_name}/{model_name}："
                "验证指标中存在NaN或无穷值。"
            )


        metric_rows.append(
            metric_row
        )


    expected_mean_r2 = float(
        np.mean(
            [
                row[
                    "R2"
                ]
                for row in metric_rows
            ]
        )
    )


    actual_objective_value = float(
        aggregated_metrics[
            "Objective_value"
        ]
    )


    if not np.isclose(
        expected_mean_r2,
        actual_objective_value,
        rtol=1e-10,
        atol=1e-10,
    ):
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "目标值不是3个内层折R²的等权平均。"
        )


    leakage_control = payload.get(
        "leakage_control",
        {},
    )


    expected_false_fields = [
        "validation_target_opened_before_prediction",
        "validation_target_used_for_training",
        "validation_used_for_scaler_fit",
        "outer_test_used",
        "old_global_feature_read",
        "external_10point_holdout_read",
    ]


    for field_name in expected_false_fields:
        if leakage_control.get(
            field_name
        ):
            raise RuntimeError(
                f"{split_name}/{model_name}："
                f"泄露控制字段{field_name}=True。"
            )


    if not leakage_control.get(
        "validation_target_opened_after_"
        "prediction_fixed"
    ):
        raise RuntimeError(
            f"{split_name}/{model_name}："
            "没有确认验证标签在预测固定后读取。"
        )


    return {
        "Split": split_name,

        "Outer_fold": int(
            outer_fold
        ),

        "Model": model_name,

        "Smoke_trial_number": int(
            trial_number
        ),

        "Inner_fold_count": int(
            len(
                fold_metrics
            )
        ),

        "Mean_R2": float(
            aggregated_metrics[
                "Mean_R2"
            ]
        ),

        "Std_R2": float(
            aggregated_metrics[
                "Std_R2"
            ]
        ),

        "Mean_RMSE": float(
            aggregated_metrics[
                "Mean_RMSE"
            ]
        ),

        "Std_RMSE": float(
            aggregated_metrics[
                "Std_RMSE"
            ]
        ),

        "Mean_MAE": float(
            aggregated_metrics[
                "Mean_MAE"
            ]
        ),

        "Std_MAE": float(
            aggregated_metrics[
                "Std_MAE"
            ]
        ),

        "Mean_Pearson_r": float(
            aggregated_metrics[
                "Mean_Pearson_r"
            ]
        ),

        "Std_Pearson_r": float(
            aggregated_metrics[
                "Std_Pearson_r"
            ]
        ),

        "Prediction_fixed_before_"
        "validation_target_loaded": True,

        "Validation_target_opened_"
        "before_prediction": False,

        "Validation_target_used_for_"
        "training": False,

        "Validation_used_for_scaler_fit": False,

        "Outer_test_used": False,

        "Audit_status": "passed",
    }


# ============================================================
# 4. 执行两种划分×三个模型的烟雾测试
# ============================================================

original_trial_result_dir = Path(
    TRIAL_RESULT_DIR
)


original_transformer_search_epochs = int(
    TRANSFORMER_SEARCH_EPOCHS
)


smoke_summary_rows = []

smoke_trial_payload_paths = []


try:

    # 将临时Trial输出重定向到独立烟雾测试目录。
    TRIAL_RESULT_DIR = (
        SMOKE_TRIAL_RESULT_DIR
    )


    # Transformer烟雾测试只运行1轮。
    TRANSFORMER_SEARCH_EPOCHS = int(
        SMOKE_TRANSFORMER_EPOCHS
    )


    for split_name in SPLIT_ORDER:

        for model_name in MODEL_ORDER:

            smoke_trial_number = int(
                SMOKE_TRIAL_NUMBER_MAP[
                    (
                        split_name,
                        model_name,
                    )
                ]
            )


            fixed_parameters = dict(
                SMOKE_FIXED_PARAMETERS[
                    model_name
                ]
            )


            fixed_trial = optuna.trial.FixedTrial(
                params=fixed_parameters,

                number=smoke_trial_number,
            )


            print(
                "\n"
                + "=" * 104
            )


            print(
                "开始烟雾测试：",
                split_name,
                "|",
                model_name,
                "| OuterFold",
                SMOKE_OUTER_FOLD,
            )


            print(
                "正式Optuna Study创建 =",
                False,
            )


            print(
                "正式Trial预算消耗 =",
                False,
            )


            print(
                "Transformer临时测试epochs =",
                (
                    SMOKE_TRANSFORMER_EPOCHS
                    if model_name
                    == "Transformer"
                    else "not applicable"
                ),
            )


            smoke_objective_value = (
                evaluate_trial_on_outer_fold(
                    trial=fixed_trial,

                    model_name=model_name,

                    split_name=split_name,

                    outer_fold=(
                        SMOKE_OUTER_FOLD
                    ),
                )
            )


            trial_payload_path = (
                SMOKE_TRIAL_RESULT_DIR
                / split_name
                / (
                    f"OuterFold_"
                    f"{SMOKE_OUTER_FOLD}"
                )
                / model_name
                / (
                    f"trial_"
                    f"{smoke_trial_number:04d}_"
                    "inner_fold_results.json"
                )
            )


            if not trial_payload_path.exists():
                raise FileNotFoundError(
                    "烟雾测试没有生成Trial审计文件：\n"
                    f"{trial_payload_path}"
                )


            smoke_payload = read_json(
                trial_payload_path
            )


            smoke_summary_row = (
                audit_one_smoke_trial_payload(
                    payload=smoke_payload,

                    split_name=split_name,

                    model_name=model_name,

                    outer_fold=(
                        SMOKE_OUTER_FOLD
                    ),

                    trial_number=(
                        smoke_trial_number
                    ),
                )
            )


            if not np.isclose(
                float(
                    smoke_objective_value
                ),
                float(
                    smoke_summary_row[
                        "Mean_R2"
                    ]
                ),
                rtol=1e-10,
                atol=1e-10,
            ):
                raise RuntimeError(
                    f"{split_name}/{model_name}："
                    "返回的目标值与审计汇总不一致。"
                )


            smoke_summary_rows.append(
                smoke_summary_row
            )


            smoke_trial_payload_paths.append(
                str(
                    trial_payload_path
                )
            )


            print(
                "烟雾测试完成 ✓"
            )


            print(
                "3折平均验证R2 =",
                smoke_summary_row[
                    "Mean_R2"
                ],
            )


            print(
                "预测前读取验证标签 =",
                False,
            )


            print(
                "验证标签用于训练 =",
                False,
            )


            print(
                "外层测试集使用 =",
                False,
            )


            del (
                fixed_trial,
                smoke_payload,
            )


            gc.collect()


            if torch.cuda.is_available():
                torch.cuda.empty_cache()


finally:

    # 无论运行成功或报错，都恢复正式目录和50轮搜索设置。
    TRIAL_RESULT_DIR = (
        original_trial_result_dir
    )


    TRANSFORMER_SEARCH_EPOCHS = int(
        original_transformer_search_epochs
    )


# ============================================================
# 5. 汇总检查
# ============================================================

df_smoke_summary = (
    pd.DataFrame(
        smoke_summary_rows
    )
    .sort_values(
        [
            "Split",
            "Model",
        ],

        key=lambda series: (
            series.map(
                {
                    "Data_split": 0,
                    "IL_split": 1,
                    "ExtraTrees": 0,
                    "LightGBM": 1,
                    "Transformer": 2,
                }
            )
            if series.name
            in [
                "Split",
                "Model",
            ]
            else series
        ),
    )
    .reset_index(
        drop=True
    )
)


if len(
    df_smoke_summary
) != 6:
    raise RuntimeError(
        "烟雾测试汇总行数不是6。"
    )


if not df_smoke_summary[
    "Audit_status"
].eq(
    "passed"
).all():
    raise RuntimeError(
        "部分烟雾测试没有通过。"
    )


if df_smoke_summary[
    "Validation_target_opened_"
    "before_prediction"
].any():
    raise RuntimeError(
        "部分烟雾测试在预测固定前"
        "读取了验证标签。"
    )


if df_smoke_summary[
    "Validation_target_used_for_"
    "training"
].any():
    raise RuntimeError(
        "部分烟雾测试将验证标签"
        "用于模型训练。"
    )


if df_smoke_summary[
    "Validation_used_for_scaler_fit"
].any():
    raise RuntimeError(
        "部分烟雾测试将验证数据"
        "用于标准化器拟合。"
    )


if df_smoke_summary[
    "Outer_test_used"
].any():
    raise RuntimeError(
        "部分烟雾测试使用了外层测试集。"
    )


if int(
    TRANSFORMER_SEARCH_EPOCHS
) != int(
    original_transformer_search_epochs
):
    raise RuntimeError(
        "Transformer正式搜索轮数"
        "没有恢复。"
    )


if Path(
    TRIAL_RESULT_DIR
) != original_trial_result_dir:
    raise RuntimeError(
        "正式Trial输出目录没有恢复。"
    )


df_smoke_summary.to_csv(
    SMOKE_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 6. 保存烟雾测试manifest
# ============================================================

smoke_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "all_objective_smoke_tests_passed"
    ),

    "test_design": {
        "splits": (
            SPLIT_ORDER
        ),

        "models": (
            MODEL_ORDER
        ),

        "outer_fold": int(
            SMOKE_OUTER_FOLD
        ),

        "smoke_case_count": int(
            len(
                df_smoke_summary
            )
        ),

        "inner_folds_per_case": int(
            N_INNER_FOLDS
        ),

        "total_smoke_model_fits": int(
            len(
                df_smoke_summary
            )
            * N_INNER_FOLDS
        ),

        "transformer_smoke_epochs": int(
            SMOKE_TRANSFORMER_EPOCHS
        ),

        "transformer_formal_search_epochs_"
        "restored": int(
            TRANSFORMER_SEARCH_EPOCHS
        ),
    },

    "formal_tuning_impact": {
        "formal_optuna_study_created": False,

        "formal_optuna_database_opened": False,

        "formal_trial_budget_consumed": False,

        "formal_trial_result_directory_used": (
            False
        ),

        "smoke_output_directory": str(
            SMOKE_TEST_ROOT
        ),
    },

    "leakage_control": {
        "all_predictions_fixed_before_"
        "validation_target_loaded": True,

        "validation_target_opened_before_"
        "prediction": False,

        "validation_target_used_for_training": (
            False
        ),

        "validation_used_for_scaler_fit": False,

        "outer_test_used": False,

        "outer_test_target_read": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "summary_records": (
        df_smoke_summary
        .to_dict(
            orient="records"
        )
    ),

    "trial_payload_paths": (
        smoke_trial_payload_paths
    ),

    "outputs": {
        "summary_csv": str(
            SMOKE_SUMMARY_PATH
        ),

        "manifest_json": str(
            SMOKE_MANIFEST_PATH
        ),
    },
}


save_json_atomic(
    data=smoke_manifest,
    output_path=(
        SMOKE_MANIFEST_PATH
    ),
)


# ============================================================
# 7. 输出检查
# ============================================================

print(
    "\n"
    + "=" * 104
)


print(
    "Cell 5 两种划分×三个模型端到端烟雾测试全部通过 ✓"
)


print(
    "=" * 104
)


print(
    "smoke cases                       =",
    len(
        df_smoke_summary
    ),
)


print(
    "inner folds/case                  =",
    N_INNER_FOLDS,
)


print(
    "total smoke model fits            =",
    int(
        len(
            df_smoke_summary
        )
        * N_INNER_FOLDS
    ),
)


print(
    "all audit statuses passed         =",
    bool(
        df_smoke_summary[
            "Audit_status"
        ].eq(
            "passed"
        ).all()
    ),
)


print(
    "prediction fixed before label read=",
    bool(
        df_smoke_summary[
            "Prediction_fixed_before_"
            "validation_target_loaded"
        ].all()
    ),
)


print(
    "validation label opened before pred=",
    bool(
        df_smoke_summary[
            "Validation_target_opened_"
            "before_prediction"
        ].any()
    ),
)


print(
    "validation label used for training=",
    bool(
        df_smoke_summary[
            "Validation_target_used_for_"
            "training"
        ].any()
    ),
)


print(
    "validation used for scaler fit    =",
    bool(
        df_smoke_summary[
            "Validation_used_for_scaler_fit"
        ].any()
    ),
)


print(
    "outer test used                   =",
    bool(
        df_smoke_summary[
            "Outer_test_used"
        ].any()
    ),
)


print(
    "formal Optuna Study created       =",
    False,
)


print(
    "formal trial budget consumed      =",
    False,
)


print(
    "formal Optuna database opened     =",
    False,
)


print(
    "Transformer smoke epochs          =",
    SMOKE_TRANSFORMER_EPOCHS,
)


print(
    "Transformer search epochs restored=",
    TRANSFORMER_SEARCH_EPOCHS,
)


print(
    "formal Trial result dir restored  =",
    Path(
        TRIAL_RESULT_DIR
    ),
)


print(
    "\n烟雾测试结果汇总："
)


display(
    df_smoke_summary[
        [
            "Split",
            "Model",
            "Inner_fold_count",
            "Mean_R2",
            "Mean_RMSE",
            "Mean_MAE",
            "Mean_Pearson_r",
            "Audit_status",
        ]
    ]
)


print(
    "\n烟雾测试汇总文件："
)


print(
    SMOKE_SUMMARY_PATH
)


print(
    "\n烟雾测试manifest："
)


print(
    SMOKE_MANIFEST_PATH
)


开始烟雾测试： Data_split | ExtraTrees | OuterFold 1
正式Optuna Study创建 = False
正式Trial预算消耗 = False
Transformer临时测试epochs = not applicable
烟雾测试完成 ✓
3折平均验证R2 = 0.9617407430790874
预测前读取验证标签 = False
验证标签用于训练 = False
外层测试集使用 = False

开始烟雾测试： Data_split | LightGBM | OuterFold 1
正式Optuna Study创建 = False
正式Trial预算消耗 = False
Transformer临时测试epochs = not applicable
烟雾测试完成 ✓
3折平均验证R2 = 0.9482573010788903
预测前读取验证标签 = False
验证标签用于训练 = False
外层测试集使用 = False

开始烟雾测试： Data_split | Transformer | OuterFold 1
正式Optuna Study创建 = False
正式Trial预算消耗 = False
Transformer临时测试epochs = 1
烟雾测试完成 ✓
3折平均验证R2 = 0.43755124234641346
预测前读取验证标签 = False
验证标签用于训练 = False
外层测试集使用 = False

开始烟雾测试： IL_split | ExtraTrees | OuterFold 1
正式Optuna Study创建 = False
正式Trial预算消耗 = False
Transformer临时测试epochs = not applicable
烟雾测试完成 ✓
3折平均验证R2 = 0.6743773280199269
预测前读取验证标签 = False
验证标签用于训练 = False
外层测试集使用 = False

开始烟雾测试： IL_split | LightGBM | OuterFold 1
正式Optuna Study创建 = False
正式Trial预算消耗 = False
Transformer临时测试epochs = not applicable
烟雾测试

,Split,Model,Inner_fold_count,Mean_R2,Mean_RMSE,Mean_MAE,Mean_Pearson_r,Audit_status
0,Data_split,ExtraTrees,3,0.961741,0.161386,0.092833,0.982301,passed
1,Data_split,LightGBM,3,0.948257,0.187651,0.100497,0.974402,passed
2,Data_split,Transformer,3,0.437551,0.619683,0.396743,0.674009,passed
3,IL_split,ExtraTrees,3,0.674377,0.470811,0.262253,0.818241,passed
4,IL_split,LightGBM,3,0.706197,0.447316,0.236226,0.836799,passed
5,IL_split,Transformer,3,0.358955,0.661240,0.437711,0.597801,passed



烟雾测试汇总文件：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\objective_smoke_tests\optuna_objective_smoke_test_summary_Data_IL.csv

烟雾测试manifest：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\logs\objective_smoke_tests\optuna_objective_smoke_test_manifest_Data_IL.json


In [7]:
# Cell 6 | 断点续跑全部30个正式Optuna Study并保存各外层折最优参数


FORMAL_TUNING_SUMMARY_DIR = (
    SUMMARY_DIR
    / STUDY_SUFFIX
)


FORMAL_TRIAL_TABLE_DIR = (
    FORMAL_TUNING_SUMMARY_DIR
    / "trial_tables"
)


FORMAL_BEST_PARAM_DIR = (
    BEST_PARAM_DIR
    / STUDY_SUFFIX
)


FORMAL_BATCH_STATE_PATH = (
    FORMAL_TUNING_SUMMARY_DIR
    / "formal_tuning_batch_state_Data_IL.json"
)


FORMAL_STUDY_SUMMARY_PATH = (
    FORMAL_TUNING_SUMMARY_DIR
    / "formal_tuning_study_summary_Data_IL.csv"
)


FORMAL_TUNING_MANIFEST_PATH = (
    FORMAL_TUNING_SUMMARY_DIR
    / "formal_tuning_manifest_Data_IL.json"
)


for folder in [
    FORMAL_TUNING_SUMMARY_DIR,
    FORMAL_TRIAL_TABLE_DIR,
    FORMAL_BEST_PARAM_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 1. 正式运行配置
# ============================================================

MODELS_TO_RUN = [
    "ExtraTrees",
    "LightGBM",
    "Transformer",
]


SPLITS_TO_RUN = [
    "Data_split",
    "IL_split",
]


OUTER_FOLDS_TO_RUN = [
    1,
    2,
    3,
    4,
    5,
]


FORMAL_STUDY_COUNT_EXPECTED = (
    len(
        MODELS_TO_RUN
    )
    * len(
        SPLITS_TO_RUN
    )
    * len(
        OUTER_FOLDS_TO_RUN
    )
)


# 先完成树模型，再运行Transformer。
FORMAL_RUN_ORDER = [
    (
        model_name,
        split_name,
        outer_fold,
    )
    for model_name in MODELS_TO_RUN
    for split_name in SPLITS_TO_RUN
    for outer_fold in OUTER_FOLDS_TO_RUN
]


if FORMAL_STUDY_COUNT_EXPECTED != 30:
    raise RuntimeError(
        "正式调参Study数量不是30。"
    )


if len(
    FORMAL_RUN_ORDER
) != 30:
    raise RuntimeError(
        "正式运行顺序数量不是30。"
    )


# ============================================================
# 2. 调参前完整性检查
# ============================================================

CELL6_SMOKE_MANIFEST_PATH = (
    TUNING_LOG_DIR
    / "objective_smoke_tests"
    / "optuna_objective_smoke_test_manifest_Data_IL.json"
)


if not CELL6_SMOKE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "找不到Cell 5烟雾测试manifest：\n"
        f"{CELL6_SMOKE_MANIFEST_PATH}"
    )


cell6_smoke_manifest = read_json(
    CELL6_SMOKE_MANIFEST_PATH
)


if (
    cell6_smoke_manifest.get(
        "status"
    )
    != "all_objective_smoke_tests_passed"
):
    raise RuntimeError(
        "Cell 5烟雾测试没有全部通过。"
    )


smoke_leakage_control = (
    cell6_smoke_manifest.get(
        "leakage_control",
        {},
    )
)


if not smoke_leakage_control.get(
    "all_predictions_fixed_before_"
    "validation_target_loaded"
):
    raise RuntimeError(
        "烟雾测试没有确认预测先于验证标签读取。"
    )


for field_name in [
    "validation_target_opened_before_prediction",
    "validation_target_used_for_training",
    "validation_used_for_scaler_fit",
    "outer_test_used",
    "outer_test_target_read",
    "old_global_mpnn_feature_read",
    "external_10point_holdout_read",
]:
    if smoke_leakage_control.get(
        field_name
    ):
        raise RuntimeError(
            "烟雾测试泄露控制未通过："
            f"{field_name}=True"
        )


formal_tuning_impact = (
    cell6_smoke_manifest.get(
        "formal_tuning_impact",
        {},
    )
)


for field_name in [
    "formal_optuna_study_created",
    "formal_optuna_database_opened",
    "formal_trial_budget_consumed",
    "formal_trial_result_directory_used",
]:
    if formal_tuning_impact.get(
        field_name
    ):
        raise RuntimeError(
            "烟雾测试影响了正式调参："
            f"{field_name}=True"
        )


if int(
    TRANSFORMER_SEARCH_EPOCHS
) != 50:
    raise RuntimeError(
        "Transformer正式调参轮数不是50。"
    )


if not TUNING_SIGNATURE_PATH.exists():
    raise FileNotFoundError(
        "找不到调参签名文件：\n"
        f"{TUNING_SIGNATURE_PATH}"
    )


saved_signature_manifest = read_json(
    TUNING_SIGNATURE_PATH
)


if (
    saved_signature_manifest.get(
        "tuning_signature"
    )
    != TUNING_SIGNATURE
):
    raise RuntimeError(
        "当前调参签名与Cell 4保存结果不一致。"
    )


if (
    saved_signature_manifest.get(
        "input_integrity_digest"
    )
    != INPUT_INTEGRITY_DIGEST
):
    raise RuntimeError(
        "当前输入完整性摘要与Cell 4不一致。"
    )


if not STUDY_PLAN_PATH.exists():
    raise FileNotFoundError(
        "找不到30个Study计划：\n"
        f"{STUDY_PLAN_PATH}"
    )


cell6_study_plan = read_csv_robust(
    STUDY_PLAN_PATH
)


if len(
    cell6_study_plan
) != 30:
    raise RuntimeError(
        "Study计划中的记录数不是30。"
    )


if int(
    cell6_study_plan[
        "Trial_budget"
    ].sum()
) != 450:
    raise RuntimeError(
        "Study计划中的总Trial预算不是450。"
    )


# ============================================================
# 3. 最优参数转换
# ============================================================

def convert_best_search_to_formal_parameters(
    model_name,
    best_search_parameters,
    split_name,
    outer_fold,
    best_trial_number,
):
    """将一个Study的最优搜索参数转换为模型可用参数。"""

    representative_seed = (
        make_trial_seed(
            split_name=split_name,
            outer_fold=outer_fold,
            model_name=model_name,
            trial_number=best_trial_number,
            inner_fold=0,
        )
    )


    if model_name == "ExtraTrees":
        return extratrees_search_to_formal(
            search_parameters=(
                best_search_parameters
            ),
            seed=representative_seed,
        )


    if model_name == "LightGBM":
        return lightgbm_search_to_formal(
            search_parameters=(
                best_search_parameters
            ),
            seed=representative_seed,
        )


    if model_name == "Transformer":
        return transformer_search_to_formal(
            search_parameters=(
                best_search_parameters
            ),
            seed=representative_seed,
        )


    raise KeyError(
        f"未知模型：{model_name}"
    )


# ============================================================
# 4. Study状态统计
# ============================================================

def get_trial_state_counts(
    study,
):
    """统计一个Study中各Trial状态数量。"""

    state_counts = {
        "COMPLETE": 0,
        "FAIL": 0,
        "PRUNED": 0,
        "RUNNING": 0,
        "WAITING": 0,
    }


    for frozen_trial in study.trials:

        state_name = (
            frozen_trial.state.name
        )


        if state_name not in state_counts:
            state_counts[
                state_name
            ] = 0


        state_counts[
            state_name
        ] += 1


    return state_counts


# ============================================================
# 5. 保存批量运行状态
# ============================================================

def save_formal_batch_state(
    status,
    current_model=None,
    current_split=None,
    current_outer_fold=None,
    current_study_name=None,
    completed_studies=None,
    error_message=None,
):
    """保存正式调参批量运行状态。"""

    if completed_studies is None:
        completed_studies = []


    payload = {
        "updated_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "tuning_signature": (
            TUNING_SIGNATURE
        ),

        "study_suffix": (
            STUDY_SUFFIX
        ),

        "status": (
            status
        ),

        "current_model": (
            current_model
        ),

        "current_split": (
            current_split
        ),

        "current_outer_fold": (
            current_outer_fold
        ),

        "current_study_name": (
            current_study_name
        ),

        "completed_study_count": int(
            len(
                completed_studies
            )
        ),

        "completed_studies": (
            completed_studies
        ),

        "error_message": (
            error_message
        ),

        "formal_design": {
            "models": (
                MODELS_TO_RUN
            ),

            "splits": (
                SPLITS_TO_RUN
            ),

            "outer_folds": (
                OUTER_FOLDS_TO_RUN
            ),

            "study_count": int(
                FORMAL_STUDY_COUNT_EXPECTED
            ),

            "trial_budget_total": 450,

            "inner_model_fits_total": 1350,
        },

        "leakage_control": {
            "validation_target_read_policy": (
                "after_prediction_fixed_only"
            ),

            "validation_target_used_for_training": (
                False
            ),

            "validation_used_for_scaler_fit": False,

            "outer_test_used_for_tuning": False,

            "outer_test_target_read": False,

            "old_global_mpnn_feature_read": False,

            "external_10point_holdout_read": False,
        },
    }


    save_json_atomic(
        data=payload,
        output_path=(
            FORMAL_BATCH_STATE_PATH
        ),
    )


# ============================================================
# 6. Trial完成回调
# ============================================================

def make_formal_progress_callback(
    study_index,
    total_studies,
    model_name,
    split_name,
    outer_fold,
    trial_budget,
    study_start_time,
):
    """创建正式调参进度回调。"""

    def callback(
        study,
        frozen_trial,
    ):
        state_counts = (
            get_trial_state_counts(
                study
            )
        )


        completed_count = int(
            state_counts.get(
                "COMPLETE",
                0,
            )
        )


        elapsed_minutes = float(
            (
                time.time()
                - study_start_time
            )
            / 60.0
        )


        trial_value = (
            None
            if frozen_trial.value is None
            else float(
                frozen_trial.value
            )
        )


        print(
            f"[Study {study_index:02d}/"
            f"{total_studies:02d}] "
            f"{model_name} | "
            f"{split_name} | "
            f"OuterFold {outer_fold} | "
            f"Trial {int(frozen_trial.number)} | "
            f"state={frozen_trial.state.name} | "
            f"value={trial_value} | "
            f"completed={completed_count}/"
            f"{trial_budget} | "
            f"elapsed={elapsed_minutes:.2f} min"
        )


        progress_payload = {
            "updated_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),

            "run_code_version": (
                RUN_CODE_VERSION
            ),

            "tuning_signature": (
                TUNING_SIGNATURE
            ),

            "study_index": int(
                study_index
            ),

            "total_studies": int(
                total_studies
            ),

            "study_name": (
                study.study_name
            ),

            "model": (
                model_name
            ),

            "split": (
                split_name
            ),

            "outer_fold": int(
                outer_fold
            ),

            "trial_budget": int(
                trial_budget
            ),

            "trial_state_counts": (
                state_counts
            ),

            "last_trial_number": int(
                frozen_trial.number
            ),

            "last_trial_state": (
                frozen_trial.state.name
            ),

            "last_trial_value": (
                trial_value
            ),

            "elapsed_minutes": (
                elapsed_minutes
            ),

            "outer_test_used": False,
        }


        progress_path = (
            FORMAL_TUNING_SUMMARY_DIR
            / (
                f"progress_"
                f"{model_name}_"
                f"{split_name}_"
                f"OuterFold{outer_fold}.json"
            )
        )


        save_json_atomic(
            data=progress_payload,
            output_path=progress_path,
        )


    return callback


# ============================================================
# 7. 保存一个Study的最终结果
# ============================================================

def save_completed_study_outputs(
    study,
    model_name,
    split_name,
    outer_fold,
    trial_budget,
):
    """保存Study Trial表、最优参数和汇总记录。"""

    state_counts = (
        get_trial_state_counts(
            study
        )
    )


    completed_trials = int(
        state_counts.get(
            "COMPLETE",
            0,
        )
    )


    if completed_trials != int(
        trial_budget
    ):
        raise RuntimeError(
            f"{study.study_name}完成Trial数量"
            f"为{completed_trials}，"
            f"预期为{trial_budget}。"
        )


    best_trial = (
        study.best_trial
    )


    best_search_parameters = dict(
        best_trial.params
    )


    best_formal_parameters = (
        convert_best_search_to_formal_parameters(
            model_name=model_name,

            best_search_parameters=(
                best_search_parameters
            ),

            split_name=split_name,

            outer_fold=outer_fold,

            best_trial_number=int(
                best_trial.number
            ),
        )
    )


    study_trial_table_path = (
        FORMAL_TRIAL_TABLE_DIR
        / (
            f"{model_name}_"
            f"{split_name}_"
            f"OuterFold{outer_fold}_"
            "trials.csv"
        )
    )


    trial_frame = save_study_trials(
        study=study,
        output_path=(
            study_trial_table_path
        ),
    )


    if len(
        trial_frame
    ) < completed_trials:
        raise RuntimeError(
            f"{study.study_name}的Trial表行数"
            "小于完成Trial数量。"
        )


    best_parameter_path = (
        FORMAL_BEST_PARAM_DIR
        / (
            f"best_params_"
            f"{model_name}_"
            f"{split_name}_"
            f"OuterFold{outer_fold}.json"
        )
    )


    best_parameter_payload = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_code_version": (
            RUN_CODE_VERSION
        ),

        "tuning_signature": (
            TUNING_SIGNATURE
        ),

        "study_suffix": (
            STUDY_SUFFIX
        ),

        "status": (
            "study_completed"
        ),

        "study_name": (
            study.study_name
        ),

        "model": (
            model_name
        ),

        "split": (
            split_name
        ),

        "outer_fold": int(
            outer_fold
        ),

        "trial_budget": int(
            trial_budget
        ),

        "trial_state_counts": (
            state_counts
        ),

        "best_trial_number": int(
            best_trial.number
        ),

        "best_objective_value": float(
            best_trial.value
        ),

        "objective_metric": (
            OPTUNA_OBJECTIVE_METRIC
        ),

        "objective_direction": (
            OPTUNA_DIRECTION
        ),

        "inner_fold_aggregation": (
            INNER_FOLD_AGGREGATION
        ),

        "best_search_parameters": (
            best_search_parameters
        ),

        "best_formal_parameters": (
            best_formal_parameters
        ),

        "best_trial_user_attributes": dict(
            best_trial.user_attrs
        ),

        "validation_target_policy": {
            "opened_before_prediction": False,

            "opened_after_prediction_fixed": True,

            "used_for_metric_calculation": True,

            "used_for_training": False,

            "used_for_scaler_fit": False,
        },

        "leakage_control": {
            "outer_test_used": False,

            "outer_test_target_read": False,

            "old_global_mpnn_feature_read": False,

            "external_10point_holdout_read": False,
        },

        "outputs": {
            "trial_table": str(
                study_trial_table_path
            ),

            "best_parameter_file": str(
                best_parameter_path
            ),
        },
    }


    save_json_atomic(
        data=best_parameter_payload,
        output_path=(
            best_parameter_path
        ),
    )


    return {
        "Study_name": (
            study.study_name
        ),

        "Model": (
            model_name
        ),

        "Split": (
            split_name
        ),

        "Outer_fold": int(
            outer_fold
        ),

        "Trial_budget": int(
            trial_budget
        ),

        "Complete_trials": (
            completed_trials
        ),

        "Failed_trials": int(
            state_counts.get(
                "FAIL",
                0,
            )
        ),

        "Pruned_trials": int(
            state_counts.get(
                "PRUNED",
                0,
            )
        ),

        "Running_trials": int(
            state_counts.get(
                "RUNNING",
                0,
            )
        ),

        "Best_trial_number": int(
            best_trial.number
        ),

        "Best_Mean_R2": float(
            best_trial.value
        ),

        "Best_Std_R2": float(
            best_trial.user_attrs.get(
                "Std_R2",
                np.nan,
            )
        ),

        "Best_Mean_RMSE": float(
            best_trial.user_attrs.get(
                "Mean_RMSE",
                np.nan,
            )
        ),

        "Best_Mean_MAE": float(
            best_trial.user_attrs.get(
                "Mean_MAE",
                np.nan,
            )
        ),

        "Best_Mean_Pearson_r": float(
            best_trial.user_attrs.get(
                "Mean_Pearson_r",
                np.nan,
            )
        ),

        "Best_parameters_json": (
            json.dumps(
                to_jsonable(
                    best_search_parameters
                ),
                ensure_ascii=False,
                sort_keys=True,
            )
        ),

        "Trial_table_path": str(
            study_trial_table_path
        ),

        "Best_parameter_path": str(
            best_parameter_path
        ),

        "Status": "completed",
    }


# ============================================================
# 8. 正式运行全部30个Study
# ============================================================

formal_batch_started_at = (
    datetime.now().isoformat(
        timespec="seconds"
    )
)


formal_batch_start_time = (
    time.time()
)


completed_study_names = []

formal_summary_rows = []


save_formal_batch_state(
    status="formal_tuning_started",

    completed_studies=(
        completed_study_names
    ),
)


try:

    for study_index, (
        model_name,
        split_name,
        outer_fold,
    ) in enumerate(
        FORMAL_RUN_ORDER,
        start=1,
    ):

        trial_budget = int(
            N_TRIALS_PER_OUTER_FOLD[
                model_name
            ]
        )


        study_name = get_study_name(
            split_name=split_name,
            outer_fold=outer_fold,
            model_name=model_name,
        )


        print(
            "\n"
            + "=" * 112
        )


        print(
            f"正式Study {study_index}/"
            f"{FORMAL_STUDY_COUNT_EXPECTED}"
        )


        print(
            "model       =",
            model_name,
        )


        print(
            "split       =",
            split_name,
        )


        print(
            "outer fold  =",
            outer_fold,
        )


        print(
            "study name  =",
            study_name,
        )


        print(
            "trial budget=",
            trial_budget,
        )


        print(
            "outer test used =",
            False,
        )


        print(
            "=" * 112
        )


        save_formal_batch_state(
            status="study_running",

            current_model=(
                model_name
            ),

            current_split=(
                split_name
            ),

            current_outer_fold=int(
                outer_fold
            ),

            current_study_name=(
                study_name
            ),

            completed_studies=(
                completed_study_names
            ),
        )


        study = (
            create_or_load_outer_fold_study(
                split_name=split_name,

                outer_fold=outer_fold,

                model_name=model_name,
            )
        )


        if (
            study.study_name
            != study_name
        ):
            raise RuntimeError(
                "实际Study名称与计划不一致。"
            )


        state_counts_before = (
            get_trial_state_counts(
                study
            )
        )


        completed_before = int(
            state_counts_before.get(
                "COMPLETE",
                0,
            )
        )


        if completed_before > trial_budget:
            raise RuntimeError(
                f"{study_name}已经完成"
                f"{completed_before}个Trial，"
                f"超过预算{trial_budget}。"
            )


        print(
            "completed before run =",
            completed_before,
        )


        print(
            "remaining trials     =",
            int(
                trial_budget
                - completed_before
            ),
        )


        study_start_time = (
            time.time()
        )


        remaining_trials = int(
            trial_budget
            - completed_before
        )


        if remaining_trials > 0:

            objective = (
                make_outer_fold_objective(
                    split_name=split_name,

                    outer_fold=outer_fold,

                    model_name=model_name,
                )
            )


            progress_callback = (
                make_formal_progress_callback(
                    study_index=study_index,

                    total_studies=(
                        FORMAL_STUDY_COUNT_EXPECTED
                    ),

                    model_name=model_name,

                    split_name=split_name,

                    outer_fold=outer_fold,

                    trial_budget=trial_budget,

                    study_start_time=(
                        study_start_time
                    ),
                )
            )


            study.optimize(
                objective,

                n_trials=(
                    remaining_trials
                ),

                callbacks=[
                    progress_callback
                ],

                gc_after_trial=True,

                show_progress_bar=False,

                catch=(),
            )


        else:

            print(
                "该Study已完成，跳过重复调参。"
            )


        state_counts_after = (
            get_trial_state_counts(
                study
            )
        )


        completed_after = int(
            state_counts_after.get(
                "COMPLETE",
                0,
            )
        )


        if completed_after != trial_budget:
            raise RuntimeError(
                f"{study_name}运行结束后"
                f"完成Trial={completed_after}，"
                f"预期={trial_budget}。"
            )


        summary_row = (
            save_completed_study_outputs(
                study=study,

                model_name=model_name,

                split_name=split_name,

                outer_fold=outer_fold,

                trial_budget=trial_budget,
            )
        )


        formal_summary_rows.append(
            summary_row
        )


        completed_study_names.append(
            study_name
        )


        save_formal_batch_state(
            status="study_completed",

            current_model=(
                model_name
            ),

            current_split=(
                split_name
            ),

            current_outer_fold=int(
                outer_fold
            ),

            current_study_name=(
                study_name
            ),

            completed_studies=(
                completed_study_names
            ),
        )


        study_elapsed_minutes = float(
            (
                time.time()
                - study_start_time
            )
            / 60.0
        )


        print(
            "Study完成 ✓"
        )


        print(
            "completed trials =",
            completed_after,
        )


        print(
            "best trial       =",
            summary_row[
                "Best_trial_number"
            ],
        )


        print(
            "best mean R2     =",
            summary_row[
                "Best_Mean_R2"
            ],
        )


        print(
            "elapsed minutes  =",
            study_elapsed_minutes,
        )


        del study


        gc.collect()


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


except KeyboardInterrupt:

    save_formal_batch_state(
        status="formal_tuning_interrupted",

        current_model=(
            model_name
            if "model_name" in locals()
            else None
        ),

        current_split=(
            split_name
            if "split_name" in locals()
            else None
        ),

        current_outer_fold=(
            int(
                outer_fold
            )
            if "outer_fold" in locals()
            else None
        ),

        current_study_name=(
            study_name
            if "study_name" in locals()
            else None
        ),

        completed_studies=(
            completed_study_names
        ),

        error_message=(
            "KeyboardInterrupt"
        ),
    )


    print(
        "\n正式调参已由用户中断。"
    )


    print(
        "已完成的Trial保存在Optuna数据库中。"
    )


    print(
        "重新运行Cell 6会从未完成位置继续。"
    )


    raise


except Exception as error:

    error_text = (
        f"{type(error).__name__}: "
        f"{str(error)}"
    )


    save_formal_batch_state(
        status="formal_tuning_failed",

        current_model=(
            model_name
            if "model_name" in locals()
            else None
        ),

        current_split=(
            split_name
            if "split_name" in locals()
            else None
        ),

        current_outer_fold=(
            int(
                outer_fold
            )
            if "outer_fold" in locals()
            else None
        ),

        current_study_name=(
            study_name
            if "study_name" in locals()
            else None
        ),

        completed_studies=(
            completed_study_names
        ),

        error_message=(
            error_text
        ),
    )


    print(
        "\n正式调参发生错误："
    )


    print(
        error_text
    )


    print(
        traceback.format_exc()
    )


    raise


# ============================================================
# 9. 正式调参结果汇总
# ============================================================

df_formal_study_summary = (
    pd.DataFrame(
        formal_summary_rows
    )
    .sort_values(
        [
            "Model",
            "Split",
            "Outer_fold",
        ],

        key=lambda series: (
            series.map(
                {
                    "ExtraTrees": 0,
                    "LightGBM": 1,
                    "Transformer": 2,
                    "Data_split": 0,
                    "IL_split": 1,
                }
            )
            if series.name
            in [
                "Model",
                "Split",
            ]
            else series
        ),
    )
    .reset_index(
        drop=True
    )
)


if len(
    df_formal_study_summary
) != FORMAL_STUDY_COUNT_EXPECTED:
    raise RuntimeError(
        "正式调参汇总Study数量不是30。"
    )


if not df_formal_study_summary[
    "Status"
].eq(
    "completed"
).all():
    raise RuntimeError(
        "部分正式Study没有完成。"
    )


if not (
    df_formal_study_summary[
        "Complete_trials"
    ]
    == df_formal_study_summary[
        "Trial_budget"
    ]
).all():
    raise RuntimeError(
        "部分Study完成Trial数量"
        "与预算不一致。"
    )


total_completed_trials = int(
    df_formal_study_summary[
        "Complete_trials"
    ].sum()
)


if total_completed_trials != 450:
    raise RuntimeError(
        "正式调参完成Trial总数不是450。"
    )


total_inner_model_fits = int(
    total_completed_trials
    * N_INNER_FOLDS
)


if total_inner_model_fits != 1350:
    raise RuntimeError(
        "正式调参内层模型拟合总数不是1350。"
    )


if not np.isfinite(
    df_formal_study_summary[
        [
            "Best_Mean_R2",
            "Best_Std_R2",
            "Best_Mean_RMSE",
            "Best_Mean_MAE",
            "Best_Mean_Pearson_r",
        ]
    ].to_numpy(
        dtype=np.float64
    )
).all():
    raise RuntimeError(
        "最优Trial汇总指标中存在NaN或无穷值。"
    )


df_formal_study_summary.to_csv(
    FORMAL_STUDY_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 10. 保存正式调参manifest
# ============================================================

formal_batch_finished_at = (
    datetime.now().isoformat(
        timespec="seconds"
    )
)


formal_batch_elapsed_minutes = float(
    (
        time.time()
        - formal_batch_start_time
    )
    / 60.0
)


formal_tuning_manifest = {
    "created_at": (
        formal_batch_finished_at
    ),

    "formal_batch_started_at": (
        formal_batch_started_at
    ),

    "formal_batch_finished_at": (
        formal_batch_finished_at
    ),

    "formal_batch_elapsed_minutes": (
        formal_batch_elapsed_minutes
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "tuning_signature": (
        TUNING_SIGNATURE
    ),

    "study_suffix": (
        STUDY_SUFFIX
    ),

    "input_integrity_digest": (
        INPUT_INTEGRITY_DIGEST
    ),

    "status": (
        "all_formal_optuna_studies_completed"
    ),

    "design": {
        "splits": (
            SPLITS_TO_RUN
        ),

        "models": (
            MODELS_TO_RUN
        ),

        "outer_folds": (
            OUTER_FOLDS_TO_RUN
        ),

        "study_count": int(
            len(
                df_formal_study_summary
            )
        ),

        "completed_trials": (
            total_completed_trials
        ),

        "inner_folds_per_trial": int(
            N_INNER_FOLDS
        ),

        "total_inner_model_fits": (
            total_inner_model_fits
        ),

        "trial_budgets_per_outer_fold": (
            N_TRIALS_PER_OUTER_FOLD
        ),

        "transformer_search_epochs": int(
            TRANSFORMER_SEARCH_EPOCHS
        ),
    },

    "objective": {
        "metric": (
            OPTUNA_OBJECTIVE_METRIC
        ),

        "direction": (
            OPTUNA_DIRECTION
        ),

        "aggregation": (
            INNER_FOLD_AGGREGATION
        ),

        "pruner": "NopPruner",
    },

    "validation_target_policy": {
        "read_before_prediction_fixed": False,

        "read_after_prediction_fixed": True,

        "used_for_metric_calculation": True,

        "used_for_model_training": False,

        "used_for_scaler_fit": False,

        "used_for_backpropagation": False,

        "used_for_early_stopping": False,

        "used_for_scheduler": False,
    },

    "leakage_control": {
        "inner_fold_specific_mpnn_features": (
            True
        ),

        "outer_test_used_for_tuning": False,

        "outer_test_target_read": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "study_records": (
        df_formal_study_summary
        .to_dict(
            orient="records"
        )
    ),

    "outputs": {
        "optuna_database": str(
            OPTUNA_DB_PATH
        ),

        "study_summary": str(
            FORMAL_STUDY_SUMMARY_PATH
        ),

        "best_parameter_directory": str(
            FORMAL_BEST_PARAM_DIR
        ),

        "trial_table_directory": str(
            FORMAL_TRIAL_TABLE_DIR
        ),

        "batch_state": str(
            FORMAL_BATCH_STATE_PATH
        ),

        "manifest": str(
            FORMAL_TUNING_MANIFEST_PATH
        ),
    },
}


save_json_atomic(
    data=formal_tuning_manifest,
    output_path=(
        FORMAL_TUNING_MANIFEST_PATH
    ),
)


save_formal_batch_state(
    status="all_formal_studies_completed",

    completed_studies=(
        completed_study_names
    ),
)


# ============================================================
# 11. 输出检查
# ============================================================

print(
    "\n"
    + "=" * 112
)


print(
    "Cell 6 全部30个正式Optuna Study运行完成 ✓"
)


print(
    "=" * 112
)


print(
    "study count                      =",
    len(
        df_formal_study_summary
    ),
)


print(
    "completed formal trials          =",
    total_completed_trials,
)


print(
    "inner folds/trial                =",
    N_INNER_FOLDS,
)


print(
    "total inner model fits           =",
    total_inner_model_fits,
)


print(
    "all study statuses completed     =",
    bool(
        df_formal_study_summary[
            "Status"
        ].eq(
            "completed"
        ).all()
    ),
)


print(
    "trial budget checks passed       =",
    bool(
        (
            df_formal_study_summary[
                "Complete_trials"
            ]
            == df_formal_study_summary[
                "Trial_budget"
            ]
        ).all()
    ),
)


print(
    "Transformer search epochs        =",
    TRANSFORMER_SEARCH_EPOCHS,
)


print(
    "objective metric                 =",
    OPTUNA_OBJECTIVE_METRIC,
)


print(
    "validation target read policy    =",
    "after_prediction_fixed_only",
)


print(
    "validation target used training  =",
    False,
)


print(
    "validation used for scaler fit   =",
    False,
)


print(
    "outer test used for tuning       =",
    False,
)


print(
    "outer test target read           =",
    False,
)


print(
    "old global MPNN feature read     =",
    False,
)


print(
    "external 10 points read          =",
    False,
)


print(
    "formal elapsed minutes           =",
    formal_batch_elapsed_minutes,
)


print(
    "\n各Study最优内层验证结果："
)


display(
    df_formal_study_summary[
        [
            "Model",
            "Split",
            "Outer_fold",
            "Complete_trials",
            "Best_trial_number",
            "Best_Mean_R2",
            "Best_Std_R2",
            "Best_Mean_RMSE",
            "Best_Mean_MAE",
            "Best_Mean_Pearson_r",
            "Status",
        ]
    ]
)


print(
    "\n正式调参Study汇总："
)


print(
    FORMAL_STUDY_SUMMARY_PATH
)


print(
    "\n各外层折最优参数目录："
)


print(
    FORMAL_BEST_PARAM_DIR
)


print(
    "\n各Study Trial明细目录："
)


print(
    FORMAL_TRIAL_TABLE_DIR
)


print(
    "\n正式调参manifest："
)


print(
    FORMAL_TUNING_MANIFEST_PATH
)


print(
    "\n正式调参批量状态："
)


print(
    FORMAL_BATCH_STATE_PATH
)


正式Study 1/30
model       = ExtraTrees
split       = Data_split
outer fold  = 1
study name  = viscosity_extratrees_data_split_outerfold1_4eafa9d6fbe7
trial budget= 15
outer test used = False
completed before run = 0
remaining trials     = 15
[Study 01/30] ExtraTrees | Data_split | OuterFold 1 | Trial 0 | state=COMPLETE | value=0.656732729389118 | completed=1/15 | elapsed=0.06 min
[Study 01/30] ExtraTrees | Data_split | OuterFold 1 | Trial 1 | state=COMPLETE | value=0.9234786260748473 | completed=2/15 | elapsed=0.15 min
[Study 01/30] ExtraTrees | Data_split | OuterFold 1 | Trial 2 | state=COMPLETE | value=0.6418740616549862 | completed=3/15 | elapsed=0.19 min
[Study 01/30] ExtraTrees | Data_split | OuterFold 1 | Trial 3 | state=COMPLETE | value=0.9093266003156755 | completed=4/15 | elapsed=0.36 min
[Study 01/30] ExtraTrees | Data_split | OuterFold 1 | Trial 4 | state=COMPLETE | value=0.9406081644633258 | completed=5/15 | elapsed=0.61 min
[Study 01/30] ExtraTrees | Data_split | OuterFold

,Model,Split,Outer_fold,Complete_trials,Best_trial_number,Best_Mean_R2,Best_Std_R2,Best_Mean_RMSE,Best_Mean_MAE,Best_Mean_Pearson_r,Status
0,ExtraTrees,Data_split,1,15,14,0.968072,0.003381,0.147465,0.080169,0.985166,completed
1,ExtraTrees,Data_split,2,15,14,0.974647,0.002301,0.131531,0.069360,0.987872,completed
2,ExtraTrees,Data_split,3,15,13,0.956614,0.002113,0.172569,0.111511,0.979853,completed
3,ExtraTrees,Data_split,4,15,7,0.976591,0.000786,0.130897,0.065432,0.988855,completed
4,ExtraTrees,Data_split,5,15,12,0.969655,0.004733,0.145336,0.078048,0.985394,completed
5,ExtraTrees,IL_split,1,15,0,0.671363,0.172055,0.472905,0.263945,0.816908,completed
6,ExtraTrees,IL_split,2,15,12,0.779982,0.018535,0.402136,0.244819,0.890275,completed
7,ExtraTrees,IL_split,3,15,4,0.692515,0.084673,0.422289,0.257158,0.833215,completed
8,ExtraTrees,IL_split,4,15,3,0.647562,0.195490,0.505879,0.258489,0.801012,completed
9,ExtraTrees,IL_split,5,15,12,0.791475,0.025516,0.386996,0.236986,0.898445,completed



正式调参Study汇总：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\summaries\4eafa9d6fbe7\formal_tuning_study_summary_Data_IL.csv

各外层折最优参数目录：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\best_params_by_outer_fold\4eafa9d6fbe7

各Study Trial明细目录：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\summaries\4eafa9d6fbe7\trial_tables

正式调参manifest：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\summaries\4eafa9d6fbe7\formal_tuning_manifest_Data_IL.json

正式调参批量状态：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mod

In [8]:
# Cell 7 | 审计30套外层折专属最优参数并生成正式外层评价交接清单


# ============================================================
# 1. 输出路径
# ============================================================

OUTER_EVALUATION_HANDOFF_DIR = (
    FORMAL_TUNING_SUMMARY_DIR
    / "outer_evaluation_handoff"
)


OUTER_EVALUATION_HANDOFF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


OUTER_EVALUATION_PARAMETER_TABLE_PATH = (
    OUTER_EVALUATION_HANDOFF_DIR
    / "outer_evaluation_parameter_table_Data_IL.csv"
)


OUTER_EVALUATION_PARAMETER_MANIFEST_PATH = (
    OUTER_EVALUATION_HANDOFF_DIR
    / "outer_evaluation_parameter_manifest_Data_IL.json"
)


# ============================================================
# 2. 文件SHA256
# ============================================================

def calculate_file_sha256_strict(
    file_path,
):
    """分块计算文件SHA256。"""

    file_path = Path(
        file_path
    )


    if not file_path.exists():
        raise FileNotFoundError(
            f"找不到文件：\n{file_path}"
        )


    sha256 = hashlib.sha256()


    with file_path.open(
        "rb"
    ) as file_handle:

        while True:

            block = file_handle.read(
                1024 * 1024
            )


            if not block:
                break


            sha256.update(
                block
            )


    return sha256.hexdigest()


# ============================================================
# 3. 检查文件位于指定目录
# ============================================================

def assert_path_inside_directory(
    file_path,
    expected_directory,
    role_name,
):
    """确认文件位于预期目录下。"""

    file_path = Path(
        file_path
    ).resolve()


    expected_directory = Path(
        expected_directory
    ).resolve()


    try:

        file_path.relative_to(
            expected_directory
        )

    except ValueError as error:

        raise RuntimeError(
            f"{role_name}不在预期目录中：\n"
            f"文件：{file_path}\n"
            f"预期目录：{expected_directory}"
        ) from error


    return file_path


# ============================================================
# 4. 读取并检查Cell 6正式调参结果
# ============================================================

if not FORMAL_TUNING_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "找不到正式调参manifest：\n"
        f"{FORMAL_TUNING_MANIFEST_PATH}"
    )


if not FORMAL_STUDY_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "找不到正式调参Study汇总：\n"
        f"{FORMAL_STUDY_SUMMARY_PATH}"
    )


formal_manifest_for_handoff = read_json(
    FORMAL_TUNING_MANIFEST_PATH
)


if (
    formal_manifest_for_handoff.get(
        "status"
    )
    != "all_formal_optuna_studies_completed"
):
    raise RuntimeError(
        "正式调参manifest未显示全部Study完成。"
    )


if (
    formal_manifest_for_handoff.get(
        "tuning_signature"
    )
    != TUNING_SIGNATURE
):
    raise RuntimeError(
        "正式调参manifest中的调参签名"
        "与当前Notebook不一致。"
    )


if (
    formal_manifest_for_handoff.get(
        "input_integrity_digest"
    )
    != INPUT_INTEGRITY_DIGEST
):
    raise RuntimeError(
        "正式调参manifest中的输入完整性摘要"
        "与当前Notebook不一致。"
    )


formal_design = (
    formal_manifest_for_handoff.get(
        "design",
        {},
    )
)


if int(
    formal_design.get(
        "study_count",
        -1,
    )
) != 30:
    raise RuntimeError(
        "正式调参Study数量不是30。"
    )


if int(
    formal_design.get(
        "completed_trials",
        -1,
    )
) != 450:
    raise RuntimeError(
        "正式调参完成Trial数量不是450。"
    )


if int(
    formal_design.get(
        "total_inner_model_fits",
        -1,
    )
) != 1350:
    raise RuntimeError(
        "正式调参内层模型拟合次数不是1350。"
    )


manifest_validation_policy = (
    formal_manifest_for_handoff.get(
        "validation_target_policy",
        {},
    )
)


expected_false_validation_fields = [
    "read_before_prediction_fixed",
    "used_for_model_training",
    "used_for_scaler_fit",
    "used_for_backpropagation",
    "used_for_early_stopping",
    "used_for_scheduler",
]


for field_name in expected_false_validation_fields:

    if manifest_validation_policy.get(
        field_name
    ):

        raise RuntimeError(
            "正式调参验证标签策略未通过："
            f"{field_name}=True"
        )


if not manifest_validation_policy.get(
    "read_after_prediction_fixed"
):
    raise RuntimeError(
        "正式调参未确认验证预测固定后"
        "才读取验证标签。"
    )


manifest_leakage_control = (
    formal_manifest_for_handoff.get(
        "leakage_control",
        {},
    )
)


if not manifest_leakage_control.get(
    "inner_fold_specific_mpnn_features"
):
    raise RuntimeError(
        "正式调参未使用内层折专属MPNN表征。"
    )


for field_name in [
    "outer_test_used_for_tuning",
    "outer_test_target_read",
    "old_global_mpnn_feature_read",
    "external_10point_holdout_read",
]:

    if manifest_leakage_control.get(
        field_name
    ):

        raise RuntimeError(
            "正式调参泄露控制未通过："
            f"{field_name}=True"
        )


df_formal_summary_handoff = pd.read_csv(
    FORMAL_STUDY_SUMMARY_PATH,
    encoding="utf-8-sig",
)


if len(
    df_formal_summary_handoff
) != 30:
    raise RuntimeError(
        "正式调参Study汇总行数不是30。"
    )


# ============================================================
# 5. 检查30个模型×划分×外层折组合
# ============================================================

expected_combinations = {
    (
        model_name,
        split_name,
        int(
            outer_fold
        ),
    )
    for model_name in MODEL_ORDER
    for split_name in SPLIT_ORDER
    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    )
}


actual_combinations = {
    (
        str(
            row[
                "Model"
            ]
        ),

        str(
            row[
                "Split"
            ]
        ),

        int(
            row[
                "Outer_fold"
            ]
        ),
    )
    for _, row
    in df_formal_summary_handoff.iterrows()
}


if actual_combinations != expected_combinations:

    missing_combinations = (
        expected_combinations
        - actual_combinations
    )


    extra_combinations = (
        actual_combinations
        - expected_combinations
    )


    raise RuntimeError(
        "正式调参组合不完整。\n"
        f"缺失：{sorted(missing_combinations)}\n"
        f"多余：{sorted(extra_combinations)}"
    )


# ============================================================
# 6. 逐一审计30套最优参数
# ============================================================

handoff_records = []


parameter_file_hashes = []


trial_table_hashes = []


for _, summary_row in (
    df_formal_summary_handoff.iterrows()
):

    model_name = str(
        summary_row[
            "Model"
        ]
    )


    split_name = str(
        summary_row[
            "Split"
        ]
    )


    outer_fold = int(
        summary_row[
            "Outer_fold"
        ]
    )


    trial_budget = int(
        summary_row[
            "Trial_budget"
        ]
    )


    complete_trials = int(
        summary_row[
            "Complete_trials"
        ]
    )


    failed_trials = int(
        summary_row[
            "Failed_trials"
        ]
    )


    pruned_trials = int(
        summary_row[
            "Pruned_trials"
        ]
    )


    running_trials = int(
        summary_row[
            "Running_trials"
        ]
    )


    best_trial_number = int(
        summary_row[
            "Best_trial_number"
        ]
    )


    best_mean_r2 = float(
        summary_row[
            "Best_Mean_R2"
        ]
    )


    if str(
        summary_row[
            "Status"
        ]
    ) != "completed":

        raise RuntimeError(
            f"{model_name}/{split_name}/"
            f"OuterFold{outer_fold}未完成。"
        )


    if complete_trials != trial_budget:

        raise RuntimeError(
            f"{model_name}/{split_name}/"
            f"OuterFold{outer_fold}完成Trial数量"
            "与预算不一致。"
        )


    if any(
        value != 0
        for value in [
            failed_trials,
            pruned_trials,
            running_trials,
        ]
    ):

        raise RuntimeError(
            f"{model_name}/{split_name}/"
            f"OuterFold{outer_fold}存在失败、"
            "剪枝或运行中的Trial。"
        )


    expected_trial_budget = int(
        N_TRIALS_PER_OUTER_FOLD[
            model_name
        ]
    )


    if trial_budget != expected_trial_budget:

        raise RuntimeError(
            f"{model_name}/{split_name}/"
            f"OuterFold{outer_fold}的Trial预算错误。"
        )


    best_parameter_path = (
        assert_path_inside_directory(
            file_path=(
                summary_row[
                    "Best_parameter_path"
                ]
            ),

            expected_directory=(
                FORMAL_BEST_PARAM_DIR
            ),

            role_name="最优参数文件",
        )
    )


    trial_table_path = (
        assert_path_inside_directory(
            file_path=(
                summary_row[
                    "Trial_table_path"
                ]
            ),

            expected_directory=(
                FORMAL_TRIAL_TABLE_DIR
            ),

            role_name="Trial明细文件",
        )
    )


    if not best_parameter_path.exists():

        raise FileNotFoundError(
            "找不到最优参数文件：\n"
            f"{best_parameter_path}"
        )


    if not trial_table_path.exists():

        raise FileNotFoundError(
            "找不到Trial明细文件：\n"
            f"{trial_table_path}"
        )


    best_parameter_payload = read_json(
        best_parameter_path
    )


    expected_parameter_fields = [
        "status",
        "model",
        "split",
        "outer_fold",
        "trial_budget",
        "best_trial_number",
        "best_objective_value",
        "best_search_parameters",
        "best_formal_parameters",
        "tuning_signature",
        "leakage_control",
    ]


    missing_parameter_fields = [
        field_name
        for field_name
        in expected_parameter_fields
        if field_name
        not in best_parameter_payload
    ]


    if missing_parameter_fields:

        raise RuntimeError(
            "最优参数文件缺少字段："
            f"{missing_parameter_fields}\n"
            f"{best_parameter_path}"
        )


    if (
        best_parameter_payload[
            "status"
        ]
        != "study_completed"
    ):

        raise RuntimeError(
            f"{best_parameter_path.name}状态错误。"
        )


    if (
        best_parameter_payload[
            "model"
        ]
        != model_name
    ):

        raise RuntimeError(
            f"{best_parameter_path.name}模型名称错误。"
        )


    if (
        best_parameter_payload[
            "split"
        ]
        != split_name
    ):

        raise RuntimeError(
            f"{best_parameter_path.name}划分名称错误。"
        )


    if int(
        best_parameter_payload[
            "outer_fold"
        ]
    ) != outer_fold:

        raise RuntimeError(
            f"{best_parameter_path.name}外层折编号错误。"
        )


    if int(
        best_parameter_payload[
            "trial_budget"
        ]
    ) != trial_budget:

        raise RuntimeError(
            f"{best_parameter_path.name}Trial预算错误。"
        )


    if int(
        best_parameter_payload[
            "best_trial_number"
        ]
    ) != best_trial_number:

        raise RuntimeError(
            f"{best_parameter_path.name}最优Trial编号错误。"
        )


    if not np.isclose(
        float(
            best_parameter_payload[
                "best_objective_value"
            ]
        ),

        best_mean_r2,

        rtol=1e-12,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{best_parameter_path.name}最优目标值"
            "与Study汇总不一致。"
        )


    if (
        best_parameter_payload[
            "tuning_signature"
        ]
        != TUNING_SIGNATURE
    ):

        raise RuntimeError(
            f"{best_parameter_path.name}调参签名错误。"
        )


    best_search_parameters = (
        best_parameter_payload[
            "best_search_parameters"
        ]
    )


    best_formal_parameters = (
        best_parameter_payload[
            "best_formal_parameters"
        ]
    )


    if not isinstance(
        best_search_parameters,
        dict,
    ) or not best_search_parameters:

        raise RuntimeError(
            f"{best_parameter_path.name}"
            "搜索参数为空。"
        )


    if not isinstance(
        best_formal_parameters,
        dict,
    ) or not best_formal_parameters:

        raise RuntimeError(
            f"{best_parameter_path.name}"
            "正式训练参数为空。"
        )


    parameter_leakage_control = (
        best_parameter_payload.get(
            "leakage_control",
            {},
        )
    )


    for field_name in [
        "outer_test_used",
        "outer_test_target_read",
        "old_global_mpnn_feature_read",
        "external_10point_holdout_read",
    ]:

        if parameter_leakage_control.get(
            field_name
        ):

            raise RuntimeError(
                f"{best_parameter_path.name}"
                f"泄露控制字段{field_name}=True。"
            )


    trial_frame = pd.read_csv(
        trial_table_path,
        encoding="utf-8-sig",
    )


    state_columns = [
        column_name
        for column_name
        in trial_frame.columns
        if str(
            column_name
        ).lower() == "state"
    ]


    if len(
        state_columns
    ) != 1:

        raise RuntimeError(
            f"{trial_table_path.name}"
            "未找到唯一的state列。"
        )


    state_column = (
        state_columns[
            0
        ]
    )


    complete_count_from_table = int(
        trial_frame[
            state_column
        ]
        .astype(str)
        .str.upper()
        .eq(
            "COMPLETE"
        )
        .sum()
    )


    if complete_count_from_table != trial_budget:

        raise RuntimeError(
            f"{trial_table_path.name}中的COMPLETE数量"
            "与Trial预算不一致。"
        )


    if len(
        trial_frame
    ) != trial_budget:

        raise RuntimeError(
            f"{trial_table_path.name}总行数"
            "与Trial预算不一致。"
        )


    parameter_sha256 = (
        calculate_file_sha256_strict(
            best_parameter_path
        )
    )


    trial_table_sha256 = (
        calculate_file_sha256_strict(
            trial_table_path
        )
    )


    parameter_file_hashes.append(
        parameter_sha256
    )


    trial_table_hashes.append(
        trial_table_sha256
    )


    handoff_records.append(
        {
            "Model": model_name,

            "Split": split_name,

            "Outer_fold": int(
                outer_fold
            ),

            "Trial_budget": int(
                trial_budget
            ),

            "Best_trial_number": int(
                best_trial_number
            ),

            "Best_inner_validation_Mean_R2": (
                best_mean_r2
            ),

            "Best_inner_validation_Std_R2": float(
                summary_row[
                    "Best_Std_R2"
                ]
            ),

            "Best_inner_validation_Mean_RMSE": float(
                summary_row[
                    "Best_Mean_RMSE"
                ]
            ),

            "Best_inner_validation_Mean_MAE": float(
                summary_row[
                    "Best_Mean_MAE"
                ]
            ),

            "Best_inner_validation_Mean_Pearson_r": float(
                summary_row[
                    "Best_Mean_Pearson_r"
                ]
            ),

            "Best_search_parameters_json": json.dumps(
                to_jsonable(
                    best_search_parameters
                ),
                ensure_ascii=False,
                sort_keys=True,
            ),

            "Best_formal_parameters_json": json.dumps(
                to_jsonable(
                    best_formal_parameters
                ),
                ensure_ascii=False,
                sort_keys=True,
            ),

            "Best_parameter_path": str(
                best_parameter_path
            ),

            "Best_parameter_sha256": (
                parameter_sha256
            ),

            "Trial_table_path": str(
                trial_table_path
            ),

            "Trial_table_sha256": (
                trial_table_sha256
            ),

            "Tuning_signature": (
                TUNING_SIGNATURE
            ),

            "Input_integrity_digest": (
                INPUT_INTEGRITY_DIGEST
            ),

            "Parameter_usage": (
                "use_only_for_matching_"
                "split_and_outer_fold"
            ),

            "Outer_test_used_during_tuning": (
                False
            ),

            "Status": "handoff_ready",
        }
    )


# ============================================================
# 7. 生成参数交接表
# ============================================================

model_order_map = {
    model_name: index
    for index, model_name
    in enumerate(
        MODEL_ORDER
    )
}


split_order_map = {
    split_name: index
    for index, split_name
    in enumerate(
        SPLIT_ORDER
    )
}


df_outer_evaluation_handoff = (
    pd.DataFrame(
        handoff_records
    )
    .assign(
        _Model_order=lambda frame: (
            frame[
                "Model"
            ].map(
                model_order_map
            )
        ),

        _Split_order=lambda frame: (
            frame[
                "Split"
            ].map(
                split_order_map
            )
        ),
    )
    .sort_values(
        [
            "_Model_order",
            "_Split_order",
            "Outer_fold",
        ]
    )
    .drop(
        columns=[
            "_Model_order",
            "_Split_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


if len(
    df_outer_evaluation_handoff
) != 30:

    raise RuntimeError(
        "正式外层评价参数交接表行数不是30。"
    )


if not df_outer_evaluation_handoff[
    "Status"
].eq(
    "handoff_ready"
).all():

    raise RuntimeError(
        "部分参数记录未准备好交接。"
    )


if df_outer_evaluation_handoff[
    "Outer_test_used_during_tuning"
].any():

    raise RuntimeError(
        "部分参数记录显示调参使用了外层测试集。"
    )


if df_outer_evaluation_handoff[
    [
        "Model",
        "Split",
        "Outer_fold",
    ]
].duplicated().any():

    raise RuntimeError(
        "参数交接表存在重复的"
        "模型×划分×外层折组合。"
    )


df_outer_evaluation_handoff.to_csv(
    OUTER_EVALUATION_PARAMETER_TABLE_PATH,
    index=False,
    encoding="utf-8-sig",
)


parameter_table_sha256 = (
    calculate_file_sha256_strict(
        OUTER_EVALUATION_PARAMETER_TABLE_PATH
    )
)


# ============================================================
# 8. 内层调参结果摘要
#    仅用于检查，不是最终外层测试指标
# ============================================================

df_inner_tuning_check_summary = (
    df_outer_evaluation_handoff
    .groupby(
        [
            "Model",
            "Split",
        ],
        as_index=False,
    )
    .agg(
        Outer_fold_count=(
            "Outer_fold",
            "nunique",
        ),

        Mean_of_best_inner_R2=(
            "Best_inner_validation_Mean_R2",
            "mean",
        ),

        Min_best_inner_R2=(
            "Best_inner_validation_Mean_R2",
            "min",
        ),

        Max_best_inner_R2=(
            "Best_inner_validation_Mean_R2",
            "max",
        ),
    )
)


# ============================================================
# 9. 保存正式外层评价参数manifest
# ============================================================

outer_evaluation_parameter_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "run_code_version": (
        RUN_CODE_VERSION
    ),

    "status": (
        "outer_evaluation_parameter_"
        "handoff_ready"
    ),

    "tuning_signature": (
        TUNING_SIGNATURE
    ),

    "study_suffix": (
        STUDY_SUFFIX
    ),

    "input_integrity_digest": (
        INPUT_INTEGRITY_DIGEST
    ),

    "source_tuning": {
        "formal_tuning_manifest": str(
            FORMAL_TUNING_MANIFEST_PATH
        ),

        "formal_study_summary": str(
            FORMAL_STUDY_SUMMARY_PATH
        ),

        "study_count": 30,

        "completed_trials": 450,

        "inner_folds_per_trial": int(
            N_INNER_FOLDS
        ),

        "total_inner_model_fits": 1350,
    },

    "handoff_design": {
        "models": (
            MODEL_ORDER
        ),

        "splits": (
            SPLIT_ORDER
        ),

        "outer_folds_per_split": int(
            N_OUTER_FOLDS
        ),

        "parameter_record_count": int(
            len(
                df_outer_evaluation_handoff
            )
        ),

        "parameter_policy": (
            "one_best_parameter_set_for_each_"
            "model_split_outer_fold"
        ),

        "cross_outer_fold_parameter_pooling": (
            False
        ),

        "global_parameter_averaging": False,

        "global_highest_trial_reused_across_"
        "outer_folds": False,
    },

    "usage_policy": {
        "formal_outer_training": (
            "train_on_matching_outer_train_only"
        ),

        "formal_outer_evaluation": (
            "predict_matching_outer_test_once"
        ),

        "parameter_matching_keys": [
            "Model",
            "Split",
            "Outer_fold",
        ],

        "inner_validation_metrics_are_"
        "final_paper_metrics": False,

        "final_paper_metrics_source": (
            "future_outer_test_evaluation"
        ),
    },

    "cell_7_files_read": {
        "formal_tuning_manifest_count": 1,

        "formal_study_summary_count": 1,

        "best_parameter_file_count": int(
            len(
                parameter_file_hashes
            )
        ),

        "trial_table_file_count": int(
            len(
                trial_table_hashes
            )
        ),

        "feature_array_file_count": 0,

        "target_array_file_count": 0,

        "outer_train_feature_file_count": 0,

        "outer_test_feature_file_count": 0,

        "outer_test_target_file_count": 0,
    },

    "leakage_control": {
        "feature_arrays_read_in_Cell_7": False,

        "target_arrays_read_in_Cell_7": False,

        "outer_train_feature_read": False,

        "outer_test_feature_read": False,

        "outer_test_target_read": False,

        "old_global_mpnn_feature_read": False,

        "external_10point_holdout_read": False,
    },

    "integrity": {
        "all_parameter_files_present": True,

        "all_trial_tables_present": True,

        "all_parameter_signatures_match": True,

        "all_parameter_objectives_match_summary": (
            True
        ),

        "all_trial_budgets_match": True,

        "all_trial_states_complete": True,

        "parameter_table_sha256": (
            parameter_table_sha256
        ),

        "unique_parameter_file_sha256_count": int(
            len(
                set(
                    parameter_file_hashes
                )
            )
        ),

        "unique_trial_table_sha256_count": int(
            len(
                set(
                    trial_table_hashes
                )
            )
        ),
    },

    "parameter_records": (
        df_outer_evaluation_handoff
        .to_dict(
            orient="records"
        )
    ),

    "outputs": {
        "parameter_table": str(
            OUTER_EVALUATION_PARAMETER_TABLE_PATH
        ),

        "parameter_manifest": str(
            OUTER_EVALUATION_PARAMETER_MANIFEST_PATH
        ),
    },
}


save_json_atomic(
    data=outer_evaluation_parameter_manifest,
    output_path=(
        OUTER_EVALUATION_PARAMETER_MANIFEST_PATH
    ),
)


# ============================================================
# 10. 输出检查
# ============================================================

print("=" * 112)

print(
    "Cell 7 30套外层折专属最优参数审计与交接完成 ✓"
)

print("=" * 112)


print(
    "parameter record count              =",
    len(
        df_outer_evaluation_handoff
    ),
)


print(
    "models                              =",
    MODEL_ORDER,
)


print(
    "splits                              =",
    SPLIT_ORDER,
)


print(
    "outer folds/split                   =",
    N_OUTER_FOLDS,
)


print(
    "matching parameter policy           =",
    "model + split + outer_fold",
)


print(
    "global parameter averaging          =",
    False,
)


print(
    "one global best set reused          =",
    False,
)


print(
    "all parameter files present         =",
    True,
)


print(
    "all Trial tables present            =",
    True,
)


print(
    "all trial states complete           =",
    True,
)


print(
    "feature arrays read in Cell 7       =",
    False,
)


print(
    "target arrays read in Cell 7        =",
    False,
)


print(
    "outer train feature read            =",
    False,
)


print(
    "outer test feature read             =",
    False,
)


print(
    "outer test target read              =",
    False,
)


print(
    "old global MPNN feature read        =",
    False,
)


print(
    "external 10 points read             =",
    False,
)


print(
    "inner validation metrics are final  =",
    False,
)


print(
    "parameter table SHA256              =",
    parameter_table_sha256,
)


print(
    "\n内层调参结果检查汇总："
)


print(
    "注意：下表不是最终外层测试性能。"
)


display(
    df_inner_tuning_check_summary
)


print(
    "\n前10条参数交接记录："
)


display(
    df_outer_evaluation_handoff[
        [
            "Model",
            "Split",
            "Outer_fold",
            "Trial_budget",
            "Best_trial_number",
            "Best_inner_validation_Mean_R2",
            "Status",
        ]
    ].head(
        10
    )
)


print(
    "\n正式外层评价参数表："
)


print(
    OUTER_EVALUATION_PARAMETER_TABLE_PATH
)


print(
    "\n正式外层评价参数manifest："
)


print(
    OUTER_EVALUATION_PARAMETER_MANIFEST_PATH
)

Cell 7 30套外层折专属最优参数审计与交接完成 ✓
parameter record count              = 30
models                              = ['ExtraTrees', 'LightGBM', 'Transformer']
splits                              = ['Data_split', 'IL_split']
outer folds/split                   = 5
matching parameter policy           = model + split + outer_fold
global parameter averaging          = False
one global best set reused          = False
all parameter files present         = True
all Trial tables present            = True
all trial states complete           = True
feature arrays read in Cell 7       = False
target arrays read in Cell 7        = False
outer train feature read            = False
outer test feature read             = False
outer test target read              = False
old global MPNN feature read        = False
external 10 points read             = False
inner validation metrics are final  = False
parameter table SHA256              = ece411a30eea7c63c7d46aae8afb9ca9aaa2573e4a3271df37206e2953cc8a36

内层调参结果检

,Model,Split,Outer_fold_count,Mean_of_best_inner_R2,Min_best_inner_R2,Max_best_inner_R2
0,ExtraTrees,Data_split,5,0.969116,0.956614,0.976591
1,ExtraTrees,IL_split,5,0.716579,0.647562,0.791475
2,LightGBM,Data_split,5,0.968035,0.961152,0.975673
3,LightGBM,IL_split,5,0.748485,0.684219,0.816231
4,Transformer,Data_split,5,0.974522,0.972434,0.978592
5,Transformer,IL_split,5,0.761679,0.722532,0.818225



前10条参数交接记录：


,Model,Split,Outer_fold,Trial_budget,Best_trial_number,Best_inner_validation_Mean_R2,Status
0,ExtraTrees,Data_split,1,15,14,0.968072,handoff_ready
1,ExtraTrees,Data_split,2,15,14,0.974647,handoff_ready
2,ExtraTrees,Data_split,3,15,13,0.956614,handoff_ready
3,ExtraTrees,Data_split,4,15,7,0.976591,handoff_ready
4,ExtraTrees,Data_split,5,15,12,0.969655,handoff_ready
5,ExtraTrees,IL_split,1,15,0,0.671363,handoff_ready
6,ExtraTrees,IL_split,2,15,12,0.779982,handoff_ready
7,ExtraTrees,IL_split,3,15,4,0.692515,handoff_ready
8,ExtraTrees,IL_split,4,15,3,0.647562,handoff_ready
9,ExtraTrees,IL_split,5,15,12,0.791475,handoff_ready



正式外层评价参数表：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\summaries\4eafa9d6fbe7\outer_evaluation_handoff\outer_evaluation_parameter_table_Data_IL.csv

正式外层评价参数manifest：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\summaries\4eafa9d6fbe7\outer_evaluation_handoff\outer_evaluation_parameter_manifest_Data_IL.json
